# NB06 — Temporal gating v1 trên Siamese identity-v1

Experiment giữ nguyên checkpoint/fusion production `0.425/0.475/0.100 @ 0.540` và chỉ cache-replay temporal gating. Legacy temporal luôn có trong grid làm control. Xem `docs/TEMPORAL_GATING_V1.md`.


In [ ]:
# ============================================================
# 02_TRAIN_AND_INFER: FINETUNE YOLO + SUBMISSION
# ============================================================
!pip install -q "protobuf~=3.20.0" ultralytics==8.3.221

In [ ]:
# ==============================================================================
# SPRINT 1 – IMPROVED INFERENCE SCRIPT
# Thay thế cell chính trong 06_inference_main.ipynb
#
# Các cải tiến so với baseline:
#   [A] Temporal smoothing parameters được tune tốt hơn (max_gap=5, min_seg=3)
#   [B] YOLO TTA (augment=True) – detect tốt hơn cho small objects
#   [C] Multi-scale reference embedding – bridge domain gap tốt hơn
#   [D] Per-video ST-IoU logging để debug dễ hơn
# ==============================================================================

# --- 1. IMPORTS & CONFIGURATION ---
import os
import copy
import bisect
import csv
import gzip
import hashlib
import json
import math
import random
import time
import cv2
import numpy as np
from PIL import Image
from tqdm.notebook import tqdm
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import transforms
from torchvision.models import mobilenet_v3_small
import matplotlib.pyplot as plt
from ultralytics import YOLO

# ================= CẤU HÌNH ĐƯỜNG DẪN (SỬA Ở ĐÂY) =================
TEST_DATA_DIR      = '/kaggle/input/datasets/tunphmnguynanh/data-gannhantay-zalo/public_test_remake_full/public_test_remake_full/samples'
GT_ANN_PATH        = '/kaggle/input/datasets/tunphmnguynanh/data-gannhantay-zalo/public_test_remake_full/public_test_remake_full/annotations/drone_annotations.json'
OUTPUT_FILE        = '/kaggle/working/predictions_identity_v1_calibrated.json'

YOLO_MODEL_PATH    = '/kaggle/input/datasets/phamnguyenanhtuan/weight-best-syn/yolo_drone_700.pt'
SIAMESE_MODEL_PATH = '/kaggle/input/YOUR_DATASET/siamese_mobilenet_best.pth'

# ================= [FUSION CALIBRATION PROFILES] =================
# Pha 1 cache bbox + raw YOLO/Siamese/Color scores. Pha 2 grid-search không chạy model lại.
# Production mặc định dùng fine optimum 0.722052. Đặt True chỉ khi cần chạy lại grid.
RUN_CALIBRATION = False
REUSE_CANDIDATE_CACHE = True
CALIBRATION_EXPERIMENT_NAME = 'identity_v1_calibration'
CALIBRATION_DIR = '/kaggle/working/siamese_identity_v1_calibration'
# Nếu đã upload cache hoàn chỉnh từ coarse run, điền path /kaggle/input/... tại đây.
# Để trống thì notebook dùng cache trong CALIBRATION_DIR và tự extract nếu chưa có.
PRECOMPUTED_CANDIDATE_CACHE_FILE = ''
LOCAL_CANDIDATE_CACHE_FILE = os.path.join(CALIBRATION_DIR, 'candidate_scores.json.gz')
CANDIDATE_CACHE_FILE = PRECOMPUTED_CANDIDATE_CACHE_FILE or LOCAL_CANDIDATE_CACHE_FILE

# Các profile được giữ để tái lập identity-v1 và mining-v2 calibration.
CALIBRATION_PROFILE = 'fine_identity_v1'
if CALIBRATION_PROFILE == 'coarse':
    CALIBRATION_THRESHOLDS = [round(v / 100, 2) for v in range(30, 61)]
    CALIBRATION_SIAMESE_WEIGHTS = [round(v / 100, 2) for v in range(15, 51, 5)]
    CALIBRATION_COLOR_WEIGHTS = [round(v / 100, 2) for v in range(15, 46, 5)]
elif CALIBRATION_PROFILE == 'fine_identity_v1':
    CALIBRATION_THRESHOLDS = [round(v / 1000, 3) for v in range(460, 561, 5)]
    CALIBRATION_SIAMESE_WEIGHTS = [round(v / 1000, 3) for v in range(250, 576, 25)]
    CALIBRATION_COLOR_WEIGHTS = [round(v / 1000, 3) for v in range(0, 226, 25)]
elif CALIBRATION_PROFILE == 'coarse_mining_v2':
    CALIBRATION_THRESHOLDS = [round(v / 100, 2) for v in range(35, 61)]
    CALIBRATION_SIAMESE_WEIGHTS = [round(v / 100, 2) for v in range(20, 66, 5)]
    CALIBRATION_COLOR_WEIGHTS = [round(v / 100, 2) for v in range(0, 31, 5)]
elif CALIBRATION_PROFILE == 'fine_mining_v2':
    CALIBRATION_THRESHOLDS = [round(v / 10000, 4) for v in range(5400, 6301, 25)]
    CALIBRATION_SIAMESE_WEIGHTS = [round(v / 10000, 4) for v in range(3250, 5001, 125)]
    CALIBRATION_COLOR_WEIGHTS = [round(v / 10000, 4) for v in range(0, 1251, 125)]
else:
    raise ValueError(f'Unknown CALIBRATION_PROFILE: {CALIBRATION_PROFILE}')
CALIBRATION_MIN_YOLO_WEIGHT = 0.10
CALIBRATION_TOP_K = 20
CALIBRATED_OUTPUT_FILE = os.path.join(CALIBRATION_DIR, f'predictions_{CALIBRATION_PROFILE}.json')
CALIBRATION_CSV = os.path.join(CALIBRATION_DIR, f'calibration_results_{CALIBRATION_PROFILE}.csv')
CALIBRATION_SUMMARY = os.path.join(CALIBRATION_DIR, f'calibration_summary_{CALIBRATION_PROFILE}.json')

# ================= [TEMPORAL GATING V1] =================
# Production is locked to the best joint-refinement result (offline ST-IoU 0.740013).
# Set True only to reproduce a temporal grid; the production path remains available below.
RUN_TEMPORAL_CALIBRATION = False
TEMPORAL_EXPERIMENT_NAME = 'temporal_gating_v1_joint_refine'
TEMPORAL_OUTPUT_DIR = os.path.join(CALIBRATION_DIR, TEMPORAL_EXPERIMENT_NAME)
TEMPORAL_MAX_GAP_GRID = [4, 5, 6, 7, 8, 10, 12, 15, 20]
TEMPORAL_MIN_SEG_GRID = [3, 22, 23, 24, 25, 26, 27, 28, 29, 30]
TEMPORAL_CENTER_SPEED_GRID = [None, 0.4, 0.45, 0.5, 0.6, 0.75, 1.0]
TEMPORAL_LOG_SCALE_SPEED_GRID = [None]
TEMPORAL_TOP_K = 30
TEMPORAL_PREDICTIONS_FILE = os.path.join(TEMPORAL_OUTPUT_DIR, f'predictions_{TEMPORAL_EXPERIMENT_NAME}.json')
TEMPORAL_RESULTS_CSV = os.path.join(TEMPORAL_OUTPUT_DIR, f'{TEMPORAL_EXPERIMENT_NAME}_results.csv')
TEMPORAL_SUMMARY_FILE = os.path.join(TEMPORAL_OUTPUT_DIR, f'{TEMPORAL_EXPERIMENT_NAME}_summary.json')

# ================= [TOP-K TEMPORAL CANDIDATE RERANKING V1] =================
# Experiment-only until identity-held-out promotion criteria are satisfied.
RUN_RERANKING_EXPERIMENT = False
USE_TEMPORAL_RERANKING = False
RERANK_EXPERIMENT_NAME = 'temporal_reranking_v1_identity_cv'
RERANK_OUTPUT_DIR = os.path.join(CALIBRATION_DIR, RERANK_EXPERIMENT_NAME)
RERANK_TOP_K_GRID = [3, 5]
RERANK_NEIGHBOR_RADIUS_GRID = [1, 3, 5]
RERANK_WEIGHT_GRID = [0.0, 0.03, 0.06, 0.10]
RERANK_MOTION_SCALE = 0.4
RERANK_MIN_MEAN_CV_DELTA = 0.003
RERANK_MIN_IMPROVED_IDENTITIES = 2
RERANK_MAX_WORST_IDENTITY_DROP = 0.010
RERANK_RESULTS_CSV = os.path.join(RERANK_OUTPUT_DIR, f'{RERANK_EXPERIMENT_NAME}_results.csv')
RERANK_FOLDS_CSV = os.path.join(RERANK_OUTPUT_DIR, f'{RERANK_EXPERIMENT_NAME}_folds.csv')
RERANK_SUMMARY_FILE = os.path.join(RERANK_OUTPUT_DIR, f'{RERANK_EXPERIMENT_NAME}_summary.json')
RERANK_PREDICTIONS_FILE = os.path.join(RERANK_OUTPUT_DIR, f'predictions_{RERANK_EXPERIMENT_NAME}.json')

# ================= [CANDIDATE-CACHE ORACLE DIAGNOSTICS V1] =================
# Read-only evaluation: never changes ranking, thresholding, temporal output, or prediction JSON.
RUN_ORACLE_DIAGNOSTICS = False
ORACLE_EXPERIMENT_NAME = 'candidate_oracle_diagnostics_v1'
ORACLE_OUTPUT_DIR = os.path.join(CALIBRATION_DIR, ORACLE_EXPERIMENT_NAME)
ORACLE_IOU_THRESHOLDS = [0.3, 0.5, 0.7]
ORACLE_GOOD_IOU = 0.5
ORACLE_GT_FAILURE_CATEGORIES = [
    'no_candidate', 'detector_localization_miss', 'matching_error',
    'threshold_rejection', 'temporal_removal',
    'temporal_localization_degradation', 'success',
]
ORACLE_FRAME_CSV = os.path.join(ORACLE_OUTPUT_DIR, f'{ORACLE_EXPERIMENT_NAME}_frames.csv')
ORACLE_VIDEO_CSV = os.path.join(ORACLE_OUTPUT_DIR, f'{ORACLE_EXPERIMENT_NAME}_videos.csv')
ORACLE_SUMMARY_FILE = os.path.join(ORACLE_OUTPUT_DIR, f'{ORACLE_EXPERIMENT_NAME}_summary.json')

# ================= [BRACKETED TEMPORAL THRESHOLD HYSTERESIS V1] =================
# Experiment-only: weak candidates require motion-consistent strong anchors on both sides.
RUN_HYSTERESIS_EXPERIMENT = False
USE_TEMPORAL_HYSTERESIS = False
HYSTERESIS_EXPERIMENT_NAME = 'temporal_hysteresis_v1_identity_cv'
HYSTERESIS_OUTPUT_DIR = os.path.join(CALIBRATION_DIR, HYSTERESIS_EXPERIMENT_NAME)
HYSTERESIS_STRONG_THRESHOLD = 0.54
HYSTERESIS_WEAK_THRESHOLD_GRID = [0.46, 0.48, 0.50, 0.52, 0.54]
HYSTERESIS_ANCHOR_SPAN_GRID = [8, 16, 32]
HYSTERESIS_MAX_CENTER_SPEED = 0.4
HYSTERESIS_MIN_MEAN_CV_DELTA = 0.003
HYSTERESIS_MIN_IMPROVED_IDENTITIES = 2
HYSTERESIS_MAX_WORST_IDENTITY_DROP = 0.010
HYSTERESIS_RESULTS_CSV = os.path.join(HYSTERESIS_OUTPUT_DIR, f'{HYSTERESIS_EXPERIMENT_NAME}_results.csv')
HYSTERESIS_FOLDS_CSV = os.path.join(HYSTERESIS_OUTPUT_DIR, f'{HYSTERESIS_EXPERIMENT_NAME}_folds.csv')
HYSTERESIS_SUMMARY_FILE = os.path.join(HYSTERESIS_OUTPUT_DIR, f'{HYSTERESIS_EXPERIMENT_NAME}_summary.json')
HYSTERESIS_PREDICTIONS_FILE = os.path.join(HYSTERESIS_OUTPUT_DIR, f'predictions_{HYSTERESIS_EXPERIMENT_NAME}.json')

# ================= [CANDIDATE MARGIN & COMPONENT DIAGNOSTICS V1] =================
# Read-only cache analysis; production ranking and predictions remain frozen.
RUN_MARGIN_DIAGNOSTICS = False
MARGIN_EXPERIMENT_NAME = 'candidate_margin_diagnostics_v1'
MARGIN_OUTPUT_DIR = os.path.join(CALIBRATION_DIR, MARGIN_EXPERIMENT_NAME)
MARGIN_WEAK_SCORE_THRESHOLD = 0.54
MARGIN_FEATURES = [
    'selected_score', 'score_margin', 'score_ratio',
    'selected_yolo', 'selected_siamese', 'selected_color',
    'negative_candidate_count',
]
MARGIN_FRAME_CSV = os.path.join(MARGIN_OUTPUT_DIR, f'{MARGIN_EXPERIMENT_NAME}_frames.csv')
MARGIN_VIDEO_CSV = os.path.join(MARGIN_OUTPUT_DIR, f'{MARGIN_EXPERIMENT_NAME}_videos.csv')
MARGIN_SUMMARY_FILE = os.path.join(MARGIN_OUTPUT_DIR, f'{MARGIN_EXPERIMENT_NAME}_summary.json')

# ================= [DETECTOR PROFILE V1] =================
# Read-only candidate/GT profiling before resolution, SAHI, or P2/P3 experiments.
RUN_DETECTOR_PROFILE = False
DETECTOR_PROFILE_EXPERIMENT_NAME = 'detector_profile_v1'
DETECTOR_PROFILE_OUTPUT_DIR = os.path.join(CALIBRATION_DIR, DETECTOR_PROFILE_EXPERIMENT_NAME)
DETECTOR_PROFILE_IOU_THRESHOLDS = [0.3, 0.5, 0.7]
DETECTOR_PROFILE_REFERENCE_IMGSZ = 640
DETECTOR_PROFILE_SIZE_BINS = [
    ('tiny_lt_8', 0.0, 8.0),
    ('very_small_8_16', 8.0, 16.0),
    ('small_16_32', 16.0, 32.0),
    ('medium_32_64', 32.0, 64.0),
    ('large_ge_64', 64.0, None),
]
DETECTOR_PROFILE_FRAME_CSV = os.path.join(
    DETECTOR_PROFILE_OUTPUT_DIR, f'{DETECTOR_PROFILE_EXPERIMENT_NAME}_frames.csv'
)
DETECTOR_PROFILE_VIDEO_CSV = os.path.join(
    DETECTOR_PROFILE_OUTPUT_DIR, f'{DETECTOR_PROFILE_EXPERIMENT_NAME}_videos.csv'
)
DETECTOR_PROFILE_SUMMARY_FILE = os.path.join(
    DETECTOR_PROFILE_OUTPUT_DIR, f'{DETECTOR_PROFILE_EXPERIMENT_NAME}_summary.json'
)

# ================= [EXPLICIT RESOLUTION CANDIDATE A/B V1] =================
# Experimental candidate caches only; production cache/output are never auto-promoted.
RUN_RESOLUTION_AB = False
RESOLUTION_EXPERIMENT_NAME = 'resolution_candidate_ab_v1'
RESOLUTION_OUTPUT_DIR = os.path.join(CALIBRATION_DIR, RESOLUTION_EXPERIMENT_NAME)
RESOLUTION_VARIANTS = [640, 960]
RESOLUTION_PROFILE_REFERENCE_IMGSZ = 640
RESOLUTION_CONTROL_TOLERANCE = 1e-6
RESOLUTION_MIN_MEAN_ST_IOU_DELTA = 0.003
RESOLUTION_MIN_IMPROVED_IDENTITIES = 2
RESOLUTION_MAX_WORST_IDENTITY_DROP = 0.010
RESOLUTION_MIN_SUB16_RECALL_DELTA = 0.002
RESOLUTION_RESULTS_CSV = os.path.join(
    RESOLUTION_OUTPUT_DIR, f'{RESOLUTION_EXPERIMENT_NAME}_results.csv'
)
RESOLUTION_VIDEO_CSV = os.path.join(
    RESOLUTION_OUTPUT_DIR, f'{RESOLUTION_EXPERIMENT_NAME}_videos.csv'
)
RESOLUTION_SUMMARY_FILE = os.path.join(
    RESOLUTION_OUTPUT_DIR, f'{RESOLUTION_EXPERIMENT_NAME}_summary.json'
)

# ================= [YOLO P2 DETECTOR CHECKPOINT A/B V1] =================
# Locked checkpoint evaluation: production vs paired P3 control vs P2 stride-4.
RUN_P2_DETECTOR_AB = False
P2_AB_EXPERIMENT_NAME = 'yolo_p2_detector_locked_eval_v1'
P2_AB_OUTPUT_DIR = os.path.join(CALIBRATION_DIR, P2_AB_EXPERIMENT_NAME)
# Upload both weights from notebook 03 and set these two paths.
P3_CONTROL_MODEL_PATH = ''
P2_STRIDE4_MODEL_PATH = ''
P2_AB_VARIANTS = {
    'p3_control': {
        'model_path': P3_CONTROL_MODEL_PATH,
        'expected_sha256': 'd317a81a494113ddc789df116a7adb9ef90be362aa4ef7a63a06242f47843234',
        'expected_strides': [8.0, 16.0, 32.0],
    },
    'p2_stride4': {
        'model_path': P2_STRIDE4_MODEL_PATH,
        'expected_sha256': '8f8279f178dc7ec1f57a8b19a28ff7b470a4652398face895a00deef67d60e16',
        'expected_strides': [4.0, 8.0, 16.0, 32.0],
    },
}
# Optional complete caches. Leave blank to extract once and save under P2_AB_OUTPUT_DIR.
P2_AB_PRECOMPUTED_CACHES = {'p3_control': '', 'p2_stride4': ''}
P2_AB_IMGSZ = 640
P2_AB_MIN_MEAN_ST_IOU_DELTA = 0.003
P2_AB_MIN_IMPROVED_IDENTITIES = 2
P2_AB_MAX_WORST_IDENTITY_DROP = 0.010
P2_AB_MIN_SUB16_RECALL_DELTA = 0.002
P2_AB_MIN_AGGREGATE_FPS = 25.0
P2_AB_RESULTS_CSV = os.path.join(P2_AB_OUTPUT_DIR, f'{P2_AB_EXPERIMENT_NAME}_results.csv')
P2_AB_VIDEO_CSV = os.path.join(P2_AB_OUTPUT_DIR, f'{P2_AB_EXPERIMENT_NAME}_videos.csv')
P2_AB_SUMMARY_FILE = os.path.join(P2_AB_OUTPUT_DIR, f'{P2_AB_EXPERIMENT_NAME}_summary.json')

# ================= [CHECKPOINT SCORE CALIBRATION V1] =================
# Cache-only calibration for production/P3/P2 with leave-one-identity-out selection.
# Reuse the complete P3/P2 caches produced by the locked detector A/B run above.
RUN_CHECKPOINT_SCORE_CALIBRATION = False
SCORE_CAL_EXPERIMENT_NAME = 'checkpoint_score_calibration_v1_identity_cv'
SCORE_CAL_OUTPUT_DIR = os.path.join(CALIBRATION_DIR, SCORE_CAL_EXPERIMENT_NAME)
SCORE_CAL_YOLO_POWERS = [0.75, 1.0, 1.25, 1.5]
SCORE_CAL_YOLO_WEIGHTS = [0.25, 0.35, 0.425, 0.50, 0.60]
SCORE_CAL_COLOR_WEIGHTS = [0.0, 0.05, 0.10, 0.15, 0.20]
SCORE_CAL_THRESHOLDS = [round(value / 100, 2) for value in range(44, 65, 2)]
SCORE_CAL_MIN_SIAMESE_WEIGHT = 0.20
SCORE_CAL_MIN_MEAN_CV_DELTA_VS_PRODUCTION = 0.003
SCORE_CAL_MIN_IMPROVED_IDENTITIES = 2
SCORE_CAL_MAX_WORST_IDENTITY_DROP = 0.010
SCORE_CAL_MIN_CONFIG_VOTES = 2
SCORE_CAL_MIN_AGGREGATE_FPS = 25.0
SCORE_CAL_RESULTS_CSV = os.path.join(
    SCORE_CAL_OUTPUT_DIR, f'{SCORE_CAL_EXPERIMENT_NAME}_results.csv'
)
SCORE_CAL_FOLDS_CSV = os.path.join(
    SCORE_CAL_OUTPUT_DIR, f'{SCORE_CAL_EXPERIMENT_NAME}_folds.csv'
)
SCORE_CAL_SUMMARY_FILE = os.path.join(
    SCORE_CAL_OUTPUT_DIR, f'{SCORE_CAL_EXPERIMENT_NAME}_summary.json'
)

# ================= [HYBRID FULL-FRAME + SAHI TILES V1] =================
# Preserve every legacy candidate; add native-scale tile candidates only.
RUN_HYBRID_SAHI = False
SAHI_EXPERIMENT_NAME = 'hybrid_sahi_candidate_v1'
SAHI_OUTPUT_DIR = os.path.join(CALIBRATION_DIR, SAHI_EXPERIMENT_NAME)
SAHI_TILE_SIZE = 640
SAHI_TILE_IMGSZ = 640
SAHI_FRAME_BATCH_SIZE = 8
SAHI_TILE_INFERENCE_BATCH_SIZE = 16
SAHI_OVERLAP_RATIO = 0.25
SAHI_BASELINE_REFERENCE_FPS = 30.308744167118398
SAHI_MIN_MEAN_ST_IOU_DELTA = 0.003
SAHI_MIN_IMPROVED_IDENTITIES = 2
SAHI_MAX_WORST_IDENTITY_DROP = 0.010
SAHI_MIN_SUB16_RECALL_DELTA = 0.002
SAHI_ORACLE_TOLERANCE = 1e-12
PRECOMPUTED_SAHI_TILE_CACHE_FILE = ''
LOCAL_SAHI_TILE_CACHE_FILE = os.path.join(
    SAHI_OUTPUT_DIR, 'tile_candidate_scores_size640_overlap025.json.gz'
)
SAHI_TILE_CACHE_FILE = (
    PRECOMPUTED_SAHI_TILE_CACHE_FILE or LOCAL_SAHI_TILE_CACHE_FILE
)

# ================= [SAHI SOURCE ADMISSION V1] =================
# Cache-replay only: restrict where tile candidates may compete with legacy.
RUN_SAHI_ADMISSION = False
SAHI_ADMISSION_EXPERIMENT_NAME = 'sahi_source_admission_v1'
SAHI_ADMISSION_OUTPUT_DIR = os.path.join(
    CALIBRATION_DIR, SAHI_ADMISSION_EXPERIMENT_NAME
)
SAHI_ADMISSION_POLICIES = [
    'legacy_only',
    'missing_legacy_only',
    'weak_legacy_rescue',
]
SAHI_ADMISSION_MIN_MEAN_CV_DELTA = 0.003
SAHI_ADMISSION_MIN_IMPROVED_IDENTITIES = 2
SAHI_ADMISSION_MAX_WORST_IDENTITY_DROP = 0.010
SAHI_ADMISSION_MIN_POLICY_FOLD_COUNT = 2
SAHI_ADMISSION_ORACLE_TOLERANCE = 1e-12
SAHI_ADMISSION_RESULTS_CSV = os.path.join(
    SAHI_ADMISSION_OUTPUT_DIR, f'{SAHI_ADMISSION_EXPERIMENT_NAME}_results.csv'
)
SAHI_ADMISSION_VIDEO_CSV = os.path.join(
    SAHI_ADMISSION_OUTPUT_DIR, f'{SAHI_ADMISSION_EXPERIMENT_NAME}_videos.csv'
)
SAHI_ADMISSION_FOLDS_CSV = os.path.join(
    SAHI_ADMISSION_OUTPUT_DIR, f'{SAHI_ADMISSION_EXPERIMENT_NAME}_folds.csv'
)
SAHI_ADMISSION_SUMMARY_FILE = os.path.join(
    SAHI_ADMISSION_OUTPUT_DIR, f'{SAHI_ADMISSION_EXPERIMENT_NAME}_summary.json'
)
SAHI_RESULTS_CSV = os.path.join(SAHI_OUTPUT_DIR, f'{SAHI_EXPERIMENT_NAME}_results.csv')
SAHI_VIDEO_CSV = os.path.join(SAHI_OUTPUT_DIR, f'{SAHI_EXPERIMENT_NAME}_videos.csv')
SAHI_SUMMARY_FILE = os.path.join(SAHI_OUTPUT_DIR, f'{SAHI_EXPERIMENT_NAME}_summary.json')
SAHI_PREDICTIONS_FILE = os.path.join(SAHI_OUTPUT_DIR, 'predictions_hybrid_sahi_v1.json')

# ================= [CANDIDATE TRACKLET GRAPH V1] =================
# Cache-only global association. Production stays frozen until identity-CV gates pass.
RUN_TRACKLET_GRAPH_EXPERIMENT = False
USE_TRACKLET_GRAPH = False
TRACKLET_GRAPH_EXPERIMENT_NAME = 'candidate_tracklet_graph_v1_identity_cv'
TRACKLET_GRAPH_OUTPUT_DIR = os.path.join(
    CALIBRATION_DIR, TRACKLET_GRAPH_EXPERIMENT_NAME
)
# Small pre-registered grid. The score threshold/fusion/temporal cleanup stay locked.
TRACKLET_GRAPH_TOP_K_GRID = [3, 5]
TRACKLET_GRAPH_MOTION_WEIGHT_GRID = [0.02, 0.05, 0.10]
TRACKLET_GRAPH_STATE_CHANGE_PENALTY_GRID = [0.02, 0.05]
TRACKLET_GRAPH_MAX_CENTER_SPEED_GRID = [0.4, 0.6]
TRACKLET_GRAPH_CAMERA_COMPENSATION_GRID = [False, True]
TRACKLET_GRAPH_EMISSION_SCALE = 1.0
TRACKLET_GRAPH_MAX_GLOBAL_SHIFT = 256.0
TRACKLET_GRAPH_MIN_GLOBAL_SHIFT_MATCHES = 3
TRACKLET_GRAPH_MIN_MEAN_CV_DELTA = 0.003
TRACKLET_GRAPH_MIN_IMPROVED_IDENTITIES = 2
TRACKLET_GRAPH_MAX_WORST_IDENTITY_DROP = 0.010
TRACKLET_GRAPH_MIN_CONFIG_VOTES = 2
TRACKLET_GRAPH_MIN_REPLAY_FPS = 25.0
TRACKLET_GRAPH_RESULTS_CSV = os.path.join(
    TRACKLET_GRAPH_OUTPUT_DIR, f'{TRACKLET_GRAPH_EXPERIMENT_NAME}_results.csv'
)
TRACKLET_GRAPH_FOLDS_CSV = os.path.join(
    TRACKLET_GRAPH_OUTPUT_DIR, f'{TRACKLET_GRAPH_EXPERIMENT_NAME}_folds.csv'
)
TRACKLET_GRAPH_VIDEO_CSV = os.path.join(
    TRACKLET_GRAPH_OUTPUT_DIR, f'{TRACKLET_GRAPH_EXPERIMENT_NAME}_videos.csv'
)
TRACKLET_GRAPH_SUMMARY_FILE = os.path.join(
    TRACKLET_GRAPH_OUTPUT_DIR, f'{TRACKLET_GRAPH_EXPERIMENT_NAME}_summary.json'
)
TRACKLET_GRAPH_PREDICTIONS_FILE = os.path.join(
    TRACKLET_GRAPH_OUTPUT_DIR, f'predictions_{TRACKLET_GRAPH_EXPERIMENT_NAME}.json'
)

# ================= [MOTION DISTRIBUTION DIAGNOSTICS V1] =================
# Read-only GT/cache analysis after Tracklet Graph V1 over-suppressed strong frames.
RUN_MOTION_PROFILE = False
MOTION_PROFILE_EXPERIMENT_NAME = 'motion_distribution_diagnostics_v1'
MOTION_PROFILE_OUTPUT_DIR = os.path.join(
    CALIBRATION_DIR, MOTION_PROFILE_EXPERIMENT_NAME
)
MOTION_PROFILE_REFERENCE_IMGSZ = 640
MOTION_PROFILE_GATE_THRESHOLDS = [0.4, 0.6]
MOTION_PROFILE_GOOD_ENDPOINT_IOU = 0.5
MOTION_PROFILE_PERCENTILES = [0.5, 0.75, 0.9, 0.95, 0.99]
MOTION_PROFILE_STAGES = [
    'gt', 'selected_top1', 'oracle_candidate', 'production_final'
]
MOTION_PROFILE_FRAME_CSV = os.path.join(
    MOTION_PROFILE_OUTPUT_DIR, f'{MOTION_PROFILE_EXPERIMENT_NAME}_frames.csv'
)
MOTION_PROFILE_GROUP_CSV = os.path.join(
    MOTION_PROFILE_OUTPUT_DIR, f'{MOTION_PROFILE_EXPERIMENT_NAME}_groups.csv'
)
MOTION_PROFILE_VIDEO_CSV = os.path.join(
    MOTION_PROFILE_OUTPUT_DIR, f'{MOTION_PROFILE_EXPERIMENT_NAME}_videos.csv'
)
MOTION_PROFILE_SUMMARY_FILE = os.path.join(
    MOTION_PROFILE_OUTPUT_DIR, f'{MOTION_PROFILE_EXPERIMENT_NAME}_summary.json'
)

# ================= [SPATIAL LOCALIZATION HEADROOM V1] =================
RUN_SPATIAL_HEADROOM = False
SPATIAL_HEADROOM_EXPERIMENT_NAME = 'spatial_localization_headroom_v1'
SPATIAL_HEADROOM_OUTPUT_DIR = os.path.join(CALIBRATION_DIR, SPATIAL_HEADROOM_EXPERIMENT_NAME)
SPATIAL_HEADROOM_REFERENCE_IMGSZ = 640
SPATIAL_HEADROOM_GOOD_IOU = 0.5
SPATIAL_HEADROOM_LOCAL_OVERLAP = 0.5
SPATIAL_HEADROOM_MIN_REFINER_CEILING_DELTA = 0.010
SPATIAL_HEADROOM_MIN_IDENTITY_CEILING_DELTA = 0.005
SPATIAL_HEADROOM_MIN_QUALIFYING_IDENTITIES = 2
SPATIAL_HEADROOM_SCENARIOS = (
    'perfect_shared', 'perfect_good_overlap', 'candidate_keep_best',
    'local_candidate_keep_best', 'center_only_good_overlap', 'size_only_good_overlap',
)

# ================= [SPATIAL REFINER A/B V2 — OFFLINE] =================
# Post-temporal bbox-only experiment. Frame support and production output stay frozen.
RUN_SPATIAL_REFINER_V2_AB = True
SPATIAL_REFINER_V2_EXPERIMENT_NAME = 'spatial_refiner_v2_locked_eval'
SPATIAL_REFINER_V2_OUTPUT_DIR = os.path.join(CALIBRATION_DIR, SPATIAL_REFINER_V2_EXPERIMENT_NAME)
# Upload both artifacts from 08-train-spatial-refiner-v2.ipynb after identity-CV passes.
SPATIAL_REFINER_V2_MODEL_PATH = '/kaggle/input/YOUR_REFINER_DATASET/spatial_refiner_v2_final.ts'
SPATIAL_REFINER_V2_RUN_SUMMARY_PATH = '/kaggle/input/YOUR_REFINER_DATASET/run_summary.json'
SPATIAL_REFINER_V2_SEARCH_SIZE = 256
SPATIAL_REFINER_V2_REFERENCE_SIZE = 128
SPATIAL_REFINER_V2_SEARCH_FACTOR = 4.0
SPATIAL_REFINER_V2_MIN_SEARCH_SIDE = 96.0
SPATIAL_REFINER_V2_BATCH_SIZE = 32
SPATIAL_REFINER_V2_MAX_CENTER_SHIFT = 0.5
SPATIAL_REFINER_V2_MAX_LOG_SCALE_CHANGE = 0.7
SPATIAL_REFINER_V2_BASELINE_REFERENCE_FPS = 30.308744167118398
SPATIAL_REFINER_V2_MIN_PUBLIC_DELTA = 0.003
SPATIAL_REFINER_V2_MIN_IMPROVED_IDENTITIES = 2
SPATIAL_REFINER_V2_MAX_WORST_IDENTITY_DROP = 0.005
SPATIAL_REFINER_V2_MIN_PIPELINED_FPS = 25.0
SPATIAL_REFINER_V2_PREDICTIONS_FILE = os.path.join(
    SPATIAL_REFINER_V2_OUTPUT_DIR, 'predictions_spatial_refiner_v2.json'
)
SPATIAL_REFINER_V2_FRAMES_CSV = os.path.join(
    SPATIAL_REFINER_V2_OUTPUT_DIR, f'{SPATIAL_REFINER_V2_EXPERIMENT_NAME}_frames.csv'
)
SPATIAL_REFINER_V2_VIDEOS_CSV = os.path.join(
    SPATIAL_REFINER_V2_OUTPUT_DIR, f'{SPATIAL_REFINER_V2_EXPERIMENT_NAME}_videos.csv'
)
SPATIAL_REFINER_V2_SUMMARY_FILE = os.path.join(
    SPATIAL_REFINER_V2_OUTPUT_DIR, f'{SPATIAL_REFINER_V2_EXPERIMENT_NAME}_summary.json'
)

# ================= [ANCHOR-PRESERVING RESIDUAL RECOVERY V2] =================
# Add-only recovery between production anchors; strong production boxes are immutable.
RUN_RESIDUAL_RECOVERY_EXPERIMENT = False
USE_RESIDUAL_RECOVERY = False
RESIDUAL_RECOVERY_EXPERIMENT_NAME = 'anchor_preserving_residual_v2_identity_cv'
RESIDUAL_RECOVERY_OUTPUT_DIR = os.path.join(
    CALIBRATION_DIR, RESIDUAL_RECOVERY_EXPERIMENT_NAME
)
RESIDUAL_RECOVERY_TOP_K = 5
RESIDUAL_RECOVERY_WEAK_THRESHOLD_GRID = [0.50, 0.52, 0.54]
RESIDUAL_RECOVERY_MAX_ANCHOR_GAP_GRID = [12, 20]
RESIDUAL_RECOVERY_MAX_CENTER_SPEED_GRID = [0.6, 1.0]
RESIDUAL_RECOVERY_MAX_CANDIDATE_GAP = 3
RESIDUAL_RECOVERY_MIN_RECOVERED_FRAMES = 3
RESIDUAL_RECOVERY_MIN_GAP_COVERAGE = 0.5
RESIDUAL_RECOVERY_MIN_MEAN_CV_DELTA = 0.003
RESIDUAL_RECOVERY_MIN_IMPROVED_IDENTITIES = 2
RESIDUAL_RECOVERY_MAX_WORST_IDENTITY_DROP = 0.010
RESIDUAL_RECOVERY_MIN_CONFIG_VOTES = 2
RESIDUAL_RECOVERY_MIN_REPLAY_FPS = 25.0
RESIDUAL_RECOVERY_RESULTS_CSV = os.path.join(
    RESIDUAL_RECOVERY_OUTPUT_DIR, f'{RESIDUAL_RECOVERY_EXPERIMENT_NAME}_results.csv'
)
RESIDUAL_RECOVERY_FOLDS_CSV = os.path.join(
    RESIDUAL_RECOVERY_OUTPUT_DIR, f'{RESIDUAL_RECOVERY_EXPERIMENT_NAME}_folds.csv'
)
RESIDUAL_RECOVERY_VIDEO_CSV = os.path.join(
    RESIDUAL_RECOVERY_OUTPUT_DIR, f'{RESIDUAL_RECOVERY_EXPERIMENT_NAME}_videos.csv'
)
RESIDUAL_RECOVERY_SUMMARY_FILE = os.path.join(
    RESIDUAL_RECOVERY_OUTPUT_DIR, f'{RESIDUAL_RECOVERY_EXPERIMENT_NAME}_summary.json'
)
RESIDUAL_RECOVERY_PREDICTIONS_FILE = os.path.join(
    RESIDUAL_RECOVERY_OUTPUT_DIR,
    f'predictions_{RESIDUAL_RECOVERY_EXPERIMENT_NAME}.json',
)

# ======================== [SPRINT 1] TUNED CONFIG ========================
CONFIDENCE_THRESHOLD = 0.05
MATCHING_THRESHOLD   = 0.54
YOLO_ONLY_THRESHOLD  = 0.2

WEIGHT_YOLO    = 0.425
WEIGHT_SIAMESE = 0.475
WEIGHT_COLOR   = 0.100

# [SPRINT 1-B] TTA: Test-Time Augmentation
USE_TTA = True

# [SPRINT 1-C] Multi-scale reference embedding
USE_MULTISCALE_REF = True
REF_SCALES = [224, 112, 56]

# Legacy names and values are retained for exact source/cache-replay parity.
TEMPORAL_MAX_GAP = 5
TEMPORAL_MIN_SEG = 3

# Production Temporal Gating V1 (joint-refinement optimum on the manual offline GT).
USE_TEMPORAL_GATING = True
PRODUCTION_TEMPORAL_MAX_GAP = 7
PRODUCTION_TEMPORAL_MIN_SEG = 24
PRODUCTION_TEMPORAL_MAX_CENTER_SPEED = 0.4
PRODUCTION_TEMPORAL_MAX_LOG_SCALE_SPEED = None
PRODUCTION_TEMPORAL_CONFIG = {
    'max_gap': PRODUCTION_TEMPORAL_MAX_GAP,
    'min_seg_len': PRODUCTION_TEMPORAL_MIN_SEG,
    'max_center_speed': PRODUCTION_TEMPORAL_MAX_CENTER_SPEED,
    'max_log_scale_speed': PRODUCTION_TEMPORAL_MAX_LOG_SCALE_SPEED,
}

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
print(f"TTA: {USE_TTA} | MultiScale Ref: {USE_MULTISCALE_REF}")
print(f"Weights: YOLO={WEIGHT_YOLO}, Siamese={WEIGHT_SIAMESE}, Color={WEIGHT_COLOR}")
print(
    f"Temporal gating: enabled={USE_TEMPORAL_GATING} | max_gap={PRODUCTION_TEMPORAL_MAX_GAP}, "
    f"min_seg={PRODUCTION_TEMPORAL_MIN_SEG}, center_speed={PRODUCTION_TEMPORAL_MAX_CENTER_SPEED}, "
    f"log_scale_speed={PRODUCTION_TEMPORAL_MAX_LOG_SCALE_SPEED}"
)

# --- 2. MODEL ---

class SiameseMobileNet(nn.Module):
    def __init__(self, embedding_dim=576):
        super(SiameseMobileNet, self).__init__()
        full_model = mobilenet_v3_small(weights=None)
        self.features = full_model.features
        self.projection = nn.Sequential(
            nn.Linear(embedding_dim, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(inplace=True),
            nn.Linear(256, 128)
        )

    def forward(self, x):
        x = self.features(x)
        x = F.adaptive_avg_pool2d(x, (1, 1)).flatten(1)
        x = self.projection(x)
        return F.normalize(x, p=2, dim=1)

def get_inference_transforms(size=224):
    return transforms.Compose([
        transforms.Resize((size, size)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std=[0.229, 0.224, 0.225]),
    ])

# --- 3. HELPERS ---

def load_reference_embeddings(video_folder, model, base_transform):
    """
    [SPRINT 1-C] Multi-scale reference embedding.
    Encode ref images tai 3 scales [224, 112, 56] roi average.
    Scale nho simulate DroneDegradation → bridge domain gap.
    """
    obj_dir = os.path.join(TEST_DATA_DIR, video_folder, "object_images")
    if not os.path.exists(obj_dir):
        return None

    img_paths = sorted([
        os.path.join(obj_dir, f) for f in os.listdir(obj_dir)
        if f.lower().endswith(('.jpg', '.jpeg', '.png'))
    ])
    if not img_paths:
        return None

    all_tensors = []
    scales_to_use = REF_SCALES if USE_MULTISCALE_REF else [224]
    transform_224 = get_inference_transforms(size=224)

    for p in img_paths:
        img = cv2.imread(p)
        if img is None:
            continue
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        pil_img = Image.fromarray(img_rgb)

        for scale in scales_to_use:
            if scale != 224:
                # Downscale then NEAREST upsample = DroneDegradation effect
                small = pil_img.resize((scale, scale), Image.BILINEAR)
                pil_input = small.resize((224, 224), Image.NEAREST)
            else:
                pil_input = pil_img
            all_tensors.append(transform_224(pil_input))

    if not all_tensors:
        return None

    batch = torch.stack(all_tensors).to(DEVICE)
    with torch.no_grad():
        embs = model(batch)
        mean_emb = torch.mean(embs, dim=0, keepdim=True)
        mean_emb = F.normalize(mean_emb, p=2, dim=1)

    return mean_emb

def compute_hs_histogram(img_bgr):
    hsv = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)
    mask = cv2.inRange(hsv, np.array([0, 30, 30]), np.array([180, 255, 255]))
    hist = cv2.calcHist([hsv], [0, 1], mask, [30, 32], [0, 180, 0, 256])
    cv2.normalize(hist, hist, alpha=0, beta=1, norm_type=cv2.NORM_MINMAX)
    return hist

def load_reference_histogram(video_folder):
    obj_dir = os.path.join(TEST_DATA_DIR, video_folder, "object_images")
    if not os.path.exists(obj_dir):
        return None
    img_paths = [os.path.join(obj_dir, f) for f in os.listdir(obj_dir)
                 if f.endswith(('.jpg', '.png'))]
    agg_hist = None
    for p in img_paths:
        img = cv2.imread(p)
        if img is None:
            continue
        hist = compute_hs_histogram(img)
        agg_hist = hist if agg_hist is None else agg_hist + hist
    if agg_hist is not None:
        cv2.normalize(agg_hist, agg_hist, alpha=0, beta=1, norm_type=cv2.NORM_MINMAX)
    return agg_hist

def calculate_color_score(crop_img, ref_hist):
    if ref_hist is None or crop_img is None or crop_img.size == 0:
        return 0.0
    crop_hist = compute_hs_histogram(crop_img)
    return max(0.0, cv2.compareHist(ref_hist, crop_hist, cv2.HISTCMP_CORREL))

# --- 4. EVALUATION ---

def load_gt_annotations(gt_path):
    if not os.path.exists(gt_path):
        return {}
    with open(gt_path, 'r') as f:
        records = json.load(f)
    gt_map = {}
    for rec in records:
        vid = rec.get("video_id")
        if not vid:
            continue
        frame2bbox = {}
        for interval in rec.get("annotations", []):
            for b in interval.get("bboxes", []):
                frame2bbox[int(b["frame"])] = [int(b["x1"]), int(b["y1"]),
                                               int(b["x2"]), int(b["y2"])]
        gt_map[vid] = frame2bbox
    return gt_map

def iou_box(b1, b2):
    ix1, iy1 = max(b1[0], b2[0]), max(b1[1], b2[1])
    ix2, iy2 = min(b1[2], b2[2]), min(b1[3], b2[3])
    inter = max(0, ix2 - ix1) * max(0, iy2 - iy1)
    area1 = max(0, b1[2] - b1[0]) * max(0, b1[3] - b1[1])
    area2 = max(0, b2[2] - b2[0]) * max(0, b2[3] - b2[1])
    union = area1 + area2 - inter
    return inter / union if union > 0 else 0.0

def compute_st_iou_video(gt_frames, pred_frames):
    inter = set(gt_frames.keys()) & set(pred_frames.keys())
    union = set(gt_frames.keys()) | set(pred_frames.keys())
    if not union:
        return 0.0
    return sum(iou_box(gt_frames[f], pred_frames[f]) for f in inter) / len(union)

def visualize_sample_results(all_predictions, debug_info, num_videos=3, num_frames_per_video=3):
    print("\n" + "="*50 + "\nVISUALIZING SAMPLE RESULTS\n" + "="*50)
    valid_preds = [p for p in all_predictions
                   if len(p["detections"]) > 0 and len(p["detections"][0]["bboxes"]) > 0]
    if not valid_preds:
        print("Khong co video nao co detection.")
        return
    selected = random.sample(valid_preds, min(len(valid_preds), num_videos))
    for pred in selected:
        vid_id = pred["video_id"]
        video_info = debug_info.get(vid_id, {})
        if not video_info:
            continue
        frame_indices = sorted(list(video_info.keys()))
        chosen_frames = sorted(random.sample(frame_indices, min(len(frame_indices), num_frames_per_video)))
        print(f"\nVideo: {vid_id} | Frames: {chosen_frames}")
        vid_path = os.path.join(TEST_DATA_DIR, vid_id, "drone_video.mp4")
        cap = cv2.VideoCapture(vid_path)
        plt.figure(figsize=(5 * len(chosen_frames), 5))
        for i, f_idx in enumerate(chosen_frames):
            cap.set(cv2.CAP_PROP_POS_FRAMES, f_idx)
            ret, frame = cap.read()
            if not ret:
                continue
            frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            for cand in video_info.get(f_idx, []):
                x1, y1, x2, y2 = map(int, cand["bbox"])
                fs = cand["final_score"]
                sc = cand.get("scores", (0, 0, 0))
                color = (0, 255, 0) if cand["is_best"] else (255, 0, 0)
                label = f"{fs:.2f}(Y{sc[0]:.1f}|S{sc[1]:.1f}|C{sc[2]:.1f})" if cand["is_best"] else f"{fs:.2f}"
                cv2.rectangle(frame_rgb, (x1, y1), (x2, y2), color, 2 if cand["is_best"] else 1)
                cv2.putText(frame_rgb, label, (x1, max(15, y1 - 5)),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 1)
            plt.subplot(1, len(chosen_frames), i + 1)
            plt.imshow(frame_rgb)
            plt.axis("off")
            plt.title(f"Frame {f_idx}")
        plt.tight_layout()
        plt.show()
        cap.release()

# --- 5. [SPRINT 1-A] IMPROVED TEMPORAL SMOOTHING ---

def temporal_smooth_detections(frame_best, max_frame_idx,
                               min_seg_len=TEMPORAL_MIN_SEG,
                               max_gap=TEMPORAL_MAX_GAP):
    """
    [SPRINT 1-A] Improved temporal smoothing.

    Changes vs baseline:
      - max_gap: 4 -> 5  (tolerate 5-frame gaps)
      - min_seg_len: 5 -> 3  (keep shorter segments, fewer false negatives)
      - int(round()) for output coordinates (vs int() which truncates)

    Logic:
      1. Fill gaps <= max_gap with linear interpolation of bbox
      2. Remove isolated segments < min_seg_len (noise suppression)
    """
    T = max_frame_idx + 1
    det = [None] * T

    for f, info in frame_best.items():
        if 0 <= f < T:
            det[f] = info

    # STEP 1: Fill gaps with linear interpolation
    for i in range(T):
        if det[i] is not None:
            next_idx = -1
            for k in range(1, max_gap + 2):
                if i + k < T and det[i + k] is not None:
                    next_idx = i + k
                    break
            # Only fill if gap size <= max_gap
            gap_size = next_idx - i - 1 if next_idx != -1 else 999
            if 0 < gap_size <= max_gap:
                b1, s1 = det[i]['bbox'], det[i]['score']
                b2, s2 = det[next_idx]['bbox'], det[next_idx]['score']
                steps = next_idx - i
                for step in range(1, steps):
                    alpha = step / steps
                    ibox = [b1[j] * (1 - alpha) + b2[j] * alpha for j in range(4)]
                    iscore = s1 * (1 - alpha) + s2 * alpha
                    det[i + step] = {'bbox': ibox, 'score': iscore}

    # STEP 2: Remove short segments
    i = 0
    while i < T:
        if det[i] is not None:
            j = i
            while j + 1 < T and det[j + 1] is not None:
                j += 1
            if (j - i + 1) < min_seg_len:
                for k in range(i, j + 1):
                    det[k] = None
            i = j + 1
        else:
            i += 1

    # Output
    out = []
    for f in range(T):
        if det[f] is not None:
            x1, y1, x2, y2 = det[f]['bbox']
            out.append({
                "frame": f,
                "x1": int(round(x1)), "y1": int(round(y1)),
                "x2": int(round(x2)), "y2": int(round(y2))
            })
    return out

def gap_motion_metrics(box1, box2, steps):
    if steps <= 0:
        raise ValueError('steps must be positive')
    w1, h1 = max(1e-6, box1[2] - box1[0]), max(1e-6, box1[3] - box1[1])
    w2, h2 = max(1e-6, box2[2] - box2[0]), max(1e-6, box2[3] - box2[1])
    c1 = ((box1[0] + box1[2]) * 0.5, (box1[1] + box1[3]) * 0.5)
    c2 = ((box2[0] + box2[2]) * 0.5, (box2[1] + box2[3]) * 0.5)
    scale1, scale2 = math.sqrt(w1 * h1), math.sqrt(w2 * h2)
    mean_scale = max(1e-6, 0.5 * (scale1 + scale2))
    center_speed = math.hypot(c2[0] - c1[0], c2[1] - c1[1]) / (steps * mean_scale)
    log_scale_speed = abs(math.log(scale2 / scale1)) / steps
    return center_speed, log_scale_speed


def gap_is_plausible(box1, box2, steps, max_center_speed=None, max_log_scale_speed=None):
    center_speed, log_scale_speed = gap_motion_metrics(box1, box2, steps)
    if max_center_speed is not None and center_speed > max_center_speed:
        return False
    if max_log_scale_speed is not None and log_scale_speed > max_log_scale_speed:
        return False
    return True


def temporal_smooth_detections_gated(frame_best, max_frame_idx, min_seg_len, max_gap,
                                       max_center_speed=None, max_log_scale_speed=None):
    # Experimental branch. With both gates disabled this intentionally mirrors legacy.
    T = max_frame_idx + 1
    det = [None] * T
    for frame_idx, info in frame_best.items():
        if 0 <= frame_idx < T:
            det[frame_idx] = info

    for i in range(T):
        if det[i] is None:
            continue
        next_idx = -1
        for offset in range(1, max_gap + 2):
            if i + offset < T and det[i + offset] is not None:
                next_idx = i + offset
                break
        gap_size = next_idx - i - 1 if next_idx != -1 else 999
        if not (0 < gap_size <= max_gap):
            continue
        box1, score1 = det[i]['bbox'], det[i]['score']
        box2, score2 = det[next_idx]['bbox'], det[next_idx]['score']
        steps = next_idx - i
        if not gap_is_plausible(
            box1, box2, steps,
            max_center_speed=max_center_speed,
            max_log_scale_speed=max_log_scale_speed,
        ):
            continue
        for step in range(1, steps):
            alpha = step / steps
            ibox = [box1[j] * (1 - alpha) + box2[j] * alpha for j in range(4)]
            iscore = score1 * (1 - alpha) + score2 * alpha
            det[i + step] = {'bbox': ibox, 'score': iscore}

    i = 0
    while i < T:
        if det[i] is None:
            i += 1
            continue
        j = i
        while j + 1 < T and det[j + 1] is not None:
            j += 1
        if (j - i + 1) < min_seg_len:
            for frame_idx in range(i, j + 1):
                det[frame_idx] = None
        i = j + 1

    output = []
    for frame_idx, item in enumerate(det):
        if item is None:
            continue
        x1, y1, x2, y2 = item['bbox']
        output.append({
            'frame': frame_idx,
            'x1': int(round(x1)), 'y1': int(round(y1)),
            'x2': int(round(x2)), 'y2': int(round(y2)),
        })
    return output


# --- 6. CALIBRATION CACHE & REPLAY ---

def file_sha256(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with open(path, 'rb') as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


def atomic_write_json(path, payload, compressed=False):
    os.makedirs(os.path.dirname(path) or '.', exist_ok=True)
    temp_path = path + '.tmp'
    if compressed:
        with gzip.open(temp_path, 'wt', encoding='utf-8') as f:
            json.dump(payload, f, separators=(',', ':'))
    else:
        with open(temp_path, 'w', encoding='utf-8') as f:
            json.dump(payload, f, indent=2)
    os.replace(temp_path, path)


def read_json(path, compressed=False):
    opener = gzip.open if compressed else open
    with opener(path, 'rt', encoding='utf-8') as f:
        return json.load(f)


def resolve_precomputed_cache_path(configured_path, label,
                                   search_root='/kaggle/input'):
    """Resolve exact, .tmp/final, and Kaggle URL-style cache paths."""
    if not configured_path:
        return ''
    configured_path = os.path.normpath(os.path.expanduser(str(configured_path)))
    candidates = [configured_path]
    if configured_path.endswith('.tmp'):
        candidates.append(configured_path[:-4])
    else:
        candidates.append(configured_path + '.tmp')

    normalized = configured_path.replace('\\', '/')
    marker = '/kaggle/input/datasets/'
    if marker in normalized:
        tail = normalized.split(marker, 1)[1].split('/')
        if len(tail) >= 3:
            # Data-source URLs may include datasets/<owner>/<slug>; mounted paths
            # commonly omit owner and the literal datasets component.
            mounted = os.path.join('/kaggle/input', tail[1], *tail[2:])
            candidates.extend([mounted, mounted[:-4] if mounted.endswith('.tmp') else mounted + '.tmp'])

    checked = []
    for candidate in candidates:
        candidate = os.path.normpath(candidate)
        if candidate not in checked:
            checked.append(candidate)
            if os.path.isfile(candidate):
                if candidate != configured_path:
                    print(f'Auto-resolved {label}: {configured_path} -> {candidate}')
                return candidate

    basenames = {os.path.basename(path) for path in checked}
    matches = []
    if os.path.isdir(search_root):
        for root, _, files in os.walk(search_root):
            for filename in files:
                if filename in basenames:
                    matches.append(os.path.normpath(os.path.join(root, filename)))
    matches = sorted(set(matches))
    if len(matches) == 1:
        print(f'Auto-resolved {label} by filename: {configured_path} -> {matches[0]}')
        return matches[0]
    if len(matches) > 1:
        preview = '\n  - '.join(matches[:10])
        raise RuntimeError(
            f'{label} path is ambiguous. Configure one exact mounted path. Matches:\n  - {preview}'
        )
    raise FileNotFoundError(
        f'{label} was configured but no file exists. Requested: {configured_path}\n'
        f'Kaggle paths usually start with /kaggle/input/<dataset-slug>/. '
        f'Checked variants: {checked}'
    )


def read_cache_json(path, label):
    """Read cache by content: gzip JSON or plain UTF-8 JSON, independent of suffix."""
    try:
        file_size = os.path.getsize(path)
        with open(path, 'rb') as raw_file:
            prefix = raw_file.read(4)
        is_gzip = prefix[:2] == b'\x1f\x8b'
        if is_gzip:
            payload = read_json(path, compressed=True)
            storage_format = 'gzip_json'
        else:
            # Some download/upload tools transparently decompress .json.gz but retain
            # its suffix. Read the actual content instead of trusting the filename.
            payload = read_json(path, compressed=False)
            storage_format = 'plain_json'
            print(
                f'Warning: {label} has no gzip header; reading as plain JSON: {path}'
            )
    except (
        EOFError, gzip.BadGzipFile, json.JSONDecodeError,
        UnicodeDecodeError, OSError, RuntimeError,
    ) as exc:
        tmp_hint = (
            ' The .tmp suffix may mean the atomic write was interrupted. If this '
            'file is truncated, upload the final cache from a completed run.'
            if str(path).endswith('.tmp') else ''
        )
        raise RuntimeError(
            f'Cannot read {label} as gzip JSON or plain JSON: {path}.{tmp_hint} '
            f'Original error: {type(exc).__name__}: {exc}'
        ) from exc
    if not isinstance(payload, dict):
        raise RuntimeError(f'{label} must contain a JSON object: {path}')
    print(
        f'Validated {label}: {path} | format={storage_format} | '
        f'size={file_size / (1024 ** 2):.2f} MiB'
    )
    return payload


def read_compressed_cache(path, label):
    """Backward-compatible alias for older tests/calls."""
    return read_cache_json(path, label)


def cache_signature_mismatch_keys(actual, expected):
    keys = sorted(set(actual or {}) | set(expected or {}))
    return [key for key in keys if (actual or {}).get(key) != (expected or {}).get(key)]


def list_video_folders():
    if not os.path.isdir(TEST_DATA_DIR):
        raise FileNotFoundError(f'TEST_DATA_DIR not found: {TEST_DATA_DIR}')
    return sorted([
        f for f in os.listdir(TEST_DATA_DIR)
        if os.path.isdir(os.path.join(TEST_DATA_DIR, f))
    ])


def build_cache_signature(video_folders):
    required = {
        'yolo_model': YOLO_MODEL_PATH,
        'siamese_model': SIAMESE_MODEL_PATH,
        'ground_truth': GT_ANN_PATH,
    }
    missing = [f'{name}: {path}' for name, path in required.items() if not os.path.isfile(path)]
    if missing:
        raise FileNotFoundError('Missing calibration inputs:\n' + '\n'.join(missing))
    return {
        'schema_version': 1,
        'test_data_dir': os.path.abspath(TEST_DATA_DIR),
        'video_folders': list(video_folders),
        'yolo_sha256': file_sha256(YOLO_MODEL_PATH),
        'siamese_sha256': file_sha256(SIAMESE_MODEL_PATH),
        'gt_sha256': file_sha256(GT_ANN_PATH),
        'confidence_threshold': CONFIDENCE_THRESHOLD,
        'use_tta': USE_TTA,
        'use_multiscale_ref': USE_MULTISCALE_REF,
        'ref_scales': list(REF_SCALES),
        'score_definition': 'yolo_conf, max(0,1-L2(ref,candidate)/2), HS-histogram-correlation',
    }


def load_or_initialize_cache(signature):
    global CANDIDATE_CACHE_FILE
    precomputed_path = globals().get('PRECOMPUTED_CANDIDATE_CACHE_FILE', '')
    if precomputed_path:
        CANDIDATE_CACHE_FILE = resolve_precomputed_cache_path(
            precomputed_path, 'candidate cache'
        )
    if REUSE_CANDIDATE_CACHE and os.path.exists(CANDIDATE_CACHE_FILE):
        cache = (
            read_cache_json(CANDIDATE_CACHE_FILE, 'candidate cache')
            if 'read_cache_json' in globals()
            else read_json(CANDIDATE_CACHE_FILE, compressed=True)
        )
        if cache.get('signature') != signature:
            mismatch_keys = (
                cache_signature_mismatch_keys(cache.get('signature'), signature)
                if 'cache_signature_mismatch_keys' in globals()
                else sorted(set(cache.get('signature', {})) | set(signature))
            )
            raise RuntimeError(
                'Candidate cache does not match checkpoint/data/config. '
                f'Differing signature keys: {mismatch_keys}. '
                'Use a cache produced by this notebook configuration.'
            )
        if precomputed_path and not cache.get('complete'):
            raise RuntimeError(
                'Precomputed candidate cache is incomplete and Kaggle input is read-only: '
                f'{CANDIDATE_CACHE_FILE}. Upload candidate_scores.json.gz from a completed run.'
            )
        print(f"Loaded candidate cache: {CANDIDATE_CACHE_FILE}")
        return cache
    return {
        'schema_version': 1,
        'signature': signature,
        'complete': False,
        'videos': {},
    }


def extract_candidate_cache(cache, video_folders):
    if cache.get('complete'):
        return cache

    yolo = YOLO(YOLO_MODEL_PATH)
    siam = SiameseMobileNet().to(DEVICE)
    siam.load_state_dict(torch.load(SIAMESE_MODEL_PATH, map_location=DEVICE))
    siam.eval()
    base_transform = get_inference_transforms(size=224)
    print('Loaded models for raw candidate extraction')

    for vid_id in tqdm(video_folders, desc='Candidate extraction'):
        if vid_id in cache['videos']:
            print(f'  [{vid_id}] already cached')
            continue
        vid_path = os.path.join(TEST_DATA_DIR, vid_id, 'drone_video.mp4')
        if not os.path.exists(vid_path):
            cache['videos'][vid_id] = {
                'max_frame_idx': -1, 'has_siamese': False, 'has_color': False, 'frames': {}
            }
            atomic_write_json(CANDIDATE_CACHE_FILE, cache, compressed=True)
            continue

        ref_emb = load_reference_embeddings(vid_id, siam, base_transform)
        ref_hist = load_reference_histogram(vid_id)
        results = yolo.predict(
            vid_path, stream=True, conf=CONFIDENCE_THRESHOLD,
            verbose=False, augment=USE_TTA,
        )
        frames = {}
        max_frame_idx = -1

        for f_idx, res in enumerate(results):
            max_frame_idx = f_idx
            if not res.boxes:
                continue
            boxes = res.boxes.xyxy.cpu().numpy()
            confs = res.boxes.conf.cpu().numpy()
            orig_img = res.orig_img
            h, w = orig_img.shape[:2]
            pil_img = Image.fromarray(cv2.cvtColor(orig_img, cv2.COLOR_BGR2RGB))
            tensors, crops, valid_indices = [], [], []
            for idx, box in enumerate(boxes):
                x1, y1, x2, y2 = map(int, box)
                x1, y1 = max(0, x1), max(0, y1)
                x2, y2 = min(w, x2), min(h, y2)
                if x2 <= x1 + 5 or y2 <= y1 + 5:
                    continue
                valid_indices.append(idx)
                tensors.append(base_transform(pil_img.crop((x1, y1, x2, y2))))
                crops.append(orig_img[y1:y2, x1:x2])
            if not tensors:
                continue

            siam_scores = np.zeros(len(tensors), dtype=np.float32)
            color_scores = np.zeros(len(tensors), dtype=np.float32)
            if ref_emb is not None:
                with torch.no_grad():
                    feats = siam(torch.stack(tensors).to(DEVICE))
                    dists = torch.cdist(ref_emb, feats).cpu().numpy()[0]
                    siam_scores = np.maximum(0, 1.0 - dists / 2.0)
            if ref_hist is not None:
                for k, crop in enumerate(crops):
                    color_scores[k] = calculate_color_score(crop, ref_hist)

            frames[str(f_idx)] = [
                {
                    'bbox': boxes[idx].tolist(),
                    'scores': [float(confs[idx]), float(siam_scores[k]), float(color_scores[k])],
                }
                for k, idx in enumerate(valid_indices)
            ]

        cache['videos'][vid_id] = {
            'max_frame_idx': max_frame_idx,
            'has_siamese': ref_emb is not None,
            'has_color': ref_hist is not None,
            'frames': frames,
        }
        # Save after every completed video so a Kaggle interruption can resume safely.
        atomic_write_json(CANDIDATE_CACHE_FILE, cache, compressed=True)
        print(f'  [{vid_id}] cached candidate frames: {len(frames)}')

    cache['complete'] = all(vid in cache['videos'] for vid in video_folders)
    atomic_write_json(CANDIDATE_CACHE_FILE, cache, compressed=True)
    if not cache['complete']:
        raise RuntimeError('Candidate cache is incomplete')
    return cache


def fuse_candidate_score(scores, weights, has_siamese=True, has_color=True):
    yolo_score, siamese_score, color_score = map(float, scores)
    wy = float(weights['yolo'])
    ws = float(weights['siamese']) if has_siamese else 0.0
    wc = float(weights['color']) if has_color else 0.0
    total = wy + ws + wc
    if total <= 0:
        return yolo_score
    return (wy * yolo_score + ws * siamese_score + wc * color_score) / total


def rank_video_frames(video_cache, weights, include_debug=False):
    ranked = {}
    debug = {} if include_debug else None
    has_siamese = bool(video_cache['has_siamese'])
    has_color = bool(video_cache['has_color'])
    for frame_key, candidates in video_cache['frames'].items():
        frame_idx = int(frame_key)
        best_idx, best_score = None, -1.0
        scored = []
        for idx, candidate in enumerate(candidates):
            score = fuse_candidate_score(candidate['scores'], weights, has_siamese, has_color)
            if include_debug:
                scored.append({
                    'bbox': candidate['bbox'], 'final_score': score,
                    'scores': candidate['scores'], 'is_best': False,
                })
            # Strict > preserves the original candidate tie behavior.
            if score > best_score:
                best_idx, best_score = idx, score
        if best_idx is not None:
            ranked[frame_idx] = {
                'bbox': candidates[best_idx]['bbox'], 'score': float(best_score)
            }
            if include_debug:
                scored[best_idx]['is_best'] = True
                debug[frame_idx] = scored
    return ranked, debug


def rank_top_k_video_frames(video_cache, weights, top_k):
    # Stable candidate indices preserve the legacy first-candidate tie break.
    ranked = {}
    has_siamese = bool(video_cache['has_siamese'])
    has_color = bool(video_cache['has_color'])
    for frame_key, candidates in video_cache['frames'].items():
        frame_idx = int(frame_key)
        scored = []
        for candidate_idx, candidate in enumerate(candidates):
            score = fuse_candidate_score(
                candidate['scores'], weights, has_siamese, has_color
            )
            scored.append({
                'bbox': candidate['bbox'],
                'score': float(score),
                'scores': candidate['scores'],
                'candidate_idx': candidate_idx,
            })
        scored.sort(key=lambda item: (-item['score'], item['candidate_idx']))
        if scored:
            ranked[frame_idx] = scored[:top_k]
    return ranked


def candidate_motion_affinity(candidate1, candidate2, frame_steps, motion_scale):
    if motion_scale <= 0:
        raise ValueError('motion_scale must be positive')
    center_speed, _ = gap_motion_metrics(
        candidate1['bbox'], candidate2['bbox'], frame_steps
    )
    return math.exp(-0.5 * (center_speed / motion_scale) ** 2)


def candidate_directional_support(candidate, frame_idx, candidates_by_frame,
                                  neighbor_radius, motion_scale, direction):
    best_support = 0.0
    for offset in range(1, neighbor_radius + 1):
        neighbor_frame = frame_idx + direction * offset
        for neighbor in candidates_by_frame.get(neighbor_frame, []):
            affinity = candidate_motion_affinity(
                candidate, neighbor, offset, motion_scale
            )
            # Query confidence prevents weak background tracks from giving full support.
            support = affinity * max(0.0, min(1.0, neighbor['score']))
            if support > best_support:
                best_support = support
    return best_support


def rerank_video_candidates(candidates_by_frame, neighbor_radius, rerank_weight,
                            motion_scale=RERANK_MOTION_SCALE):
    selected = {}
    for frame_idx in sorted(candidates_by_frame):
        best_item, best_contextual_score = None, -float('inf')
        for candidate in candidates_by_frame[frame_idx]:
            past = candidate_directional_support(
                candidate, frame_idx, candidates_by_frame, neighbor_radius,
                motion_scale, direction=-1,
            )
            future = candidate_directional_support(
                candidate, frame_idx, candidates_by_frame, neighbor_radius,
                motion_scale, direction=1,
            )
            temporal_support = 0.5 * (past + future)
            contextual_score = candidate['score'] + rerank_weight * temporal_support
            # Strict > preserves the stable top-k order on exact ties and at weight zero.
            if contextual_score > best_contextual_score:
                best_item = candidate
                best_contextual_score = contextual_score
        if best_item is not None:
            selected[frame_idx] = {
                'bbox': best_item['bbox'],
                # Acceptance always uses the original fusion score, never context score.
                'score': best_item['score'],
                'contextual_score': float(best_contextual_score),
                'candidate_idx': best_item['candidate_idx'],
            }
    return selected


def identity_id_from_video_id(video_id):
    # Evaluation grouping only; this identity is never read by candidate scoring.
    if '_' not in video_id:
        return video_id
    identity_id, suffix = video_id.rsplit('_', 1)
    return identity_id if suffix.isdigit() else video_id


def build_identity_folds(video_ids):
    groups = {}
    for video_id in video_ids:
        groups.setdefault(identity_id_from_video_id(video_id), []).append(video_id)
    return {identity: sorted(videos) for identity, videos in sorted(groups.items())}


def render_ranked_predictions(cache, ranked_by_video, gt_map, matching_threshold,
                              temporal_config=None):
    all_predictions, per_video, detected_frames = [], {}, 0
    for vid_id in cache['signature']['video_folders']:
        video_cache = cache['videos'][vid_id]
        threshold = matching_threshold if video_cache['has_siamese'] else YOLO_ONLY_THRESHOLD
        accepted = {
            frame_idx: item for frame_idx, item in ranked_by_video[vid_id].items()
            if item['score'] >= threshold
        }
        if not accepted:
            final_bboxes = []
        elif temporal_config is None:
            final_bboxes = temporal_smooth_detections(
                accepted, video_cache['max_frame_idx'],
                min_seg_len=TEMPORAL_MIN_SEG, max_gap=TEMPORAL_MAX_GAP,
            )
        else:
            final_bboxes = temporal_smooth_detections_gated(
                accepted, video_cache['max_frame_idx'],
                min_seg_len=temporal_config['min_seg_len'],
                max_gap=temporal_config['max_gap'],
                max_center_speed=temporal_config['max_center_speed'],
                max_log_scale_speed=temporal_config['max_log_scale_speed'],
            )
        all_predictions.append({
            'video_id': vid_id,
            'detections': [{'bboxes': final_bboxes}] if final_bboxes else [],
        })
        pred_frames = {
            b['frame']: [b['x1'], b['y1'], b['x2'], b['y2']] for b in final_bboxes
        }
        per_video[vid_id] = compute_st_iou_video(gt_map.get(vid_id, {}), pred_frames)
        detected_frames += len(final_bboxes)
    mean_score = sum(per_video.values()) / max(1, len(per_video))
    return {
        'mean_st_iou': mean_score,
        'per_video': per_video,
        'detected_frames': detected_frames,
        'predictions': all_predictions,
    }


def evaluate_config(cache, gt_map, weights, matching_threshold, include_debug=False,
                    temporal_config=None):
    ranked, debug = {}, {}
    for vid_id, video_cache in cache['videos'].items():
        ranked[vid_id], video_debug = rank_video_frames(video_cache, weights, include_debug)
        if include_debug:
            threshold = matching_threshold if video_cache['has_siamese'] else YOLO_ONLY_THRESHOLD
            for frame_idx, candidates in video_debug.items():
                if ranked[vid_id][frame_idx]['score'] < threshold:
                    for candidate in candidates:
                        candidate['is_best'] = False
            debug[vid_id] = video_debug
    result = render_ranked_predictions(
        cache, ranked, gt_map, matching_threshold, temporal_config=temporal_config
    )
    result['debug_info'] = debug
    return result


def calibration_weight_grid():
    configs = []
    for ws in CALIBRATION_SIAMESE_WEIGHTS:
        for wc in CALIBRATION_COLOR_WEIGHTS:
            wy = round(1.0 - ws - wc, 10)
            if wy + 1e-12 < CALIBRATION_MIN_YOLO_WEIGHT:
                continue
            configs.append({'yolo': wy, 'siamese': ws, 'color': wc})
    default = {'yolo': WEIGHT_YOLO, 'siamese': WEIGHT_SIAMESE, 'color': WEIGHT_COLOR}
    if default not in configs:
        configs.append(default)
    return configs


def result_row(weights, threshold, result):
    row = {
        'weight_yolo': weights['yolo'],
        'weight_siamese': weights['siamese'],
        'weight_color': weights['color'],
        'matching_threshold': threshold,
        'mean_st_iou': result['mean_st_iou'],
        'detected_frames': result['detected_frames'],
    }
    for vid, score in sorted(result['per_video'].items()):
        row[f'st_iou__{vid}'] = score
    return row


def select_best_row(rows):
    # Primary objective is mean ST-IoU. Exact ties prefer the setting closest to baseline.
    def key(row):
        distance = (
            abs(row['weight_yolo'] - WEIGHT_YOLO)
            + abs(row['weight_siamese'] - WEIGHT_SIAMESE)
            + abs(row['weight_color'] - WEIGHT_COLOR)
            + abs(row['matching_threshold'] - MATCHING_THRESHOLD)
        )
        return row['mean_st_iou'], -distance
    return max(rows, key=key)


def run_calibration(cache, gt_map):
    rows = []
    for weights in tqdm(calibration_weight_grid(), desc='Weight grid'):
        ranked = {
            vid: rank_video_frames(video_cache, weights, include_debug=False)[0]
            for vid, video_cache in cache['videos'].items()
        }
        for threshold in CALIBRATION_THRESHOLDS:
            result = render_ranked_predictions(cache, ranked, gt_map, threshold)
            rows.append(result_row(weights, threshold, result))

    best = select_best_row(rows)
    rows_sorted = sorted(rows, key=lambda r: r['mean_st_iou'], reverse=True)
    os.makedirs(CALIBRATION_DIR, exist_ok=True)
    with open(CALIBRATION_CSV + '.tmp', 'w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
        writer.writeheader()
        writer.writerows(rows_sorted)
    os.replace(CALIBRATION_CSV + '.tmp', CALIBRATION_CSV)

    best_weights = {
        'yolo': best['weight_yolo'],
        'siamese': best['weight_siamese'],
        'color': best['weight_color'],
    }
    best_result = evaluate_config(
        cache, gt_map, best_weights, best['matching_threshold'], include_debug=True
    )
    atomic_write_json(CALIBRATED_OUTPUT_FILE, best_result['predictions'])
    return rows_sorted, best, best_result


def temporal_config_grid():
    return [
        {
            'max_gap': max_gap,
            'min_seg_len': min_seg_len,
            'max_center_speed': max_center_speed,
            'max_log_scale_speed': max_log_scale_speed,
        }
        for max_gap in TEMPORAL_MAX_GAP_GRID
        for min_seg_len in TEMPORAL_MIN_SEG_GRID
        for max_center_speed in TEMPORAL_CENTER_SPEED_GRID
        for max_log_scale_speed in TEMPORAL_LOG_SCALE_SPEED_GRID
    ]


def temporal_result_row(config, result):
    row = {
        **config,
        'mean_st_iou': result['mean_st_iou'],
        'detected_frames': result['detected_frames'],
    }
    for vid, score in sorted(result['per_video'].items()):
        row[f'st_iou__{vid}'] = score
    return row


def select_best_temporal_row(rows):
    # Exact metric ties prefer the legacy temporal configuration.
    def key(row):
        gate_penalty = int(row['max_center_speed'] is not None) + int(
            row['max_log_scale_speed'] is not None
        )
        distance = (
            abs(row['max_gap'] - TEMPORAL_MAX_GAP)
            + abs(row['min_seg_len'] - TEMPORAL_MIN_SEG)
            + gate_penalty
        )
        return row['mean_st_iou'], -distance
    return max(rows, key=key)


def run_temporal_calibration(cache, gt_map, weights, matching_threshold):
    ranked = {
        vid: rank_video_frames(video_cache, weights, include_debug=False)[0]
        for vid, video_cache in cache['videos'].items()
    }
    rows = []
    for config in tqdm(temporal_config_grid(), desc='Temporal grid'):
        result = render_ranked_predictions(
            cache, ranked, gt_map, matching_threshold, temporal_config=config
        )
        rows.append(temporal_result_row(config, result))

    rows_sorted = sorted(rows, key=lambda row: row['mean_st_iou'], reverse=True)
    best = select_best_temporal_row(rows)
    best_config = {
        key: best[key]
        for key in ('max_gap', 'min_seg_len', 'max_center_speed', 'max_log_scale_speed')
    }
    best_result = render_ranked_predictions(
        cache, ranked, gt_map, matching_threshold, temporal_config=best_config
    )
    os.makedirs(TEMPORAL_OUTPUT_DIR, exist_ok=True)
    with open(TEMPORAL_RESULTS_CSV + '.tmp', 'w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
        writer.writeheader()
        writer.writerows(rows_sorted)
    os.replace(TEMPORAL_RESULTS_CSV + '.tmp', TEMPORAL_RESULTS_CSV)
    atomic_write_json(TEMPORAL_PREDICTIONS_FILE, best_result['predictions'])
    return rows_sorted, best_config, best_result


def reranking_config_grid():
    return [
        {
            'top_k': top_k,
            'neighbor_radius': neighbor_radius,
            'rerank_weight': rerank_weight,
            'motion_scale': RERANK_MOTION_SCALE,
        }
        for top_k in RERANK_TOP_K_GRID
        for neighbor_radius in RERANK_NEIGHBOR_RADIUS_GRID
        for rerank_weight in RERANK_WEIGHT_GRID
    ]


def evaluate_reranking_config(cache, gt_map, weights, matching_threshold, config):
    ranked = {}
    for video_id, video_cache in cache['videos'].items():
        top_candidates = rank_top_k_video_frames(
            video_cache, weights, top_k=config['top_k']
        )
        ranked[video_id] = rerank_video_candidates(
            top_candidates,
            neighbor_radius=config['neighbor_radius'],
            rerank_weight=config['rerank_weight'],
            motion_scale=config['motion_scale'],
        )
    return render_ranked_predictions(
        cache, ranked, gt_map, matching_threshold,
        temporal_config=PRODUCTION_TEMPORAL_CONFIG,
    )


def reranking_result_row(config, result):
    row = {
        **config,
        'mean_st_iou': result['mean_st_iou'],
        'detected_frames': result['detected_frames'],
    }
    for video_id, score in sorted(result['per_video'].items()):
        row[f'st_iou__{video_id}'] = score
    return row


def mean_row_score(row, video_ids):
    scores = [row[f'st_iou__{video_id}'] for video_id in video_ids]
    return sum(scores) / len(scores) if scores else 0.0


def reranking_config_from_row(row):
    return {
        key: row[key]
        for key in ('top_k', 'neighbor_radius', 'rerank_weight', 'motion_scale')
    }


def select_reranking_row(rows, selection_video_ids):
    # Exact validation ties prefer the least contextual intervention.
    def key(row):
        return (
            mean_row_score(row, selection_video_ids),
            -row['rerank_weight'], -row['top_k'], -row['neighbor_radius'],
        )
    return max(rows, key=key)


def select_voted_reranking_config(fold_rows):
    vote_counts = {}
    for fold in fold_rows:
        key = (
            fold['top_k'], fold['neighbor_radius'],
            fold['rerank_weight'], fold['motion_scale'],
        )
        vote_counts[key] = vote_counts.get(key, 0) + 1
    # Vote ties prefer the lowest weight, then the smaller candidate/radius budget.
    best_key = max(
        vote_counts,
        key=lambda key: (vote_counts[key], -key[2], -key[0], -key[1]),
    )
    return {
        'top_k': best_key[0], 'neighbor_radius': best_key[1],
        'rerank_weight': best_key[2], 'motion_scale': best_key[3],
    }, vote_counts[best_key]


def run_reranking_identity_cv(cache, gt_map, weights, matching_threshold, baseline_result):
    configs = reranking_config_grid()
    rows, results_by_config = [], {}
    for config in tqdm(configs, desc='Top-k reranking grid'):
        result = evaluate_reranking_config(
            cache, gt_map, weights, matching_threshold, config
        )
        row = reranking_result_row(config, result)
        rows.append(row)
        results_by_config[tuple(config.values())] = result

    control_config = {
        'top_k': min(RERANK_TOP_K_GRID),
        'neighbor_radius': min(RERANK_NEIGHBOR_RADIUS_GRID),
        'rerank_weight': 0.0,
        'motion_scale': RERANK_MOTION_SCALE,
    }
    control_result = results_by_config[tuple(control_config.values())]
    if control_result['predictions'] != baseline_result['predictions']:
        raise RuntimeError(
            'Reranking weight-zero control does not exactly match production predictions'
        )

    identity_folds = build_identity_folds(cache['signature']['video_folders'])
    all_video_ids = sorted(cache['signature']['video_folders'])
    fold_rows = []
    for held_identity, held_video_ids in identity_folds.items():
        selection_video_ids = [
            video_id for video_id in all_video_ids if video_id not in held_video_ids
        ]
        selected = select_reranking_row(rows, selection_video_ids)
        heldout_score = mean_row_score(selected, held_video_ids)
        baseline_score = sum(
            baseline_result['per_video'][video_id] for video_id in held_video_ids
        ) / len(held_video_ids)
        fold_rows.append({
            'heldout_identity': held_identity,
            'heldout_videos': '|'.join(held_video_ids),
            'selection_videos': '|'.join(selection_video_ids),
            **reranking_config_from_row(selected),
            'selection_mean_st_iou': mean_row_score(selected, selection_video_ids),
            'heldout_mean_st_iou': heldout_score,
            'baseline_heldout_mean_st_iou': baseline_score,
            'heldout_delta': heldout_score - baseline_score,
        })

    recommended_config, recommendation_votes = select_voted_reranking_config(fold_rows)
    recommended_key = tuple(recommended_config.values())
    recommended_result = results_by_config[recommended_key]
    mean_cv_score = sum(row['heldout_mean_st_iou'] for row in fold_rows) / len(fold_rows)
    mean_cv_baseline = sum(
        row['baseline_heldout_mean_st_iou'] for row in fold_rows
    ) / len(fold_rows)
    cv_delta = mean_cv_score - mean_cv_baseline
    improved_identities = sum(row['heldout_delta'] > 0 for row in fold_rows)
    worst_identity_delta = min(row['heldout_delta'] for row in fold_rows)
    promotion_passed = (
        cv_delta >= RERANK_MIN_MEAN_CV_DELTA
        and improved_identities >= RERANK_MIN_IMPROVED_IDENTITIES
        and worst_identity_delta >= -RERANK_MAX_WORST_IDENTITY_DROP
        and recommended_config['rerank_weight'] > 0
    )
    diagnostics = {
        'weight_zero_control_exact_prediction_parity': True,
        'identity_folds': identity_folds,
        'mean_cv_st_iou': mean_cv_score,
        'mean_cv_baseline_st_iou': mean_cv_baseline,
        'mean_cv_delta': cv_delta,
        'improved_identity_count': improved_identities,
        'worst_identity_delta': worst_identity_delta,
        'promotion_passed': promotion_passed,
        'recommended_config': recommended_config,
        'recommendation_votes': recommendation_votes,
    }
    rows_sorted = sorted(rows, key=lambda row: row['mean_st_iou'], reverse=True)
    return rows_sorted, fold_rows, diagnostics, recommended_result


def tracklet_graph_config_grid():
    return [
        {
            'top_k': top_k,
            'motion_weight': motion_weight,
            'state_change_penalty': state_change_penalty,
            'max_center_speed': max_center_speed,
            'camera_compensation': camera_compensation,
            'emission_scale': TRACKLET_GRAPH_EMISSION_SCALE,
        }
        for top_k in TRACKLET_GRAPH_TOP_K_GRID
        for motion_weight in TRACKLET_GRAPH_MOTION_WEIGHT_GRID
        for state_change_penalty in TRACKLET_GRAPH_STATE_CHANGE_PENALTY_GRID
        for max_center_speed in TRACKLET_GRAPH_MAX_CENTER_SPEED_GRID
        for camera_compensation in TRACKLET_GRAPH_CAMERA_COMPENSATION_GRID
    ]


def tracklet_graph_config_key(config):
    return (
        int(config['top_k']), float(config['motion_weight']),
        float(config['state_change_penalty']), float(config['max_center_speed']),
        bool(config['camera_compensation']), float(config['emission_scale']),
    )


def median_value(values):
    ordered = sorted(values)
    if not ordered:
        return 0.0
    middle = len(ordered) // 2
    if len(ordered) % 2:
        return float(ordered[middle])
    return 0.5 * (float(ordered[middle - 1]) + float(ordered[middle]))


def bbox_center(bbox):
    return (
        0.5 * (float(bbox[0]) + float(bbox[2])),
        0.5 * (float(bbox[1]) + float(bbox[3])),
    )


def estimate_candidate_global_shift(previous_candidates, current_candidates,
                                    max_shift, min_matches):
    """Estimate a conservative translation proxy from mutual nearest candidates.

    This is intentionally cache-only and is not a replacement for optical-flow or
    homography camera-motion compensation. Insufficient/ambiguous matches return zero.
    """
    if not previous_candidates or not current_candidates:
        return 0.0, 0.0, 0
    previous_centers = [bbox_center(item['bbox']) for item in previous_candidates]
    current_centers = [bbox_center(item['bbox']) for item in current_candidates]

    nearest_current = []
    for px, py in previous_centers:
        nearest_current.append(min(
            range(len(current_centers)),
            key=lambda idx: (
                (current_centers[idx][0] - px) ** 2
                + (current_centers[idx][1] - py) ** 2,
                idx,
            ),
        ))
    nearest_previous = []
    for cx, cy in current_centers:
        nearest_previous.append(min(
            range(len(previous_centers)),
            key=lambda idx: (
                (previous_centers[idx][0] - cx) ** 2
                + (previous_centers[idx][1] - cy) ** 2,
                idx,
            ),
        ))

    displacements = []
    for previous_idx, current_idx in enumerate(nearest_current):
        if nearest_previous[current_idx] != previous_idx:
            continue
        px, py = previous_centers[previous_idx]
        cx, cy = current_centers[current_idx]
        dx, dy = cx - px, cy - py
        if math.hypot(dx, dy) <= max_shift:
            displacements.append((dx, dy))
    if len(displacements) < min_matches:
        return 0.0, 0.0, len(displacements)
    return (
        median_value([item[0] for item in displacements]),
        median_value([item[1] for item in displacements]),
        len(displacements),
    )


def shifted_bbox(bbox, dx, dy):
    return [
        float(bbox[0]) + dx, float(bbox[1]) + dy,
        float(bbox[2]) + dx, float(bbox[3]) + dy,
    ]


def tracklet_transition_score(previous, current, config, global_shift):
    penalty = float(config['state_change_penalty'])
    if previous is None and current is None:
        return 0.0
    if previous is None or current is None:
        return -penalty
    previous_bbox = previous['bbox']
    if config['camera_compensation']:
        previous_bbox = shifted_bbox(
            previous_bbox, global_shift[0], global_shift[1]
        )
    center_speed, _ = gap_motion_metrics(previous_bbox, current['bbox'], 1)
    max_speed = float(config['max_center_speed'])
    if center_speed > max_speed:
        return -float('inf')
    normalized_motion = center_speed / max(1e-12, max_speed)
    return -float(config['motion_weight']) * normalized_motion ** 2


def decode_candidate_tracklet(video_cache, weights, matching_threshold, config):
    """Viterbi-decode one candidate or ABSENT for every frame."""
    candidates_by_frame = rank_top_k_video_frames(
        video_cache, weights, top_k=config['top_k']
    )
    max_frame_idx = int(video_cache['max_frame_idx'])
    if max_frame_idx < 0:
        return {}, {
            'total_frames': 0, 'candidate_frames': 0, 'absent_frames': 0,
            'rescued_below_threshold': 0, 'suppressed_above_threshold': 0,
            'camera_shift_frames': 0, 'mean_camera_shift_pixels': 0.0,
        }
    threshold = (
        matching_threshold if video_cache['has_siamese'] else YOLO_ONLY_THRESHOLD
    )
    states_by_frame, backpointers, score_history = [], [], []
    camera_shifts = {}
    shift_magnitudes = []
    for frame_idx in range(1, max_frame_idx + 1):
        if not config['camera_compensation']:
            camera_shifts[frame_idx] = (0.0, 0.0, 0)
            continue
        shift = estimate_candidate_global_shift(
            candidates_by_frame.get(frame_idx - 1, []),
            candidates_by_frame.get(frame_idx, []),
            max_shift=TRACKLET_GRAPH_MAX_GLOBAL_SHIFT,
            min_matches=TRACKLET_GRAPH_MIN_GLOBAL_SHIFT_MATCHES,
        )
        camera_shifts[frame_idx] = shift
        if shift[2] >= TRACKLET_GRAPH_MIN_GLOBAL_SHIFT_MATCHES:
            shift_magnitudes.append(math.hypot(shift[0], shift[1]))

    for frame_idx in range(max_frame_idx + 1):
        # ABSENT is index zero; stable ranked candidates follow.
        states = [None] + list(candidates_by_frame.get(frame_idx, []))
        states_by_frame.append(states)
        emissions = [0.0] + [
            float(config['emission_scale']) * (candidate['score'] - threshold)
            for candidate in states[1:]
        ]
        if frame_idx == 0:
            scores = [0.0] + [
                emission - float(config['state_change_penalty'])
                for emission in emissions[1:]
            ]
            parents = [-1] * len(states)
        else:
            previous_states = states_by_frame[frame_idx - 1]
            previous_scores = score_history[-1]
            shift = camera_shifts.get(frame_idx, (0.0, 0.0, 0))
            scores, parents = [], []
            for current_idx, current in enumerate(states):
                best_score, best_previous_idx = -float('inf'), 0
                for previous_idx, previous in enumerate(previous_states):
                    transition = tracklet_transition_score(
                        previous, current, config, shift
                    )
                    candidate_score = previous_scores[previous_idx] + transition
                    # Strict comparison preserves deterministic state-order ties.
                    if candidate_score > best_score:
                        best_score = candidate_score
                        best_previous_idx = previous_idx
                scores.append(best_score + emissions[current_idx])
                parents.append(best_previous_idx)
        score_history.append(scores)
        backpointers.append(parents)

    terminal_scores = [
        score - (float(config['state_change_penalty']) if state is not None else 0.0)
        for score, state in zip(score_history[-1], states_by_frame[-1])
    ]
    state_idx = max(range(len(terminal_scores)), key=lambda idx: terminal_scores[idx])
    decoded_states = [None] * (max_frame_idx + 1)
    for frame_idx in range(max_frame_idx, -1, -1):
        decoded_states[frame_idx] = states_by_frame[frame_idx][state_idx]
        state_idx = backpointers[frame_idx][state_idx]

    selected = {}
    rescued_below_threshold = 0
    for frame_idx, candidate in enumerate(decoded_states):
        if candidate is None:
            continue
        selected[frame_idx] = {
            'bbox': candidate['bbox'], 'score': candidate['score'],
            'candidate_idx': candidate['candidate_idx'],
        }
        rescued_below_threshold += candidate['score'] < threshold
    top1 = {
        frame_idx: candidates[0]
        for frame_idx, candidates in candidates_by_frame.items() if candidates
    }
    suppressed_above_threshold = sum(
        item['score'] >= threshold and frame_idx not in selected
        for frame_idx, item in top1.items()
    )
    diagnostics = {
        'total_frames': max_frame_idx + 1,
        'candidate_frames': len(selected),
        'absent_frames': max_frame_idx + 1 - len(selected),
        'rescued_below_threshold': int(rescued_below_threshold),
        'suppressed_above_threshold': int(suppressed_above_threshold),
        'camera_shift_frames': len(shift_magnitudes),
        'mean_camera_shift_pixels': (
            sum(shift_magnitudes) / len(shift_magnitudes)
            if shift_magnitudes else 0.0
        ),
    }
    return selected, diagnostics


def render_decoded_tracklets(cache, decoded_by_video, gt_map, temporal_config):
    all_predictions, per_video, detected_frames = [], {}, 0
    for video_id in cache['signature']['video_folders']:
        video_cache = cache['videos'][video_id]
        decoded = decoded_by_video.get(video_id, {})
        final_bboxes = temporal_smooth_detections_gated(
            decoded, video_cache['max_frame_idx'],
            min_seg_len=temporal_config['min_seg_len'],
            max_gap=temporal_config['max_gap'],
            max_center_speed=temporal_config['max_center_speed'],
            max_log_scale_speed=temporal_config['max_log_scale_speed'],
        ) if decoded else []
        all_predictions.append({
            'video_id': video_id,
            'detections': [{'bboxes': final_bboxes}] if final_bboxes else [],
        })
        pred_frames = {
            bbox['frame']: [bbox['x1'], bbox['y1'], bbox['x2'], bbox['y2']]
            for bbox in final_bboxes
        }
        per_video[video_id] = compute_st_iou_video(
            gt_map.get(video_id, {}), pred_frames
        )
        detected_frames += len(final_bboxes)
    return {
        'mean_st_iou': sum(per_video.values()) / max(1, len(per_video)),
        'per_video': per_video,
        'detected_frames': detected_frames,
        'predictions': all_predictions,
    }


def evaluate_tracklet_graph_config(cache, gt_map, weights, matching_threshold,
                                   config, temporal_config):
    start_time = time.perf_counter()
    decoded_by_video, per_video_diagnostics = {}, {}
    for video_id, video_cache in cache['videos'].items():
        decoded_by_video[video_id], per_video_diagnostics[video_id] = (
            decode_candidate_tracklet(
                video_cache, weights, matching_threshold, config
            )
        )
    result = render_decoded_tracklets(
        cache, decoded_by_video, gt_map, temporal_config
    )
    elapsed = max(1e-12, time.perf_counter() - start_time)
    total_frames = sum(
        max(0, int(video['max_frame_idx']) + 1)
        for video in cache['videos'].values()
    )
    result['graph_diagnostics'] = {
        'elapsed_seconds': elapsed,
        'replay_fps': total_frames / elapsed,
        'total_frames': total_frames,
        'candidate_frames_before_temporal': sum(
            item['candidate_frames'] for item in per_video_diagnostics.values()
        ),
        'absent_frames_before_temporal': sum(
            item['absent_frames'] for item in per_video_diagnostics.values()
        ),
        'rescued_below_threshold': sum(
            item['rescued_below_threshold'] for item in per_video_diagnostics.values()
        ),
        'suppressed_above_threshold': sum(
            item['suppressed_above_threshold'] for item in per_video_diagnostics.values()
        ),
        'camera_shift_frames': sum(
            item['camera_shift_frames'] for item in per_video_diagnostics.values()
        ),
        'per_video': per_video_diagnostics,
    }
    return result


def tracklet_graph_result_row(config, result):
    diagnostics = result['graph_diagnostics']
    row = {
        **config,
        'mean_st_iou': result['mean_st_iou'],
        'detected_frames': result['detected_frames'],
        'replay_fps': diagnostics['replay_fps'],
        'candidate_frames_before_temporal': diagnostics['candidate_frames_before_temporal'],
        'rescued_below_threshold': diagnostics['rescued_below_threshold'],
        'suppressed_above_threshold': diagnostics['suppressed_above_threshold'],
        'camera_shift_frames': diagnostics['camera_shift_frames'],
    }
    for video_id, score in sorted(result['per_video'].items()):
        row[f'st_iou__{video_id}'] = score
    return row


def tracklet_graph_config_from_row(row):
    return {
        key: row[key] for key in (
            'top_k', 'motion_weight', 'state_change_penalty',
            'max_center_speed', 'camera_compensation', 'emission_scale',
        )
    }


def select_tracklet_graph_row(rows, selection_video_ids):
    # Exact metric ties prefer less intervention and no translation proxy.
    def key(row):
        return (
            mean_row_score(row, selection_video_ids),
            -float(row['motion_weight']), -float(row['state_change_penalty']),
            -int(row['top_k']), not bool(row['camera_compensation']),
            -float(row['max_center_speed']),
        )
    return max(rows, key=key)


def select_voted_tracklet_graph_config(fold_rows):
    votes = {}
    for row in fold_rows:
        config = tracklet_graph_config_from_row(row)
        key = tracklet_graph_config_key(config)
        votes[key] = votes.get(key, 0) + 1
    best_key = max(
        votes,
        key=lambda key: (
            votes[key], -key[1], -key[2], -key[0], not key[4], -key[3]
        ),
    )
    return {
        'top_k': best_key[0], 'motion_weight': best_key[1],
        'state_change_penalty': best_key[2], 'max_center_speed': best_key[3],
        'camera_compensation': best_key[4], 'emission_scale': best_key[5],
    }, votes[best_key]


def run_tracklet_graph_identity_cv(cache, gt_map, weights, matching_threshold,
                                   baseline_result, temporal_config):
    baseline_snapshot = json.dumps(
        baseline_result['predictions'], sort_keys=True, separators=(',', ':')
    )
    control_result = evaluate_config(
        cache, gt_map, weights, matching_threshold,
        include_debug=False, temporal_config=temporal_config,
    )
    if control_result['predictions'] != baseline_result['predictions']:
        raise RuntimeError(
            'Tracklet graph production control does not exactly reproduce predictions'
        )
    rows, results_by_config = [], {}
    for config in tqdm(tracklet_graph_config_grid(), desc='Tracklet graph grid'):
        result = evaluate_tracklet_graph_config(
            cache, gt_map, weights, matching_threshold, config, temporal_config
        )
        rows.append(tracklet_graph_result_row(config, result))
        results_by_config[tracklet_graph_config_key(config)] = result
    if json.dumps(
        baseline_result['predictions'], sort_keys=True, separators=(',', ':')
    ) != baseline_snapshot:
        raise RuntimeError('Tracklet graph experiment mutated production predictions')

    identity_folds = build_identity_folds(cache['signature']['video_folders'])
    all_video_ids = sorted(cache['signature']['video_folders'])
    fold_rows = []
    for held_identity, held_video_ids in identity_folds.items():
        selection_video_ids = [
            video_id for video_id in all_video_ids if video_id not in held_video_ids
        ]
        selected = select_tracklet_graph_row(rows, selection_video_ids)
        heldout_score = mean_row_score(selected, held_video_ids)
        baseline_score = sum(
            baseline_result['per_video'][video_id] for video_id in held_video_ids
        ) / len(held_video_ids)
        fold_rows.append({
            'heldout_identity': held_identity,
            'heldout_videos': '|'.join(held_video_ids),
            'selection_videos': '|'.join(selection_video_ids),
            **tracklet_graph_config_from_row(selected),
            'selection_mean_st_iou': mean_row_score(selected, selection_video_ids),
            'heldout_mean_st_iou': heldout_score,
            'baseline_heldout_mean_st_iou': baseline_score,
            'heldout_delta': heldout_score - baseline_score,
        })

    recommended_config, recommendation_votes = (
        select_voted_tracklet_graph_config(fold_rows)
    )
    recommended_result = results_by_config[
        tracklet_graph_config_key(recommended_config)
    ]
    mean_cv_score = sum(
        row['heldout_mean_st_iou'] for row in fold_rows
    ) / len(fold_rows)
    mean_cv_baseline = sum(
        row['baseline_heldout_mean_st_iou'] for row in fold_rows
    ) / len(fold_rows)
    deltas = [row['heldout_delta'] for row in fold_rows]
    full_delta = recommended_result['mean_st_iou'] - baseline_result['mean_st_iou']
    replay_fps = recommended_result['graph_diagnostics']['replay_fps']
    promotion_passed = (
        mean_cv_score - mean_cv_baseline >= TRACKLET_GRAPH_MIN_MEAN_CV_DELTA
        and sum(delta > 0 for delta in deltas)
            >= TRACKLET_GRAPH_MIN_IMPROVED_IDENTITIES
        and min(deltas) >= -TRACKLET_GRAPH_MAX_WORST_IDENTITY_DROP
        and recommendation_votes >= TRACKLET_GRAPH_MIN_CONFIG_VOTES
        and full_delta >= TRACKLET_GRAPH_MIN_MEAN_CV_DELTA
        and replay_fps >= TRACKLET_GRAPH_MIN_REPLAY_FPS
    )
    diagnostics = {
        'production_control_exact_prediction_parity': True,
        'production_control_untouched': True,
        'identity_folds': identity_folds,
        'mean_cv_st_iou': mean_cv_score,
        'mean_cv_baseline_st_iou': mean_cv_baseline,
        'mean_cv_delta': mean_cv_score - mean_cv_baseline,
        'improved_identity_count': sum(delta > 0 for delta in deltas),
        'worst_identity_delta': min(deltas),
        'recommended_config': recommended_config,
        'recommendation_votes': recommendation_votes,
        'recommended_full_public_delta': full_delta,
        'recommended_replay_fps': replay_fps,
        'promotion_passed': promotion_passed,
    }
    video_rows = [
        {
            'video_id': video_id,
            'identity_id': identity_id_from_video_id(video_id),
            'production_st_iou': baseline_result['per_video'][video_id],
            'tracklet_graph_st_iou': recommended_result['per_video'][video_id],
            'delta': (
                recommended_result['per_video'][video_id]
                - baseline_result['per_video'][video_id]
            ),
            **recommended_result['graph_diagnostics']['per_video'][video_id],
        }
        for video_id in all_video_ids
    ]
    rows_sorted = sorted(rows, key=lambda row: row['mean_st_iou'], reverse=True)
    return rows_sorted, fold_rows, video_rows, diagnostics, recommended_result


def prediction_frame_maps(predictions):
    output = {}
    for prediction in predictions:
        frame_map = {}
        for detection in prediction.get('detections', []):
            for bbox in detection.get('bboxes', []):
                frame_map[int(bbox['frame'])] = [
                    bbox['x1'], bbox['y1'], bbox['x2'], bbox['y2']
                ]
        output[prediction['video_id']] = frame_map
    return output


def oracle_candidate_for_gt(candidates, gt_bbox):
    best_idx, best_iou = None, 0.0
    for candidate_idx, candidate in enumerate(candidates):
        candidate_iou = iou_box(candidate['bbox'], gt_bbox)
        if candidate_iou > best_iou:
            best_idx, best_iou = candidate_idx, candidate_iou
    return best_idx, best_iou


def classify_oracle_gt_frame(candidate_count, oracle_iou, selected_iou,
                             accepted, final_present, final_iou,
                             good_iou=ORACLE_GOOD_IOU):
    # Mutually exclusive first-failure hierarchy; categories sum to GT frame count.
    if candidate_count == 0:
        return 'no_candidate'
    if oracle_iou < good_iou:
        return 'detector_localization_miss'
    if selected_iou < good_iou:
        return 'matching_error'
    if not accepted:
        return 'threshold_rejection'
    if not final_present:
        return 'temporal_removal'
    if final_iou < good_iou:
        return 'temporal_localization_degradation'
    return 'success'


def analyze_oracle_video(video_id, video_cache, gt_frames, ranked_frames,
                         final_frames, weights, matching_threshold):
    threshold = matching_threshold if video_cache['has_siamese'] else YOLO_ONLY_THRESHOLD
    selected_frames = {
        frame_idx: item['bbox'] for frame_idx, item in ranked_frames.items()
    }
    accepted_frames = {
        frame_idx: item['bbox'] for frame_idx, item in ranked_frames.items()
        if item['score'] >= threshold
    }
    oracle_frames = {}
    analysis_frames = sorted(
        set(gt_frames) | set(ranked_frames) | set(final_frames)
    )
    frame_rows = []
    failure_counts = {category: 0 for category in ORACLE_GT_FAILURE_CATEGORIES}
    oracle_iou_sum = selected_iou_sum = accepted_iou_sum = final_iou_sum = 0.0
    matching_regret_sum = 0.0
    recall_counts = {
        stage: {threshold_value: 0 for threshold_value in ORACLE_IOU_THRESHOLDS}
        for stage in ('oracle', 'selected', 'accepted', 'final')
    }
    temporal_helped = temporal_harmed = temporal_unchanged = 0

    for frame_idx in analysis_frames:
        candidates = video_cache['frames'].get(str(frame_idx), [])
        gt_bbox = gt_frames.get(frame_idx)
        selected = ranked_frames.get(frame_idx)
        selected_bbox = selected['bbox'] if selected is not None else None
        selected_score = selected['score'] if selected is not None else None
        accepted = selected is not None and selected_score >= threshold
        final_bbox = final_frames.get(frame_idx)
        has_gt = gt_bbox is not None
        oracle_idx, oracle_iou = (None, 0.0)
        oracle_score = None
        selected_iou = accepted_iou = final_iou = 0.0
        failure_category = ''

        if has_gt:
            oracle_idx, oracle_iou = oracle_candidate_for_gt(candidates, gt_bbox)
            if oracle_idx is not None:
                oracle_candidate = candidates[oracle_idx]
                oracle_frames[frame_idx] = oracle_candidate['bbox']
                oracle_score = fuse_candidate_score(
                    oracle_candidate['scores'], weights,
                    bool(video_cache['has_siamese']), bool(video_cache['has_color']),
                )
            selected_iou = iou_box(selected_bbox, gt_bbox) if selected_bbox is not None else 0.0
            accepted_iou = selected_iou if accepted else 0.0
            final_iou = iou_box(final_bbox, gt_bbox) if final_bbox is not None else 0.0
            failure_category = classify_oracle_gt_frame(
                len(candidates), oracle_iou, selected_iou, accepted,
                final_bbox is not None, final_iou,
            )
            failure_counts[failure_category] += 1
            oracle_iou_sum += oracle_iou
            selected_iou_sum += selected_iou
            accepted_iou_sum += accepted_iou
            final_iou_sum += final_iou
            matching_regret_sum += max(0.0, oracle_iou - selected_iou)
            for threshold_value in ORACLE_IOU_THRESHOLDS:
                recall_counts['oracle'][threshold_value] += oracle_iou >= threshold_value
                recall_counts['selected'][threshold_value] += selected_iou >= threshold_value
                recall_counts['accepted'][threshold_value] += accepted_iou >= threshold_value
                recall_counts['final'][threshold_value] += final_iou >= threshold_value
            temporal_delta = final_iou - accepted_iou
            if temporal_delta > 1e-12:
                temporal_helped += 1
            elif temporal_delta < -1e-12:
                temporal_harmed += 1
            else:
                temporal_unchanged += 1
        elif final_bbox is not None:
            failure_category = 'background_final_false_positive'
        elif accepted:
            failure_category = 'background_removed_by_temporal'
        elif selected is not None:
            failure_category = 'background_below_threshold'

        frame_rows.append({
            'video_id': video_id, 'identity_id': identity_id_from_video_id(video_id),
            'frame': frame_idx, 'has_gt': has_gt,
            'candidate_count': len(candidates), 'oracle_candidate_idx': oracle_idx,
            'oracle_iou': oracle_iou if has_gt else None,
            'oracle_fusion_score': oracle_score,
            'selected_score': selected_score,
            'selected_iou': selected_iou if has_gt else None,
            'accepted': accepted,
            'accepted_iou': accepted_iou if has_gt else None,
            'final_present': final_bbox is not None,
            'final_iou': final_iou if has_gt else None,
            'failure_category': failure_category,
        })

    gt_count = len(gt_frames)
    safe_gt_count = max(1, gt_count)
    video_row = {
        'video_id': video_id, 'identity_id': identity_id_from_video_id(video_id),
        'gt_frames': gt_count, 'candidate_frames': len(video_cache['frames']),
        'selected_frames': len(selected_frames), 'accepted_frames': len(accepted_frames),
        'final_frames': len(final_frames),
        'oracle_st_iou_upper_bound': compute_st_iou_video(gt_frames, oracle_frames),
        'selected_prethreshold_st_iou': compute_st_iou_video(gt_frames, selected_frames),
        'accepted_pretemporal_st_iou': compute_st_iou_video(gt_frames, accepted_frames),
        'final_production_st_iou': compute_st_iou_video(gt_frames, final_frames),
        'mean_oracle_iou_on_gt': oracle_iou_sum / safe_gt_count,
        'mean_selected_iou_on_gt': selected_iou_sum / safe_gt_count,
        'mean_accepted_iou_on_gt': accepted_iou_sum / safe_gt_count,
        'mean_final_iou_on_gt': final_iou_sum / safe_gt_count,
        'mean_matching_regret': matching_regret_sum / safe_gt_count,
        'temporal_helped_gt_frames': temporal_helped,
        'temporal_harmed_gt_frames': temporal_harmed,
        'temporal_unchanged_gt_frames': temporal_unchanged,
        'accepted_background_frames': sum(
            frame_idx not in gt_frames for frame_idx in accepted_frames
        ),
        'final_background_frames': sum(
            frame_idx not in gt_frames for frame_idx in final_frames
        ),
    }
    for stage, stage_counts in recall_counts.items():
        for threshold_value, count in stage_counts.items():
            suffix = str(threshold_value).replace('.', '')
            video_row[f'{stage}_recall_iou_{suffix}'] = count / safe_gt_count
    for category in ORACLE_GT_FAILURE_CATEGORIES:
        video_row[f'failure__{category}'] = failure_counts[category]
        video_row[f'failure_rate__{category}'] = failure_counts[category] / safe_gt_count
    return frame_rows, video_row


def aggregate_identity_oracle_rows(video_rows):
    identity_groups = {}
    for row in video_rows:
        identity_groups.setdefault(row['identity_id'], []).append(row)
    output = {}
    mean_fields = [
        'oracle_st_iou_upper_bound', 'selected_prethreshold_st_iou',
        'accepted_pretemporal_st_iou', 'final_production_st_iou',
        'mean_oracle_iou_on_gt', 'mean_selected_iou_on_gt',
        'mean_accepted_iou_on_gt', 'mean_final_iou_on_gt', 'mean_matching_regret',
    ]
    for identity_id, rows in identity_groups.items():
        identity_result = {
            'videos': [row['video_id'] for row in rows],
            'gt_frames': sum(row['gt_frames'] for row in rows),
        }
        for field in mean_fields:
            identity_result[f'mean_video__{field}'] = (
                sum(row[field] for row in rows) / len(rows)
            )
        identity_result['failure_counts'] = {
            category: sum(row[f'failure__{category}'] for row in rows)
            for category in ORACLE_GT_FAILURE_CATEGORIES
        }
        output[identity_id] = identity_result
    return output


def run_oracle_diagnostics(cache, gt_map, weights, matching_threshold, production_result):
    prediction_snapshot = json.dumps(
        production_result['predictions'], sort_keys=True, separators=(',', ':')
    )
    final_by_video = prediction_frame_maps(production_result['predictions'])
    frame_rows, video_rows = [], []
    for video_id in cache['signature']['video_folders']:
        video_cache = cache['videos'][video_id]
        ranked_frames, _ = rank_video_frames(video_cache, weights, include_debug=False)
        video_frame_rows, video_row = analyze_oracle_video(
            video_id, video_cache, gt_map.get(video_id, {}), ranked_frames,
            final_by_video.get(video_id, {}), weights, matching_threshold,
        )
        frame_rows.extend(video_frame_rows)
        video_rows.append(video_row)

    for row in video_rows:
        expected_score = production_result['per_video'][row['video_id']]
        if abs(row['final_production_st_iou'] - expected_score) > 1e-12:
            raise RuntimeError(
                f"Oracle final metric mismatch for {row['video_id']}: "
                f"{row['final_production_st_iou']} vs {expected_score}"
            )
    if json.dumps(production_result['predictions'], sort_keys=True, separators=(',', ':')) != prediction_snapshot:
        raise RuntimeError('Oracle diagnostics mutated production predictions')
    total_gt_frames = sum(row['gt_frames'] for row in video_rows)
    global_failures = {
        category: sum(row[f'failure__{category}'] for row in video_rows)
        for category in ORACLE_GT_FAILURE_CATEGORIES
    }
    non_success_failures = {
        key: value for key, value in global_failures.items() if key != 'success'
    }
    dominant_failure = (
        max(non_success_failures, key=non_success_failures.get)
        if sum(non_success_failures.values()) > 0 else None
    )
    mean_video_fields = [
        'oracle_st_iou_upper_bound', 'selected_prethreshold_st_iou',
        'accepted_pretemporal_st_iou', 'final_production_st_iou',
    ]
    global_summary = {
        'total_gt_frames': total_gt_frames,
        'failure_counts': global_failures,
        'failure_rates': {
            key: value / max(1, total_gt_frames) for key, value in global_failures.items()
        },
        'dominant_non_success_failure': dominant_failure,
        'mean_video_metrics': {
            field: sum(row[field] for row in video_rows) / len(video_rows)
            for field in mean_video_fields
        },
        'temporal_gt_frame_effect': {
            'helped': sum(row['temporal_helped_gt_frames'] for row in video_rows),
            'harmed': sum(row['temporal_harmed_gt_frames'] for row in video_rows),
            'unchanged': sum(row['temporal_unchanged_gt_frames'] for row in video_rows),
        },
        'accepted_background_frames': sum(
            row['accepted_background_frames'] for row in video_rows
        ),
        'final_background_frames': sum(
            row['final_background_frames'] for row in video_rows
        ),
    }
    return frame_rows, video_rows, {
        'production_predictions_unchanged': True,
        'production_per_video_metric_exact_parity': True,
        'global': global_summary,
        'per_identity': aggregate_identity_oracle_rows(video_rows),
    }


def hysteresis_config_grid():
    return [
        {
            'weak_threshold': weak_threshold,
            'max_anchor_span': max_anchor_span,
            'strong_threshold': HYSTERESIS_STRONG_THRESHOLD,
            'max_center_speed': HYSTERESIS_MAX_CENTER_SPEED,
        }
        for weak_threshold in HYSTERESIS_WEAK_THRESHOLD_GRID
        for max_anchor_span in HYSTERESIS_ANCHOR_SPAN_GRID
    ]


def bracketed_hysteresis_accept(ranked_frames, weak_threshold, strong_threshold,
                                 max_anchor_span, max_center_speed):
    if weak_threshold > strong_threshold:
        raise ValueError('weak_threshold must not exceed strong_threshold')
    if max_anchor_span < 1:
        raise ValueError('max_anchor_span must be positive')
    strong_frame_ids = sorted(
        frame_idx for frame_idx, item in ranked_frames.items()
        if item['score'] >= strong_threshold
    )
    accepted = {frame_idx: ranked_frames[frame_idx] for frame_idx in strong_frame_ids}
    if weak_threshold == strong_threshold or len(strong_frame_ids) < 2:
        return accepted

    for frame_idx in sorted(ranked_frames):
        candidate = ranked_frames[frame_idx]
        if candidate['score'] < weak_threshold or candidate['score'] >= strong_threshold:
            continue
        insertion = bisect.bisect_left(strong_frame_ids, frame_idx)
        if insertion == 0 or insertion == len(strong_frame_ids):
            continue
        past_frame = strong_frame_ids[insertion - 1]
        future_frame = strong_frame_ids[insertion]
        past_steps, future_steps = frame_idx - past_frame, future_frame - frame_idx
        if past_steps > max_anchor_span or future_steps > max_anchor_span:
            continue
        past_item, future_item = ranked_frames[past_frame], ranked_frames[future_frame]
        motion_checks = [
            gap_is_plausible(
                past_item['bbox'], candidate['bbox'], past_steps,
                max_center_speed=max_center_speed, max_log_scale_speed=None,
            ),
            gap_is_plausible(
                candidate['bbox'], future_item['bbox'], future_steps,
                max_center_speed=max_center_speed, max_log_scale_speed=None,
            ),
            gap_is_plausible(
                past_item['bbox'], future_item['bbox'], future_frame - past_frame,
                max_center_speed=max_center_speed, max_log_scale_speed=None,
            ),
        ]
        if all(motion_checks):
            accepted[frame_idx] = candidate
    return accepted


def render_hysteresis_predictions(cache, ranked_by_video, gt_map, config):
    predictions, per_video = [], {}
    detected_frames = rescued_frames = 0
    rescued_per_video = {}
    for video_id in cache['signature']['video_folders']:
        video_cache = cache['videos'][video_id]
        ranked_frames = ranked_by_video[video_id]
        if not video_cache['has_siamese']:
            accepted = {
                frame_idx: item for frame_idx, item in ranked_frames.items()
                if item['score'] >= YOLO_ONLY_THRESHOLD
            }
            rescued_count = 0
        else:
            accepted = bracketed_hysteresis_accept(
                ranked_frames,
                weak_threshold=config['weak_threshold'],
                strong_threshold=config['strong_threshold'],
                max_anchor_span=config['max_anchor_span'],
                max_center_speed=config['max_center_speed'],
            )
            strong_count = sum(
                item['score'] >= config['strong_threshold']
                for item in ranked_frames.values()
            )
            rescued_count = len(accepted) - strong_count
        if accepted:
            final_bboxes = temporal_smooth_detections_gated(
                accepted, video_cache['max_frame_idx'],
                min_seg_len=PRODUCTION_TEMPORAL_CONFIG['min_seg_len'],
                max_gap=PRODUCTION_TEMPORAL_CONFIG['max_gap'],
                max_center_speed=PRODUCTION_TEMPORAL_CONFIG['max_center_speed'],
                max_log_scale_speed=PRODUCTION_TEMPORAL_CONFIG['max_log_scale_speed'],
            )
        else:
            final_bboxes = []
        predictions.append({
            'video_id': video_id,
            'detections': [{'bboxes': final_bboxes}] if final_bboxes else [],
        })
        pred_frames = {
            bbox['frame']: [bbox['x1'], bbox['y1'], bbox['x2'], bbox['y2']]
            for bbox in final_bboxes
        }
        per_video[video_id] = compute_st_iou_video(gt_map.get(video_id, {}), pred_frames)
        detected_frames += len(final_bboxes)
        rescued_frames += rescued_count
        rescued_per_video[video_id] = rescued_count
    return {
        'mean_st_iou': sum(per_video.values()) / max(1, len(per_video)),
        'per_video': per_video, 'detected_frames': detected_frames,
        'predictions': predictions, 'rescued_frames': rescued_frames,
        'rescued_per_video': rescued_per_video,
    }


def hysteresis_result_row(config, result):
    row = {
        **config, 'mean_st_iou': result['mean_st_iou'],
        'detected_frames': result['detected_frames'],
        'rescued_frames': result['rescued_frames'],
    }
    for video_id, score in sorted(result['per_video'].items()):
        row[f'st_iou__{video_id}'] = score
        row[f'rescued__{video_id}'] = result['rescued_per_video'][video_id]
    return row


def hysteresis_config_from_row(row):
    return {
        key: row[key] for key in (
            'weak_threshold', 'max_anchor_span',
            'strong_threshold', 'max_center_speed',
        )
    }


def select_hysteresis_row(rows, selection_video_ids):
    # Exact validation ties prefer less intervention: higher weak threshold, shorter span.
    return max(
        rows,
        key=lambda row: (
            mean_row_score(row, selection_video_ids),
            row['weak_threshold'], -row['max_anchor_span'],
        ),
    )


def select_voted_hysteresis_config(fold_rows):
    votes = {}
    for fold in fold_rows:
        key = (fold['weak_threshold'], fold['max_anchor_span'])
        votes[key] = votes.get(key, 0) + 1
    best_key = max(votes, key=lambda key: (votes[key], key[0], -key[1]))
    return {
        'weak_threshold': best_key[0], 'max_anchor_span': best_key[1],
        'strong_threshold': HYSTERESIS_STRONG_THRESHOLD,
        'max_center_speed': HYSTERESIS_MAX_CENTER_SPEED,
    }, votes[best_key]


def run_hysteresis_identity_cv(cache, gt_map, weights, production_result):
    ranked_by_video = {
        video_id: rank_video_frames(video_cache, weights, include_debug=False)[0]
        for video_id, video_cache in cache['videos'].items()
    }
    rows, results_by_config = [], {}
    for config in tqdm(hysteresis_config_grid(), desc='Threshold hysteresis grid'):
        result = render_hysteresis_predictions(cache, ranked_by_video, gt_map, config)
        rows.append(hysteresis_result_row(config, result))
        results_by_config[(config['weak_threshold'], config['max_anchor_span'])] = result

    control_key = (HYSTERESIS_STRONG_THRESHOLD, min(HYSTERESIS_ANCHOR_SPAN_GRID))
    control_result = results_by_config[control_key]
    if control_result['predictions'] != production_result['predictions']:
        raise RuntimeError(
            'Hysteresis weak-equals-strong control does not match production predictions'
        )
    identity_folds = build_identity_folds(cache['signature']['video_folders'])
    all_video_ids = sorted(cache['signature']['video_folders'])
    fold_rows = []
    for held_identity, held_video_ids in identity_folds.items():
        selection_video_ids = [
            video_id for video_id in all_video_ids if video_id not in held_video_ids
        ]
        selected = select_hysteresis_row(rows, selection_video_ids)
        heldout_score = mean_row_score(selected, held_video_ids)
        baseline_score = sum(
            production_result['per_video'][video_id] for video_id in held_video_ids
        ) / len(held_video_ids)
        fold_rows.append({
            'heldout_identity': held_identity,
            'heldout_videos': '|'.join(held_video_ids),
            'selection_videos': '|'.join(selection_video_ids),
            **hysteresis_config_from_row(selected),
            'selection_mean_st_iou': mean_row_score(selected, selection_video_ids),
            'heldout_mean_st_iou': heldout_score,
            'baseline_heldout_mean_st_iou': baseline_score,
            'heldout_delta': heldout_score - baseline_score,
        })

    recommended_config, recommendation_votes = select_voted_hysteresis_config(fold_rows)
    recommended_result = results_by_config[(
        recommended_config['weak_threshold'], recommended_config['max_anchor_span']
    )]
    mean_cv_score = sum(row['heldout_mean_st_iou'] for row in fold_rows) / len(fold_rows)
    mean_cv_baseline = sum(
        row['baseline_heldout_mean_st_iou'] for row in fold_rows
    ) / len(fold_rows)
    cv_delta = mean_cv_score - mean_cv_baseline
    improved_identities = sum(row['heldout_delta'] > 0 for row in fold_rows)
    worst_identity_delta = min(row['heldout_delta'] for row in fold_rows)
    promotion_passed = (
        cv_delta >= HYSTERESIS_MIN_MEAN_CV_DELTA
        and improved_identities >= HYSTERESIS_MIN_IMPROVED_IDENTITIES
        and worst_identity_delta >= -HYSTERESIS_MAX_WORST_IDENTITY_DROP
        and recommended_config['weak_threshold'] < HYSTERESIS_STRONG_THRESHOLD
    )
    diagnostics = {
        'weak_equals_strong_exact_prediction_parity': True,
        'identity_folds': identity_folds,
        'mean_cv_st_iou': mean_cv_score,
        'mean_cv_baseline_st_iou': mean_cv_baseline,
        'mean_cv_delta': cv_delta,
        'improved_identity_count': improved_identities,
        'worst_identity_delta': worst_identity_delta,
        'promotion_passed': promotion_passed,
        'recommended_config': recommended_config,
        'recommendation_votes': recommendation_votes,
    }
    return (
        sorted(rows, key=lambda row: row['mean_st_iou'], reverse=True),
        fold_rows, diagnostics, recommended_result,
    )


def score_candidate_details(candidates, weights, has_siamese, has_color):
    scored = []
    for candidate_idx, candidate in enumerate(candidates):
        scored.append({
            'candidate_idx': candidate_idx,
            'bbox': candidate['bbox'],
            'components': [float(value) for value in candidate['scores']],
            'score': float(fuse_candidate_score(
                candidate['scores'], weights, has_siamese, has_color
            )),
        })
    scored.sort(key=lambda item: (-item['score'], item['candidate_idx']))
    return scored


def binary_metrics_at_threshold(labels, scores, threshold):
    tp = sum(label == 1 and score >= threshold for label, score in zip(labels, scores))
    fp = sum(label == 0 and score >= threshold for label, score in zip(labels, scores))
    fn = sum(label == 1 and score < threshold for label, score in zip(labels, scores))
    tn = sum(label == 0 and score < threshold for label, score in zip(labels, scores))
    precision = tp / max(1, tp + fp)
    recall = tp / max(1, tp + fn)
    return {
        'threshold': float(threshold), 'tp': tp, 'fp': fp, 'fn': fn, 'tn': tn,
        'precision': precision, 'recall': recall,
        'false_positive_rate': fp / max(1, fp + tn),
        'f1': 2 * tp / max(1, 2 * tp + fp + fn),
        'balanced_accuracy': 0.5 * (recall + tn / max(1, tn + fp)),
    }


def binary_roc_auc(labels, scores):
    positive_count = sum(labels)
    negative_count = len(labels) - positive_count
    if positive_count == 0 or negative_count == 0:
        return None
    ordered = sorted(zip(scores, labels), key=lambda pair: pair[0])
    positive_rank_sum = 0.0
    index = 0
    while index < len(ordered):
        end = index + 1
        while end < len(ordered) and ordered[end][0] == ordered[index][0]:
            end += 1
        average_rank = 0.5 * ((index + 1) + end)
        positive_rank_sum += average_rank * sum(label for _, label in ordered[index:end])
        index = end
    return (
        positive_rank_sum - positive_count * (positive_count + 1) / 2
    ) / (positive_count * negative_count)


def binary_average_precision(labels, scores):
    positive_count = sum(labels)
    if positive_count == 0:
        return None
    ordered = sorted(zip(scores, labels), key=lambda pair: pair[0], reverse=True)
    true_positives = seen = 0
    precision_sum = 0.0
    index = 0
    while index < len(ordered):
        end = index + 1
        while end < len(ordered) and ordered[end][0] == ordered[index][0]:
            end += 1
        group_positives = sum(label for _, label in ordered[index:end])
        true_positives += group_positives
        seen += end - index
        precision_sum += group_positives * (true_positives / seen)
        index = end
    return precision_sum / positive_count


def feature_separability(rows, feature, weak_only):
    filtered = [
        row for row in rows
        if row.get(feature) is not None
        and (not weak_only or row['is_weak_region'])
    ]
    labels = [int(row['is_correct_target']) for row in filtered]
    scores = [float(row[feature]) for row in filtered]
    positives = sum(labels)
    negatives = len(labels) - positives
    positive_values = [score for label, score in zip(labels, scores) if label == 1]
    negative_values = [score for label, score in zip(labels, scores) if label == 0]
    return {
        'rows': len(filtered), 'positive': positives, 'negative': negatives,
        'positive_prevalence': positives / max(1, len(filtered)),
        'roc_auc': binary_roc_auc(labels, scores),
        'average_precision': binary_average_precision(labels, scores),
        'positive_median': float(np.median(positive_values)) if positive_values else None,
        'negative_median': float(np.median(negative_values)) if negative_values else None,
    }


def select_best_f1_threshold(rows, feature):
    filtered = [
        row for row in rows if row.get(feature) is not None and row['is_weak_region']
    ]
    labels = [int(row['is_correct_target']) for row in filtered]
    scores = [float(row[feature]) for row in filtered]
    if not scores or sum(labels) == 0:
        return None, None
    ordered = sorted(zip(scores, labels), key=lambda pair: pair[0], reverse=True)
    total_positives = sum(labels)
    total_negatives = len(labels) - total_positives
    tp = fp = 0
    best_metrics = None
    index = 0
    while index < len(ordered):
        end = index + 1
        while end < len(ordered) and ordered[end][0] == ordered[index][0]:
            end += 1
        group_positives = sum(label for _, label in ordered[index:end])
        tp += group_positives
        fp += (end - index) - group_positives
        fn, tn = total_positives - tp, total_negatives - fp
        recall = tp / max(1, tp + fn)
        metrics = {
            'threshold': float(ordered[index][0]),
            'tp': tp, 'fp': fp, 'fn': fn, 'tn': tn,
            'precision': tp / max(1, tp + fp), 'recall': recall,
            'false_positive_rate': fp / max(1, fp + tn),
            'f1': 2 * tp / max(1, 2 * tp + fp + fn),
            'balanced_accuracy': 0.5 * (recall + tn / max(1, tn + fp)),
        }
        if best_metrics is None or metrics['f1'] > best_metrics['f1'] + 1e-15:
            best_metrics = metrics
        index = end
    return best_metrics['threshold'], best_metrics


def margin_frame_rows_for_video(video_id, video_cache, gt_frames, final_frames,
                                weights, matching_threshold):
    rows = []
    has_siamese = bool(video_cache['has_siamese'])
    has_color = bool(video_cache['has_color'])
    threshold = matching_threshold if has_siamese else YOLO_ONLY_THRESHOLD
    for frame_key in sorted(video_cache['frames'], key=lambda key: int(key)):
        frame_idx = int(frame_key)
        candidates = video_cache['frames'][frame_key]
        scored = score_candidate_details(candidates, weights, has_siamese, has_color)
        if not scored:
            continue
        top1 = scored[0]
        top2 = scored[1] if len(scored) > 1 else None
        gt_bbox = gt_frames.get(frame_idx)
        has_gt = gt_bbox is not None
        selected_iou = iou_box(top1['bbox'], gt_bbox) if has_gt else 0.0
        oracle_idx, oracle_iou = (None, 0.0)
        oracle = None
        if has_gt:
            oracle_idx, oracle_iou = oracle_candidate_for_gt(candidates, gt_bbox)
            if oracle_idx is not None:
                oracle = next(item for item in scored if item['candidate_idx'] == oracle_idx)
        accepted = top1['score'] >= threshold
        final_bbox = final_frames.get(frame_idx)
        final_iou = iou_box(final_bbox, gt_bbox) if has_gt and final_bbox is not None else 0.0
        if has_gt:
            failure_category = classify_oracle_gt_frame(
                len(candidates), oracle_iou, selected_iou, accepted,
                final_bbox is not None, final_iou,
            )
        elif final_bbox is not None:
            failure_category = 'background_final_false_positive'
        elif accepted:
            failure_category = 'background_removed_by_temporal'
        else:
            failure_category = 'background_below_threshold'
        margin = top1['score'] - (top2['score'] if top2 is not None else 0.0)
        ratio = (
            top1['score'] / max(1e-12, top2['score']) if top2 is not None else None
        )
        rows.append({
            'video_id': video_id, 'identity_id': identity_id_from_video_id(video_id),
            'frame': frame_idx, 'has_gt': has_gt,
            'candidate_count': len(candidates),
            'negative_candidate_count': -len(candidates),
            'is_correct_target': int(has_gt and selected_iou >= ORACLE_GOOD_IOU),
            'is_weak_region': top1['score'] < MARGIN_WEAK_SCORE_THRESHOLD,
            'selected_candidate_idx': top1['candidate_idx'],
            'selected_score': top1['score'],
            'selected_yolo': top1['components'][0],
            'selected_siamese': top1['components'][1],
            'selected_color': top1['components'][2],
            'selected_iou': selected_iou if has_gt else None,
            'has_top2': top2 is not None,
            'top2_score': top2['score'] if top2 is not None else None,
            'top2_yolo': top2['components'][0] if top2 is not None else None,
            'top2_siamese': top2['components'][1] if top2 is not None else None,
            'top2_color': top2['components'][2] if top2 is not None else None,
            'score_margin': margin, 'score_ratio': ratio,
            'oracle_candidate_idx': oracle_idx, 'oracle_iou': oracle_iou if has_gt else None,
            'oracle_fusion_score': oracle['score'] if oracle is not None else None,
            'oracle_yolo': oracle['components'][0] if oracle is not None else None,
            'oracle_siamese': oracle['components'][1] if oracle is not None else None,
            'oracle_color': oracle['components'][2] if oracle is not None else None,
            'accepted': accepted, 'final_present': final_bbox is not None,
            'failure_category': failure_category,
        })
    return rows


def run_margin_diagnostics(cache, gt_map, weights, matching_threshold, production_result):
    prediction_snapshot = json.dumps(
        production_result['predictions'], sort_keys=True, separators=(',', ':')
    )
    final_by_video = prediction_frame_maps(production_result['predictions'])
    frame_rows = []
    for video_id in cache['signature']['video_folders']:
        final_frames = final_by_video.get(video_id, {})
        expected_score = production_result['per_video'][video_id]
        actual_score = compute_st_iou_video(gt_map.get(video_id, {}), final_frames)
        if abs(actual_score - expected_score) > 1e-12:
            raise RuntimeError(f'Margin diagnostic final metric mismatch for {video_id}')
        frame_rows.extend(margin_frame_rows_for_video(
            video_id, cache['videos'][video_id], gt_map.get(video_id, {}),
            final_frames, weights, matching_threshold,
        ))
    if json.dumps(production_result['predictions'], sort_keys=True, separators=(',', ':')) != prediction_snapshot:
        raise RuntimeError('Margin diagnostics mutated production predictions')

    feature_reports = {
        feature: {
            'all_selected_frames': feature_separability(frame_rows, feature, False),
            'weak_region': feature_separability(frame_rows, feature, True),
        }
        for feature in MARGIN_FEATURES
    }
    video_rows = []
    for video_id in cache['signature']['video_folders']:
        rows = [row for row in frame_rows if row['video_id'] == video_id]
        video_row = {
            'video_id': video_id, 'identity_id': identity_id_from_video_id(video_id),
            'selected_frames': len(rows),
            'weak_frames': sum(row['is_weak_region'] for row in rows),
            'weak_positive_frames': sum(
                row['is_weak_region'] and row['is_correct_target'] for row in rows
            ),
        }
        for feature in MARGIN_FEATURES:
            report = feature_separability(rows, feature, True)
            video_row[f'weak_auc__{feature}'] = report['roc_auc']
            video_row[f'weak_ap__{feature}'] = report['average_precision']
        video_rows.append(video_row)

    identities = sorted({row['identity_id'] for row in frame_rows})
    per_identity = {}
    for identity_id in identities:
        identity_rows = [row for row in frame_rows if row['identity_id'] == identity_id]
        per_identity[identity_id] = {
            feature: feature_separability(identity_rows, feature, True)
            for feature in MARGIN_FEATURES
        }

    loio_transfer = {}
    for feature in MARGIN_FEATURES:
        feature_folds = {}
        for held_identity in identities:
            train_rows = [row for row in frame_rows if row['identity_id'] != held_identity]
            held_rows = [
                row for row in frame_rows
                if row['identity_id'] == held_identity
                and row.get(feature) is not None and row['is_weak_region']
            ]
            threshold, train_metrics = select_best_f1_threshold(train_rows, feature)
            held_labels = [int(row['is_correct_target']) for row in held_rows]
            held_scores = [float(row[feature]) for row in held_rows]
            held_metrics = (
                binary_metrics_at_threshold(held_labels, held_scores, threshold)
                if threshold is not None and held_scores else None
            )
            feature_folds[held_identity] = {
                'selected_threshold': threshold,
                'train_metrics': train_metrics, 'heldout_metrics': held_metrics,
            }
        loio_transfer[feature] = feature_folds

    return frame_rows, video_rows, {
        'production_predictions_unchanged': True,
        'production_per_video_metric_exact_parity': True,
        'label_definition': 'positive iff GT exists and selected bbox IoU >= 0.5',
        'weak_region_definition': f'selected_score < {MARGIN_WEAK_SCORE_THRESHOLD}',
        'feature_reports': feature_reports,
        'per_identity_weak_reports': per_identity,
        'leave_one_identity_out_weak_f1_transfer': loio_transfer,
    }






# --- SAHI SOURCE ADMISSION V1: cache replay only ---

def build_sahi_admission_cache(base_cache, tile_cache, policy, weights,
                               matching_threshold):
    if policy not in SAHI_ADMISSION_POLICIES:
        raise ValueError(f'Unknown SAHI admission policy: {policy}')
    video_ids = list(base_cache['signature']['video_folders'])
    if video_ids != list(tile_cache['signature']['video_folders']):
        raise RuntimeError('Legacy and SAHI tile cache video lists differ')
    if policy == 'legacy_only':
        return base_cache

    signature = dict(base_cache['signature'])
    signature['schema_version'] = 5
    signature['candidate_generation'] = {
        'mode': 'source_aware_sahi_admission_cache_replay',
        'policy': policy,
        'tile_signature': tile_cache['signature']['candidate_generation'],
        'matching_threshold': float(matching_threshold),
        'legacy_candidates_preserved': True,
    }
    output = {
        'schema_version': 5,
        'signature': signature,
        'complete': bool(base_cache.get('complete') and tile_cache.get('complete')),
        'videos': {},
    }
    for video_id in video_ids:
        base_video = base_cache['videos'][video_id]
        tile_video = tile_cache['videos'][video_id]
        threshold = matching_threshold if base_video['has_siamese'] else YOLO_ONLY_THRESHOLD
        frame_keys = sorted(
            set(base_video['frames']) | set(tile_video['frames']), key=int
        )
        frames = {}
        tile_admitted_count = 0
        tile_admitted_frames = 0
        weak_legacy_frames = 0
        missing_legacy_frames = 0
        for frame_key in frame_keys:
            legacy_candidates = [
                dict(candidate, source=candidate.get('source', 'legacy_full_frame'))
                for candidate in base_video['frames'].get(frame_key, [])
            ]
            tile_candidates = [
                dict(candidate) for candidate in tile_video['frames'].get(frame_key, [])
            ]
            admit_tiles = False
            if not legacy_candidates:
                missing_legacy_frames += 1
                admit_tiles = True
            elif policy == 'weak_legacy_rescue':
                best_legacy_score = max(
                    fuse_candidate_score(
                        candidate['scores'], weights,
                        base_video['has_siamese'], base_video['has_color'],
                    )
                    for candidate in legacy_candidates
                )
                if best_legacy_score < threshold:
                    weak_legacy_frames += 1
                    admit_tiles = True
            # missing_legacy_only intentionally never admits tiles when legacy exists.
            admitted_tiles = tile_candidates if admit_tiles else []
            combined = legacy_candidates + admitted_tiles
            if combined:
                frames[frame_key] = combined
            if admitted_tiles:
                tile_admitted_frames += 1
                tile_admitted_count += len(admitted_tiles)

        output['videos'][video_id] = {
            'max_frame_idx': max(
                base_video.get('max_frame_idx', -1), tile_video.get('max_frame_idx', -1)
            ),
            'has_siamese': bool(base_video.get('has_siamese') or tile_video.get('has_siamese')),
            'has_color': bool(base_video.get('has_color') or tile_video.get('has_color')),
            'frames': frames,
            'legacy_candidate_count': sum(
                len(items) for items in base_video['frames'].values()
            ),
            'available_tile_candidate_count': sum(
                len(items) for items in tile_video['frames'].values()
            ),
            'tile_admitted_count': tile_admitted_count,
            'tile_admitted_frames': tile_admitted_frames,
            'missing_legacy_frames': missing_legacy_frames,
            'weak_legacy_frames': weak_legacy_frames,
        }
    return output


def sahi_admission_identity_cv(policy_results):
    policy_order = list(SAHI_ADMISSION_POLICIES)
    video_ids = sorted(policy_results['legacy_only']['per_video'])
    identities = sorted({identity_id_from_video_id(video_id) for video_id in video_ids})
    folds = []
    selected_counts = {policy: 0 for policy in policy_order}
    for held_identity in identities:
        train_videos = [
            video_id for video_id in video_ids
            if identity_id_from_video_id(video_id) != held_identity
        ]
        held_videos = [
            video_id for video_id in video_ids
            if identity_id_from_video_id(video_id) == held_identity
        ]
        train_scores = {
            policy: sum(
                policy_results[policy]['per_video'][video_id]
                for video_id in train_videos
            ) / len(train_videos)
            for policy in policy_order
        }
        # Stable conservative tie-break: legacy, then missing-only, then weak-rescue.
        selected_policy = max(
            policy_order,
            key=lambda policy: (train_scores[policy], -policy_order.index(policy)),
        )
        selected_counts[selected_policy] += 1
        baseline_held = sum(
            policy_results['legacy_only']['per_video'][video_id]
            for video_id in held_videos
        ) / len(held_videos)
        selected_held = sum(
            policy_results[selected_policy]['per_video'][video_id]
            for video_id in held_videos
        ) / len(held_videos)
        folds.append({
            'heldout_identity': held_identity,
            'train_videos': '|'.join(train_videos),
            'heldout_videos': '|'.join(held_videos),
            'selected_policy': selected_policy,
            'train_score_legacy_only': train_scores['legacy_only'],
            'train_score_missing_legacy_only': train_scores['missing_legacy_only'],
            'train_score_weak_legacy_rescue': train_scores['weak_legacy_rescue'],
            'heldout_baseline_st_iou': baseline_held,
            'heldout_selected_st_iou': selected_held,
            'heldout_delta': selected_held - baseline_held,
        })

    mean_cv_delta = sum(row['heldout_delta'] for row in folds) / len(folds)
    improved_count = sum(row['heldout_delta'] > 0 for row in folds)
    worst_delta = min(row['heldout_delta'] for row in folds)
    recommended_policy = max(
        policy_order,
        key=lambda policy: (
            policy_results[policy]['mean_st_iou'], -policy_order.index(policy)
        ),
    )
    dominant_policy = max(
        policy_order,
        key=lambda policy: (selected_counts[policy], -policy_order.index(policy)),
    )
    policy_consistent = (
        recommended_policy != 'legacy_only'
        and dominant_policy == recommended_policy
        and selected_counts[dominant_policy] >= SAHI_ADMISSION_MIN_POLICY_FOLD_COUNT
    )

    baseline_identity = resolution_identity_scores(policy_results['legacy_only'])
    recommended_identity = resolution_identity_scores(policy_results[recommended_policy])
    full_identity_deltas = {
        identity_id: recommended_identity[identity_id] - baseline_identity[identity_id]
        for identity_id in baseline_identity
    }
    full_mean_delta = (
        policy_results[recommended_policy]['mean_st_iou']
        - policy_results['legacy_only']['mean_st_iou']
    )
    full_improved_count = sum(delta > 0 for delta in full_identity_deltas.values())
    full_worst_delta = min(full_identity_deltas.values())
    cv_passed = (
        mean_cv_delta >= SAHI_ADMISSION_MIN_MEAN_CV_DELTA
        and improved_count >= SAHI_ADMISSION_MIN_IMPROVED_IDENTITIES
        and worst_delta >= -SAHI_ADMISSION_MAX_WORST_IDENTITY_DROP
    )
    full_passed = (
        recommended_policy != 'legacy_only'
        and full_mean_delta >= SAHI_ADMISSION_MIN_MEAN_CV_DELTA
        and full_improved_count >= SAHI_ADMISSION_MIN_IMPROVED_IDENTITIES
        and full_worst_delta >= -SAHI_ADMISSION_MAX_WORST_IDENTITY_DROP
    )
    promotion_passed = bool(cv_passed and full_passed and policy_consistent)
    summary = {
        'promotion_passed': promotion_passed,
        'recommended_policy_descriptive_full_public': recommended_policy,
        'dominant_policy_across_folds': dominant_policy,
        'selected_policy_counts': selected_counts,
        'policy_consistent': policy_consistent,
        'mean_cv_delta': mean_cv_delta,
        'improved_heldout_identities': improved_count,
        'worst_heldout_delta': worst_delta,
        'full_public_mean_delta': full_mean_delta,
        'full_public_identity_deltas': full_identity_deltas,
        'full_public_improved_identities': full_improved_count,
        'full_public_worst_identity_delta': full_worst_delta,
        'criteria': {
            'minimum_mean_cv_and_full_delta': SAHI_ADMISSION_MIN_MEAN_CV_DELTA,
            'minimum_improved_identities': SAHI_ADMISSION_MIN_IMPROVED_IDENTITIES,
            'maximum_worst_identity_drop': SAHI_ADMISSION_MAX_WORST_IDENTITY_DROP,
            'minimum_dominant_policy_fold_count': SAHI_ADMISSION_MIN_POLICY_FOLD_COUNT,
        },
    }
    return folds, summary


def run_sahi_admission_experiment(base_cache, tile_cache, gt_map,
                                  baseline_result, temporal_config):
    prediction_snapshot = json.dumps(
        baseline_result['predictions'], sort_keys=True, separators=(',', ':')
    )
    video_ids = list(base_cache['signature']['video_folders'])
    weights = {
        'yolo': WEIGHT_YOLO,
        'siamese': WEIGHT_SIAMESE,
        'color': WEIGHT_COLOR,
    }
    video_metadata = collect_video_metadata(video_ids)
    architecture = inspect_detector_architecture()
    policy_caches, policy_results = {}, {}
    policy_frame_rows, policy_profile_videos, policy_diagnostics = {}, {}, {}
    baseline_frames = None
    baseline_oracle = None

    for policy in SAHI_ADMISSION_POLICIES:
        policy_cache = build_sahi_admission_cache(
            base_cache, tile_cache, policy, weights, MATCHING_THRESHOLD
        )
        result = (
            baseline_result if policy == 'legacy_only'
            else evaluate_config(
                policy_cache, gt_map, weights, MATCHING_THRESHOLD,
                include_debug=False, temporal_config=temporal_config,
            )
        )
        frame_rows, profile_videos, diagnostics = run_detector_profile(
            policy_cache, gt_map, result,
            video_metadata=video_metadata, detector_architecture=architecture,
        )
        if baseline_frames is None:
            baseline_frames = frame_rows
            baseline_oracle = {
                (row['video_id'], row['frame']): row['oracle_iou'] for row in frame_rows
            }
        else:
            policy_oracle = {
                (row['video_id'], row['frame']): row['oracle_iou'] for row in frame_rows
            }
            if set(policy_oracle) != set(baseline_oracle):
                raise RuntimeError(f'{policy} changed GT frame coverage')
            regression_count = sum(
                policy_oracle[key] < baseline_oracle[key] - SAHI_ADMISSION_ORACLE_TOLERANCE
                for key in baseline_oracle
            )
            if regression_count:
                raise RuntimeError(
                    f'{policy} regressed candidate oracle on {regression_count} GT frames'
                )
        policy_caches[policy] = policy_cache
        policy_results[policy] = result
        policy_frame_rows[policy] = frame_rows
        policy_profile_videos[policy] = profile_videos
        policy_diagnostics[policy] = diagnostics

    legacy_unchanged = (
        json.dumps(baseline_result['predictions'], sort_keys=True, separators=(',', ':'))
        == prediction_snapshot
    )
    if not legacy_unchanged:
        raise RuntimeError('SAHI admission replay mutated production predictions')

    fold_rows, cv_summary = sahi_admission_identity_cv(policy_results)
    baseline_identity = resolution_identity_scores(baseline_result)
    result_rows, video_rows = [], []
    details = {}
    for policy in SAHI_ADMISSION_POLICIES:
        result = policy_results[policy]
        diagnostics = policy_diagnostics[policy]
        cache = policy_caches[policy]
        identity_scores = resolution_identity_scores(result)
        identity_deltas = {
            identity_id: identity_scores[identity_id] - baseline_identity[identity_id]
            for identity_id in baseline_identity
        }
        candidate_count = sum(
            sum(len(items) for items in video['frames'].values())
            for video in cache['videos'].values()
        )
        tile_admitted_count = sum(
            int(video.get('tile_admitted_count', 0))
            for video in cache['videos'].values()
        )
        tile_admitted_frames = sum(
            int(video.get('tile_admitted_frames', 0))
            for video in cache['videos'].values()
        )
        source_counts = candidate_source_selection_counts(
            cache, weights, MATCHING_THRESHOLD
        )
        result_rows.append({
            'policy': policy,
            'mean_st_iou': result['mean_st_iou'],
            'delta_st_iou_vs_legacy': result['mean_st_iou'] - baseline_result['mean_st_iou'],
            'detected_frames': result['detected_frames'],
            'candidate_count': candidate_count,
            'tile_admitted_count': tile_admitted_count,
            'tile_admitted_frames': tile_admitted_frames,
            'top1_tile_frames': source_counts['top1_sahi_tile'],
            'accepted_tile_frames': source_counts['accepted_sahi_tile'],
            'no_candidate_gt_frames': diagnostics['global']['no_candidate_frames'],
            'mean_oracle_iou': diagnostics['global']['mean_oracle_iou'],
            'oracle_recall_iou_05': diagnostics['global']['oracle_recall_iou_05'],
            'sub16_oracle_recall_iou_05': profile_sub16_recall(policy_frame_rows[policy]),
            'improved_identities': sum(delta > 0 for delta in identity_deltas.values()),
            'worst_identity_delta': min(identity_deltas.values()),
        })
        profile_by_video = {
            row['video_id']: row for row in policy_profile_videos[policy]
        }
        for video_id in video_ids:
            profile = profile_by_video[video_id]
            video_rows.append({
                'policy': policy,
                'video_id': video_id,
                'identity_id': identity_id_from_video_id(video_id),
                'st_iou': result['per_video'][video_id],
                'delta_st_iou_vs_legacy': (
                    result['per_video'][video_id]
                    - baseline_result['per_video'][video_id]
                ),
                'no_candidate_gt_frames': profile['no_candidate_frames'],
                'oracle_recall_iou_05': profile['oracle_recall_iou_05'],
                'mean_oracle_iou': profile['mean_oracle_iou'],
                'tile_admitted_count': int(
                    cache['videos'][video_id].get('tile_admitted_count', 0)
                ),
                'tile_admitted_frames': int(
                    cache['videos'][video_id].get('tile_admitted_frames', 0)
                ),
            })
        details[policy] = {
            'mean_st_iou': result['mean_st_iou'],
            'detected_frames': result['detected_frames'],
            'per_video': result['per_video'],
            'identity_scores': identity_scores,
            'identity_deltas': identity_deltas,
            'source_selection_counts': source_counts,
            'global_candidate_profile': diagnostics['global'],
            'by_size_bin': diagnostics['by_size_bin'],
            'by_identity': diagnostics['by_identity'],
        }

    summary = {
        'legacy_predictions_unchanged': legacy_unchanged,
        'oracle_monotonic_for_all_policies': True,
        'cross_validation': cv_summary,
        'policies': details,
        'tile_cache_signature': tile_cache['signature'],
    }
    return result_rows, video_rows, fold_rows, summary, policy_results

# --- YOLO P2 DETECTOR CHECKPOINT LOCKED EVALUATION V1 ---

def build_detector_checkpoint_cache_signature(base_signature, variant_name, model_path):
    signature = dict(base_signature)
    signature['schema_version'] = 6
    signature['yolo_sha256'] = file_sha256(model_path)
    signature['candidate_generation'] = {
        'mode': 'detector_checkpoint_ab_full_frame',
        'variant': variant_name,
        'imgsz': int(P2_AB_IMGSZ),
        'confidence_threshold': CONFIDENCE_THRESHOLD,
        'augment_tta': bool(USE_TTA),
        'ultralytics_version': '8.3.221',
    }
    return signature


def detector_checkpoint_cache_file(variant_name):
    precomputed = P2_AB_PRECOMPUTED_CACHES.get(variant_name, '')
    if precomputed:
        return resolve_precomputed_cache_path(
            precomputed, f'{variant_name} detector candidate cache'
        )
    return os.path.join(P2_AB_OUTPUT_DIR, f'candidate_scores_{variant_name}.json.gz')


def validate_detector_ab_checkpoint(variant_name, config):
    model_path = config.get('model_path', '')
    if not model_path:
        raise ValueError(
            f'Set {variant_name} model path in P2_AB_VARIANTS before running locked evaluation'
        )
    if not os.path.isfile(model_path):
        raise FileNotFoundError(f'{variant_name} checkpoint not found: {model_path}')
    actual_sha = file_sha256(model_path)
    if actual_sha != config['expected_sha256']:
        raise RuntimeError(
            f'{variant_name} SHA mismatch: expected={config["expected_sha256"]}, actual={actual_sha}'
        )
    architecture = inspect_detector_architecture(model_path)
    actual_strides = architecture['detection_strides']
    if actual_strides != config['expected_strides']:
        raise RuntimeError(
            f'{variant_name} stride mismatch: expected={config["expected_strides"]}, '
            f'actual={actual_strides}'
        )
    return architecture


def detector_ab_variant_gate(variant_name, variant, production_variant):
    result = variant['result']
    production_result = production_variant['result']
    production_identity = resolution_identity_scores(production_result)
    identity_scores = resolution_identity_scores(result)
    identity_deltas = {
        identity_id: identity_scores[identity_id] - production_identity[identity_id]
        for identity_id in production_identity
    }
    mean_delta = result['mean_st_iou'] - production_result['mean_st_iou']
    sub16 = profile_sub16_recall(variant['frame_rows'])
    production_sub16 = profile_sub16_recall(production_variant['frame_rows'])
    sub16_delta = sub16 - production_sub16
    improved_count = sum(delta > 0 for delta in identity_deltas.values())
    worst_delta = min(identity_deltas.values())
    fps = variant['cache'].get('variant_runtime', {}).get('aggregate_fps')
    fps_passed = fps is not None and fps >= P2_AB_MIN_AGGREGATE_FPS
    passed = (
        mean_delta >= P2_AB_MIN_MEAN_ST_IOU_DELTA
        and improved_count >= P2_AB_MIN_IMPROVED_IDENTITIES
        and worst_delta >= -P2_AB_MAX_WORST_IDENTITY_DROP
        and sub16_delta >= P2_AB_MIN_SUB16_RECALL_DELTA
        and fps_passed
    )
    return {
        'variant': variant_name,
        'passed': bool(passed),
        'mean_st_iou_delta_vs_production': mean_delta,
        'identity_deltas_vs_production': identity_deltas,
        'improved_identity_count': improved_count,
        'worst_identity_delta': worst_delta,
        'sub16_recall_iou_05': sub16,
        'sub16_recall_iou_05_delta_vs_production': sub16_delta,
        'aggregate_fps': fps,
        'fps_passed': fps_passed,
    }


def run_detector_checkpoint_ab(base_cache, base_signature, gt_map,
                               production_result, temporal_config):
    snapshot = json.dumps(
        production_result['predictions'], sort_keys=True, separators=(',', ':')
    )
    video_ids = list(base_signature['video_folders'])
    video_metadata = collect_video_metadata(video_ids)
    production_architecture = inspect_detector_architecture(YOLO_MODEL_PATH)
    production_frames, production_videos, production_diagnostics = run_detector_profile(
        base_cache, gt_map, production_result,
        video_metadata=video_metadata, detector_architecture=production_architecture,
    )
    variants = {
        'production': {
            'cache': base_cache,
            'cache_file': CANDIDATE_CACHE_FILE,
            'result': production_result,
            'frame_rows': production_frames,
            'profile_videos': production_videos,
            'diagnostics': production_diagnostics,
            'architecture': production_architecture,
        }
    }
    weights = {
        'yolo': WEIGHT_YOLO,
        'siamese': WEIGHT_SIAMESE,
        'color': WEIGHT_COLOR,
    }
    checkpoint_profiles = {}
    for variant_name in ('p3_control', 'p2_stride4'):
        config = P2_AB_VARIANTS[variant_name]
        checkpoint_profiles[variant_name] = validate_detector_ab_checkpoint(
            variant_name, config
        )
        cache_file = detector_checkpoint_cache_file(variant_name)
        cache_signature = build_detector_checkpoint_cache_signature(
            base_signature, variant_name, config['model_path']
        )
        variant_cache = load_or_initialize_resolution_cache(cache_signature, cache_file)
        if P2_AB_PRECOMPUTED_CACHES.get(variant_name) and not variant_cache.get('complete'):
            raise RuntimeError(f'Precomputed {variant_name} candidate cache is incomplete')
        variant_cache = extract_explicit_resolution_cache(
            variant_cache, video_ids, P2_AB_IMGSZ, cache_file,
            model_path=config['model_path'],
        )
        result = evaluate_config(
            variant_cache, gt_map, weights, MATCHING_THRESHOLD,
            include_debug=False, temporal_config=temporal_config,
        )
        frame_rows, profile_videos, diagnostics = run_detector_profile(
            variant_cache, gt_map, result,
            video_metadata=video_metadata,
            detector_architecture=checkpoint_profiles[variant_name],
        )
        variants[variant_name] = {
            'cache': variant_cache,
            'cache_file': cache_file,
            'result': result,
            'frame_rows': frame_rows,
            'profile_videos': profile_videos,
            'diagnostics': diagnostics,
            'architecture': checkpoint_profiles[variant_name],
        }

    unchanged = (
        json.dumps(production_result['predictions'], sort_keys=True, separators=(',', ':'))
        == snapshot
    )
    if not unchanged:
        raise RuntimeError('Detector checkpoint A/B mutated production predictions')

    gates = {
        name: detector_ab_variant_gate(name, variants[name], variants['production'])
        for name in ('p3_control', 'p2_stride4')
    }
    eligible = [name for name, gate in gates.items() if gate['passed']]
    recommended = (
        max(eligible, key=lambda name: variants[name]['result']['mean_st_iou'])
        if eligible else 'production'
    )
    p2_identity = resolution_identity_scores(variants['p2_stride4']['result'])
    p3_identity = resolution_identity_scores(variants['p3_control']['result'])
    p2_vs_p3 = {
        'mean_st_iou_delta': (
            variants['p2_stride4']['result']['mean_st_iou']
            - variants['p3_control']['result']['mean_st_iou']
        ),
        'identity_deltas': {
            identity_id: p2_identity[identity_id] - p3_identity[identity_id]
            for identity_id in p3_identity
        },
        'sub16_recall_iou_05_delta': (
            profile_sub16_recall(variants['p2_stride4']['frame_rows'])
            - profile_sub16_recall(variants['p3_control']['frame_rows'])
        ),
    }

    result_rows, video_rows = [], []
    for variant_name, variant in variants.items():
        row = resolution_variant_row(
            variant_name, P2_AB_IMGSZ, variant['cache'], variant['cache_file'],
            variant['result'], variant['diagnostics'], variant['frame_rows'],
            production_result,
        )
        row['checkpoint_sha256'] = variant['architecture']['checkpoint_sha256']
        row['detection_strides'] = '|'.join(
            str(int(value)) for value in variant['architecture']['detection_strides']
        )
        row['promotion_gate_passed'] = (
            '' if variant_name == 'production' else gates[variant_name]['passed']
        )
        result_rows.append(row)
        video_rows.extend(resolution_video_rows(
            variant_name, P2_AB_IMGSZ, variant['profile_videos'],
            variant['result'], production_result,
        ))

    diagnostics = {
        'production_predictions_unchanged': unchanged,
        'recommended_variant': recommended,
        'promotion_passed': recommended != 'production',
        'variant_gates': gates,
        'p2_vs_paired_p3': p2_vs_p3,
        'criteria': {
            'minimum_mean_st_iou_delta_vs_production': P2_AB_MIN_MEAN_ST_IOU_DELTA,
            'minimum_improved_identities': P2_AB_MIN_IMPROVED_IDENTITIES,
            'maximum_worst_identity_drop': P2_AB_MAX_WORST_IDENTITY_DROP,
            'minimum_sub16_recall_delta_vs_production': P2_AB_MIN_SUB16_RECALL_DELTA,
            'minimum_aggregate_fps': P2_AB_MIN_AGGREGATE_FPS,
        },
        'variants': {
            name: {
                'checkpoint': variant['architecture'],
                'cache_signature': variant['cache']['signature'],
                'cache_file': variant['cache_file'],
                'runtime': variant['cache'].get('variant_runtime'),
                'mean_st_iou': variant['result']['mean_st_iou'],
                'detected_frames': variant['result']['detected_frames'],
                'per_video': variant['result']['per_video'],
                'global_candidate_profile': variant['diagnostics']['global'],
                'by_size_bin': variant['diagnostics']['by_size_bin'],
                'by_identity': variant['diagnostics']['by_identity'],
            }
            for name, variant in variants.items()
        },
    }
    return result_rows, video_rows, diagnostics, variants


# --- HYBRID FULL-FRAME + SAHI TILES V1 ---

def sahi_axis_partitions(length, tile_size, overlap_ratio):
    length = int(length)
    tile_size = int(tile_size)
    if length <= 0 or tile_size <= 0:
        raise ValueError('length and tile_size must be positive')
    if not 0.0 <= float(overlap_ratio) < 1.0:
        raise ValueError('overlap_ratio must be in [0, 1)')
    if length <= tile_size:
        return [{
            'start': 0, 'end': length,
            'owner_start': 0.0, 'owner_end': float(length),
            'is_last': True,
        }]

    max_start = length - tile_size
    stride = max(1, int(round(tile_size * (1.0 - float(overlap_ratio)))))
    starts = list(range(0, max_start + 1, stride))
    if not starts or starts[-1] != max_start:
        starts.append(max_start)
    starts = sorted(set(starts))
    ends = [min(length, start + tile_size) for start in starts]
    partitions = []
    for index, (start, end) in enumerate(zip(starts, ends)):
        owner_start = (
            0.0 if index == 0 else 0.5 * (ends[index - 1] + start)
        )
        owner_end = (
            float(length) if index == len(starts) - 1
            else 0.5 * (end + starts[index + 1])
        )
        partitions.append({
            'start': start,
            'end': end,
            'owner_start': owner_start,
            'owner_end': owner_end,
            'is_last': index == len(starts) - 1,
        })
    return partitions


def build_sahi_tile_layout(frame_width, frame_height):
    x_parts = sahi_axis_partitions(
        frame_width, SAHI_TILE_SIZE, SAHI_OVERLAP_RATIO
    )
    y_parts = sahi_axis_partitions(
        frame_height, SAHI_TILE_SIZE, SAHI_OVERLAP_RATIO
    )
    # A single tile equal to the full frame adds no spatial-scale information.
    if len(x_parts) * len(y_parts) <= 1:
        return []
    layout = []
    for y_index, y_part in enumerate(y_parts):
        for x_index, x_part in enumerate(x_parts):
            layout.append({
                'tile_index': len(layout),
                'x1': x_part['start'], 'x2': x_part['end'],
                'y1': y_part['start'], 'y2': y_part['end'],
                'owner_x1': x_part['owner_start'],
                'owner_x2': x_part['owner_end'],
                'owner_y1': y_part['owner_start'],
                'owner_y2': y_part['owner_end'],
                'is_last_x': x_part['is_last'],
                'is_last_y': y_part['is_last'],
                'x_index': x_index,
                'y_index': y_index,
            })
    return layout


def sahi_center_owned(global_bbox, tile):
    center_x = 0.5 * (float(global_bbox[0]) + float(global_bbox[2]))
    center_y = 0.5 * (float(global_bbox[1]) + float(global_bbox[3]))
    owned_x = (
        center_x >= tile['owner_x1']
        and (
            center_x < tile['owner_x2']
            or (tile['is_last_x'] and center_x <= tile['owner_x2'])
        )
    )
    owned_y = (
        center_y >= tile['owner_y1']
        and (
            center_y < tile['owner_y2']
            or (tile['is_last_y'] and center_y <= tile['owner_y2'])
        )
    )
    return bool(owned_x and owned_y)


def build_sahi_tile_cache_signature(base_signature):
    signature = dict(base_signature)
    signature['schema_version'] = 3
    signature['candidate_generation'] = {
        'mode': 'native_scale_sahi_tiles_only',
        'tile_size': int(SAHI_TILE_SIZE),
        'tile_imgsz': int(SAHI_TILE_IMGSZ),
        'requested_overlap_ratio': float(SAHI_OVERLAP_RATIO),
        'frame_batch_size': int(SAHI_FRAME_BATCH_SIZE),
        'tile_inference_batch_size': int(SAHI_TILE_INFERENCE_BATCH_SIZE),
        'ownership_rule': 'split overlap at midpoint; center belongs to exactly one tile',
        'confidence_threshold': CONFIDENCE_THRESHOLD,
        'augment_tta': bool(USE_TTA),
        'ultralytics_version': '8.3.221',
    }
    return signature


def load_or_initialize_sahi_tile_cache(signature):
    global SAHI_TILE_CACHE_FILE
    precomputed_path = globals().get('PRECOMPUTED_SAHI_TILE_CACHE_FILE', '')
    if precomputed_path:
        SAHI_TILE_CACHE_FILE = resolve_precomputed_cache_path(
            precomputed_path, 'SAHI tile cache'
        )
    if REUSE_CANDIDATE_CACHE and os.path.exists(SAHI_TILE_CACHE_FILE):
        cache = (
            read_cache_json(SAHI_TILE_CACHE_FILE, 'SAHI tile cache')
            if 'read_cache_json' in globals()
            else read_json(SAHI_TILE_CACHE_FILE, compressed=True)
        )
        if cache.get('signature') != signature:
            mismatch_keys = (
                cache_signature_mismatch_keys(cache.get('signature'), signature)
                if 'cache_signature_mismatch_keys' in globals()
                else sorted(set(cache.get('signature', {})) | set(signature))
            )
            raise RuntimeError(
                'SAHI tile cache signature mismatch. '
                f'Differing signature keys: {mismatch_keys}. '
                'Use the tile cache produced by Hybrid SAHI V1 with identical inputs.'
            )
        if precomputed_path and not cache.get('complete'):
            raise RuntimeError(
                'Precomputed SAHI tile cache is incomplete and Kaggle input is read-only: '
                f'{SAHI_TILE_CACHE_FILE}. Upload the final tile cache from a completed run.'
            )
        print(f'Loaded SAHI tile cache: {SAHI_TILE_CACHE_FILE}')
        return cache
    return {
        'schema_version': 3,
        'signature': signature,
        'complete': False,
        'videos': {},
    }


def infer_sahi_frame_batch(frame_records, tile_layout, yolo, siam,
                           ref_emb, ref_hist, base_transform):
    if not frame_records or not tile_layout:
        return {}
    tile_images, tile_mapping = [], []
    for frame_idx, frame in frame_records:
        for tile in tile_layout:
            tile_images.append(frame[tile['y1']:tile['y2'], tile['x1']:tile['x2']])
            tile_mapping.append((frame_idx, tile))
    tile_results = yolo.predict(
        tile_images,
        stream=False,
        batch=min(SAHI_TILE_INFERENCE_BATCH_SIZE, len(tile_images)),
        conf=CONFIDENCE_THRESHOLD,
        verbose=False,
        augment=USE_TTA,
        imgsz=SAHI_TILE_IMGSZ,
    )
    if len(tile_results) != len(tile_mapping):
        raise RuntimeError(
            f'SAHI result count mismatch: {len(tile_results)} vs {len(tile_mapping)}'
        )

    frame_lookup = {frame_idx: frame for frame_idx, frame in frame_records}
    raw_by_frame = {frame_idx: [] for frame_idx, _ in frame_records}
    for (frame_idx, tile), result in zip(tile_mapping, tile_results):
        if not result.boxes:
            continue
        frame = frame_lookup[frame_idx]
        frame_height, frame_width = frame.shape[:2]
        boxes = result.boxes.xyxy.cpu().numpy()
        confidences = result.boxes.conf.cpu().numpy()
        for local_idx, box in enumerate(boxes):
            global_bbox = [
                max(0.0, float(box[0]) + tile['x1']),
                max(0.0, float(box[1]) + tile['y1']),
                min(float(frame_width), float(box[2]) + tile['x1']),
                min(float(frame_height), float(box[3]) + tile['y1']),
            ]
            if not sahi_center_owned(global_bbox, tile):
                continue
            x1, y1, x2, y2 = map(int, global_bbox)
            if x2 <= x1 + 5 or y2 <= y1 + 5:
                continue
            raw_by_frame[frame_idx].append({
                'bbox': global_bbox,
                'yolo_confidence': float(confidences[local_idx]),
                'tile_index': int(tile['tile_index']),
                'crop_box': (x1, y1, x2, y2),
            })

    tensor_records, tensors = [], []
    for frame_idx, raw_candidates in raw_by_frame.items():
        if not raw_candidates:
            continue
        frame = frame_lookup[frame_idx]
        pil_image = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        for item in raw_candidates:
            tensors.append(base_transform(pil_image.crop(item['crop_box'])))
            tensor_records.append((frame_idx, item))

    if not tensors:
        return {}
    siamese_scores = np.zeros(len(tensors), dtype=np.float32)
    color_scores = np.zeros(len(tensors), dtype=np.float32)
    if ref_emb is not None:
        with torch.no_grad():
            features = siam(torch.stack(tensors).to(DEVICE))
            distances = torch.cdist(ref_emb, features).cpu().numpy()[0]
            siamese_scores = np.maximum(0, 1.0 - distances / 2.0)
    if ref_hist is not None:
        for record_idx, (frame_idx, item) in enumerate(tensor_records):
            frame = frame_lookup[frame_idx]
            x1, y1, x2, y2 = item['crop_box']
            color_scores[record_idx] = calculate_color_score(
                frame[y1:y2, x1:x2], ref_hist
            )

    output = {}
    for record_idx, (frame_idx, item) in enumerate(tensor_records):
        output.setdefault(str(frame_idx), []).append({
            'bbox': item['bbox'],
            'scores': [
                item['yolo_confidence'],
                float(siamese_scores[record_idx]),
                float(color_scores[record_idx]),
            ],
            'source': 'sahi_tile',
            'tile_index': item['tile_index'],
        })
    return output


def extract_sahi_tile_cache(cache, video_ids):
    if cache.get('complete'):
        return cache
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()
    invocation_start = time.perf_counter()
    yolo = YOLO(YOLO_MODEL_PATH)
    siam = SiameseMobileNet().to(DEVICE)
    siam.load_state_dict(torch.load(SIAMESE_MODEL_PATH, map_location=DEVICE))
    siam.eval()
    base_transform = get_inference_transforms(size=224)
    print('Loaded models for batched native-scale SAHI tile extraction')

    for video_id in tqdm(video_ids, desc='SAHI tile extraction'):
        if video_id in cache['videos']:
            print(f'  [{video_id}] SAHI tiles already cached')
            continue
        video_path = os.path.join(TEST_DATA_DIR, video_id, 'drone_video.mp4')
        if not os.path.exists(video_path):
            cache['videos'][video_id] = {
                'max_frame_idx': -1, 'has_siamese': False, 'has_color': False,
                'frames': {}, 'processed_frames': 0, 'candidate_count': 0,
                'extraction_elapsed_seconds': 0.0, 'tile_layout': [],
            }
            atomic_write_json(SAHI_TILE_CACHE_FILE, cache, compressed=True)
            continue

        video_start = time.perf_counter()
        ref_emb = load_reference_embeddings(video_id, siam, base_transform)
        ref_hist = load_reference_histogram(video_id)
        capture = cv2.VideoCapture(video_path)
        if not capture.isOpened():
            capture.release()
            raise RuntimeError(f'Cannot open video for SAHI extraction: {video_path}')
        frames = {}
        frame_idx = 0
        candidate_count = 0
        tile_layout = None
        try:
            while True:
                frame_records = []
                for _ in range(SAHI_FRAME_BATCH_SIZE):
                    ok, frame = capture.read()
                    if not ok:
                        break
                    frame_height, frame_width = frame.shape[:2]
                    if tile_layout is None:
                        tile_layout = build_sahi_tile_layout(frame_width, frame_height)
                    frame_records.append((frame_idx, frame))
                    frame_idx += 1
                if not frame_records:
                    break
                batch_frames = infer_sahi_frame_batch(
                    frame_records, tile_layout or [], yolo, siam,
                    ref_emb, ref_hist, base_transform,
                )
                frames.update(batch_frames)
                candidate_count += sum(len(items) for items in batch_frames.values())
        finally:
            capture.release()

        elapsed = time.perf_counter() - video_start
        cache['videos'][video_id] = {
            'max_frame_idx': frame_idx - 1,
            'has_siamese': ref_emb is not None,
            'has_color': ref_hist is not None,
            'frames': frames,
            'processed_frames': frame_idx,
            'candidate_count': candidate_count,
            'extraction_elapsed_seconds': elapsed,
            'tile_layout': tile_layout or [],
        }
        atomic_write_json(SAHI_TILE_CACHE_FILE, cache, compressed=True)
        print(
            f'  [{video_id}] SAHI | frames={len(frames)} | '
            f'candidates={candidate_count} | elapsed={elapsed:.1f}s'
        )

    cache['complete'] = all(video_id in cache['videos'] for video_id in video_ids)
    total_seconds = sum(
        float(video.get('extraction_elapsed_seconds', 0.0))
        for video in cache['videos'].values()
    )
    total_frames = sum(
        int(video.get('processed_frames', video.get('max_frame_idx', -1) + 1))
        for video in cache['videos'].values()
    )
    tile_fps = total_frames / total_seconds if total_seconds > 0 else None
    sequential_hybrid_fps = (
        1.0 / (1.0 / SAHI_BASELINE_REFERENCE_FPS + 1.0 / tile_fps)
        if tile_fps and SAHI_BASELINE_REFERENCE_FPS > 0 else None
    )
    cache['variant_runtime'] = {
        'wall_seconds_this_invocation': time.perf_counter() - invocation_start,
        'total_video_seconds': total_seconds,
        'total_processed_frames': total_frames,
        'frame_batch_size': SAHI_FRAME_BATCH_SIZE,
        'tile_inference_batch_size': SAHI_TILE_INFERENCE_BATCH_SIZE,
        'tile_branch_fps': tile_fps,
        'baseline_reference_fps': SAHI_BASELINE_REFERENCE_FPS,
        'estimated_sequential_hybrid_fps': sequential_hybrid_fps,
        'estimated_parallel_two_gpu_fps': (
            min(SAHI_BASELINE_REFERENCE_FPS, tile_fps) if tile_fps else None
        ),
        'peak_cuda_memory_mb_this_invocation': (
            torch.cuda.max_memory_allocated() / (1024 ** 2)
            if torch.cuda.is_available() else None
        ),
    }
    atomic_write_json(SAHI_TILE_CACHE_FILE, cache, compressed=True)
    if not cache['complete']:
        raise RuntimeError('SAHI tile cache is incomplete')
    del yolo, siam
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return cache


def build_hybrid_sahi_cache(base_cache, tile_cache):
    video_ids = list(base_cache['signature']['video_folders'])
    if video_ids != list(tile_cache['signature']['video_folders']):
        raise RuntimeError('Legacy and SAHI tile cache video lists differ')
    hybrid_signature = dict(base_cache['signature'])
    hybrid_signature['schema_version'] = 4
    hybrid_signature['candidate_generation'] = {
        'mode': 'immutable_legacy_plus_sahi_tile_union',
        'tile_signature': tile_cache['signature']['candidate_generation'],
        'deduplication': 'ownership between tiles; no tile-vs-legacy suppression',
    }
    hybrid = {
        'schema_version': 4,
        'signature': hybrid_signature,
        'complete': bool(base_cache.get('complete') and tile_cache.get('complete')),
        'videos': {},
    }
    for video_id in video_ids:
        base_video = base_cache['videos'][video_id]
        tile_video = tile_cache['videos'][video_id]
        frames = {}
        frame_keys = sorted(
            set(base_video['frames']) | set(tile_video['frames']),
            key=int,
        )
        for frame_key in frame_keys:
            full_candidates = [
                dict(candidate, source=candidate.get('source', 'legacy_full_frame'))
                for candidate in base_video['frames'].get(frame_key, [])
            ]
            tile_candidates = [
                dict(candidate) for candidate in tile_video['frames'].get(frame_key, [])
            ]
            frames[frame_key] = full_candidates + tile_candidates
        hybrid['videos'][video_id] = {
            'max_frame_idx': max(
                base_video.get('max_frame_idx', -1), tile_video.get('max_frame_idx', -1)
            ),
            'has_siamese': bool(base_video.get('has_siamese') or tile_video.get('has_siamese')),
            'has_color': bool(base_video.get('has_color') or tile_video.get('has_color')),
            'frames': frames,
            'legacy_candidate_count': sum(
                len(items) for items in base_video['frames'].values()
            ),
            'tile_candidate_count': sum(
                len(items) for items in tile_video['frames'].values()
            ),
        }
    return hybrid


def candidate_source_selection_counts(cache, weights, matching_threshold):
    counts = {
        'top1_legacy_full_frame': 0,
        'top1_sahi_tile': 0,
        'accepted_legacy_full_frame': 0,
        'accepted_sahi_tile': 0,
    }
    for video in cache['videos'].values():
        threshold = matching_threshold if video['has_siamese'] else YOLO_ONLY_THRESHOLD
        for candidates in video['frames'].values():
            best_candidate, best_score = None, -1.0
            for candidate in candidates:
                score = fuse_candidate_score(
                    candidate['scores'], weights, video['has_siamese'], video['has_color']
                )
                if score > best_score:
                    best_candidate, best_score = candidate, score
            if best_candidate is None:
                continue
            source = best_candidate.get('source', 'legacy_full_frame')
            source_key = 'sahi_tile' if source == 'sahi_tile' else 'legacy_full_frame'
            counts[f'top1_{source_key}'] += 1
            if best_score >= threshold:
                counts[f'accepted_{source_key}'] += 1
    return counts


def run_hybrid_sahi_experiment(base_cache, base_signature, gt_map,
                               baseline_result, temporal_config):
    prediction_snapshot = json.dumps(
        baseline_result['predictions'], sort_keys=True, separators=(',', ':')
    )
    video_ids = list(base_signature['video_folders'])
    tile_signature = build_sahi_tile_cache_signature(base_signature)
    tile_cache = load_or_initialize_sahi_tile_cache(tile_signature)
    tile_cache = extract_sahi_tile_cache(tile_cache, video_ids)
    hybrid_cache = build_hybrid_sahi_cache(base_cache, tile_cache)
    weights = {
        'yolo': WEIGHT_YOLO,
        'siamese': WEIGHT_SIAMESE,
        'color': WEIGHT_COLOR,
    }
    hybrid_result = evaluate_config(
        hybrid_cache,
        gt_map,
        weights,
        MATCHING_THRESHOLD,
        include_debug=False,
        temporal_config=temporal_config,
    )

    video_metadata = collect_video_metadata(video_ids)
    architecture = inspect_detector_architecture()
    baseline_frames, baseline_profile_videos, baseline_diagnostics = run_detector_profile(
        base_cache, gt_map, baseline_result,
        video_metadata=video_metadata, detector_architecture=architecture,
    )
    hybrid_frames, hybrid_profile_videos, hybrid_diagnostics = run_detector_profile(
        hybrid_cache, gt_map, hybrid_result,
        video_metadata=video_metadata, detector_architecture=architecture,
    )
    baseline_oracle = {
        (row['video_id'], row['frame']): row['oracle_iou'] for row in baseline_frames
    }
    hybrid_oracle = {
        (row['video_id'], row['frame']): row['oracle_iou'] for row in hybrid_frames
    }
    if set(baseline_oracle) != set(hybrid_oracle):
        raise RuntimeError('Hybrid SAHI changed GT frame coverage')
    oracle_deltas = [
        hybrid_oracle[key] - baseline_oracle[key] for key in baseline_oracle
    ]
    oracle_regression_count = sum(
        delta < -SAHI_ORACLE_TOLERANCE for delta in oracle_deltas
    )
    oracle_monotonic = oracle_regression_count == 0
    if not oracle_monotonic:
        raise RuntimeError(
            f'Hybrid candidate oracle regressed on {oracle_regression_count} GT frames'
        )

    baseline_identity = resolution_identity_scores(baseline_result)
    hybrid_identity = resolution_identity_scores(hybrid_result)
    identity_deltas = {
        identity_id: hybrid_identity[identity_id] - baseline_identity[identity_id]
        for identity_id in baseline_identity
    }
    improved_identity_count = sum(delta > 0 for delta in identity_deltas.values())
    worst_identity_delta = min(identity_deltas.values())
    mean_delta = hybrid_result['mean_st_iou'] - baseline_result['mean_st_iou']
    baseline_sub16 = profile_sub16_recall(baseline_frames)
    hybrid_sub16 = profile_sub16_recall(hybrid_frames)
    sub16_delta = hybrid_sub16 - baseline_sub16
    global_recall_delta = (
        hybrid_diagnostics['global']['oracle_recall_iou_05']
        - baseline_diagnostics['global']['oracle_recall_iou_05']
    )
    promotion_passed = (
        oracle_monotonic
        and global_recall_delta >= -SAHI_ORACLE_TOLERANCE
        and mean_delta >= SAHI_MIN_MEAN_ST_IOU_DELTA
        and improved_identity_count >= SAHI_MIN_IMPROVED_IDENTITIES
        and worst_identity_delta >= -SAHI_MAX_WORST_IDENTITY_DROP
        and sub16_delta >= SAHI_MIN_SUB16_RECALL_DELTA
    )

    legacy_unchanged = (
        json.dumps(baseline_result['predictions'], sort_keys=True, separators=(',', ':'))
        == prediction_snapshot
    )
    if not legacy_unchanged:
        raise RuntimeError('Hybrid SAHI mutated production predictions')

    def cache_candidate_count(cache):
        return sum(
            sum(len(items) for items in video['frames'].values())
            for video in cache['videos'].values()
        )

    runtime = tile_cache.get('variant_runtime', {})
    result_rows = []
    for variant_name, cache, result, diagnostics, frame_rows in (
        ('legacy', base_cache, baseline_result, baseline_diagnostics, baseline_frames),
        ('hybrid_sahi', hybrid_cache, hybrid_result, hybrid_diagnostics, hybrid_frames),
    ):
        result_rows.append({
            'variant': variant_name,
            'mean_st_iou': result['mean_st_iou'],
            'delta_st_iou_vs_legacy': result['mean_st_iou'] - baseline_result['mean_st_iou'],
            'detected_frames': result['detected_frames'],
            'total_candidate_count': cache_candidate_count(cache),
            'no_candidate_gt_frames': diagnostics['global']['no_candidate_frames'],
            'mean_oracle_iou': diagnostics['global']['mean_oracle_iou'],
            'oracle_recall_iou_03': diagnostics['global']['oracle_recall_iou_03'],
            'oracle_recall_iou_05': diagnostics['global']['oracle_recall_iou_05'],
            'oracle_recall_iou_07': diagnostics['global']['oracle_recall_iou_07'],
            'sub16_oracle_recall_iou_05': profile_sub16_recall(frame_rows),
            'tile_branch_fps': runtime.get('tile_branch_fps') if variant_name == 'hybrid_sahi' else None,
            'estimated_sequential_hybrid_fps': (
                runtime.get('estimated_sequential_hybrid_fps')
                if variant_name == 'hybrid_sahi' else None
            ),
            'estimated_parallel_two_gpu_fps': (
                runtime.get('estimated_parallel_two_gpu_fps')
                if variant_name == 'hybrid_sahi' else None
            ),
        })

    baseline_profile_by_video = {
        row['video_id']: row for row in baseline_profile_videos
    }
    hybrid_profile_by_video = {
        row['video_id']: row for row in hybrid_profile_videos
    }
    video_rows = []
    for video_id in video_ids:
        baseline_profile = baseline_profile_by_video[video_id]
        hybrid_profile = hybrid_profile_by_video[video_id]
        video_rows.append({
            'video_id': video_id,
            'identity_id': identity_id_from_video_id(video_id),
            'legacy_st_iou': baseline_result['per_video'][video_id],
            'hybrid_st_iou': hybrid_result['per_video'][video_id],
            'delta_st_iou': (
                hybrid_result['per_video'][video_id] - baseline_result['per_video'][video_id]
            ),
            'legacy_candidate_count': hybrid_cache['videos'][video_id]['legacy_candidate_count'],
            'tile_candidate_count': hybrid_cache['videos'][video_id]['tile_candidate_count'],
            'legacy_no_candidate_gt_frames': baseline_profile['no_candidate_frames'],
            'hybrid_no_candidate_gt_frames': hybrid_profile['no_candidate_frames'],
            'legacy_oracle_recall_iou_05': baseline_profile['oracle_recall_iou_05'],
            'hybrid_oracle_recall_iou_05': hybrid_profile['oracle_recall_iou_05'],
            'oracle_recall_iou_05_delta': (
                hybrid_profile['oracle_recall_iou_05']
                - baseline_profile['oracle_recall_iou_05']
            ),
            'legacy_mean_oracle_iou': baseline_profile['mean_oracle_iou'],
            'hybrid_mean_oracle_iou': hybrid_profile['mean_oracle_iou'],
        })

    diagnostics = {
        'legacy_predictions_unchanged': legacy_unchanged,
        'oracle_monotonic_per_gt_frame': oracle_monotonic,
        'oracle_regression_count': oracle_regression_count,
        'minimum_per_frame_oracle_iou_delta': min(oracle_deltas) if oracle_deltas else None,
        'improved_oracle_gt_frames': sum(delta > SAHI_ORACLE_TOLERANCE for delta in oracle_deltas),
        'source_selection_counts': candidate_source_selection_counts(
            hybrid_cache, weights, MATCHING_THRESHOLD
        ),
        'promotion': {
            'passed': promotion_passed,
            'mean_st_iou_delta': mean_delta,
            'identity_deltas': identity_deltas,
            'improved_identity_count': improved_identity_count,
            'worst_identity_delta': worst_identity_delta,
            'global_oracle_recall_iou_05_delta': global_recall_delta,
            'sub16_recall_iou_05_baseline': baseline_sub16,
            'sub16_recall_iou_05_hybrid': hybrid_sub16,
            'sub16_recall_iou_05_delta': sub16_delta,
            'criteria': {
                'minimum_mean_st_iou_delta': SAHI_MIN_MEAN_ST_IOU_DELTA,
                'minimum_improved_identities': SAHI_MIN_IMPROVED_IDENTITIES,
                'maximum_worst_identity_drop': SAHI_MAX_WORST_IDENTITY_DROP,
                'minimum_sub16_recall_iou_05_delta': SAHI_MIN_SUB16_RECALL_DELTA,
                'oracle_must_be_monotonic_per_gt_frame': True,
            },
        },
        'tile_runtime': runtime,
        'tile_signature': tile_cache['signature'],
        'detector_architecture': architecture,
        'baseline_candidate_profile': {
            'global': baseline_diagnostics['global'],
            'by_size_bin': baseline_diagnostics['by_size_bin'],
            'by_identity': baseline_diagnostics['by_identity'],
        },
        'hybrid_candidate_profile': {
            'global': hybrid_diagnostics['global'],
            'by_size_bin': hybrid_diagnostics['by_size_bin'],
            'by_identity': hybrid_diagnostics['by_identity'],
        },
    }
    return result_rows, video_rows, diagnostics, hybrid_result

# --- EXPLICIT RESOLUTION CANDIDATE A/B V1 ---

def build_resolution_cache_signature(base_signature, imgsz):
    signature = dict(base_signature)
    signature['schema_version'] = 2
    signature['candidate_generation'] = {
        'mode': 'full_frame_explicit_resolution',
        'imgsz': int(imgsz),
        'confidence_threshold': CONFIDENCE_THRESHOLD,
        'augment_tta': bool(USE_TTA),
        'ultralytics_version': '8.3.221',
    }
    return signature


def resolution_cache_file(imgsz):
    return os.path.join(
        RESOLUTION_OUTPUT_DIR, f'candidate_scores_imgsz_{int(imgsz)}.json.gz'
    )


def load_or_initialize_resolution_cache(signature, cache_file):
    if REUSE_CANDIDATE_CACHE and os.path.exists(cache_file):
        cache = read_json(cache_file, compressed=True)
        if cache.get('signature') != signature:
            raise RuntimeError(
                f'Resolution cache signature mismatch: {cache_file}. '
                'Use a new output directory or remove only this variant cache.'
            )
        print(f'Loaded resolution candidate cache: {cache_file}')
        return cache
    return {
        'schema_version': 2,
        'signature': signature,
        'complete': False,
        'videos': {},
    }


def extract_explicit_resolution_cache(cache, video_folders, imgsz, cache_file,
                                      model_path=None):
    if cache.get('complete'):
        return cache
    if int(imgsz) <= 0:
        raise ValueError(f'imgsz must be positive, got {imgsz}')

    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()
    variant_start = time.perf_counter()
    selected_model_path = model_path or YOLO_MODEL_PATH
    yolo = YOLO(selected_model_path)
    siam = SiameseMobileNet().to(DEVICE)
    siam.load_state_dict(torch.load(SIAMESE_MODEL_PATH, map_location=DEVICE))
    siam.eval()
    base_transform = get_inference_transforms(size=224)
    print(f'Loaded models for explicit imgsz={imgsz} candidate extraction')

    for video_id in tqdm(video_folders, desc=f'Candidate extraction imgsz={imgsz}'):
        if video_id in cache['videos']:
            print(f'  [{video_id}] imgsz={imgsz} already cached')
            continue
        video_path = os.path.join(TEST_DATA_DIR, video_id, 'drone_video.mp4')
        if not os.path.exists(video_path):
            cache['videos'][video_id] = {
                'max_frame_idx': -1,
                'has_siamese': False,
                'has_color': False,
                'frames': {},
                'explicit_imgsz': int(imgsz),
                'processed_frames': 0,
                'extraction_elapsed_seconds': 0.0,
                'candidate_count': 0,
            }
            atomic_write_json(cache_file, cache, compressed=True)
            continue

        video_start = time.perf_counter()
        ref_emb = load_reference_embeddings(video_id, siam, base_transform)
        ref_hist = load_reference_histogram(video_id)
        results = yolo.predict(
            video_path,
            stream=True,
            conf=CONFIDENCE_THRESHOLD,
            verbose=False,
            augment=USE_TTA,
            imgsz=int(imgsz),
        )
        frames = {}
        max_frame_idx = -1
        candidate_count = 0

        for frame_idx, result in enumerate(results):
            max_frame_idx = frame_idx
            if not result.boxes:
                continue
            boxes = result.boxes.xyxy.cpu().numpy()
            confidences = result.boxes.conf.cpu().numpy()
            original_image = result.orig_img
            height, width = original_image.shape[:2]
            pil_image = Image.fromarray(cv2.cvtColor(original_image, cv2.COLOR_BGR2RGB))
            tensors, crops, valid_indices = [], [], []
            for candidate_idx, box in enumerate(boxes):
                x1, y1, x2, y2 = map(int, box)
                x1, y1 = max(0, x1), max(0, y1)
                x2, y2 = min(width, x2), min(height, y2)
                if x2 <= x1 + 5 or y2 <= y1 + 5:
                    continue
                valid_indices.append(candidate_idx)
                tensors.append(base_transform(pil_image.crop((x1, y1, x2, y2))))
                crops.append(original_image[y1:y2, x1:x2])
            if not tensors:
                continue

            siamese_scores = np.zeros(len(tensors), dtype=np.float32)
            color_scores = np.zeros(len(tensors), dtype=np.float32)
            if ref_emb is not None:
                with torch.no_grad():
                    features = siam(torch.stack(tensors).to(DEVICE))
                    distances = torch.cdist(ref_emb, features).cpu().numpy()[0]
                    siamese_scores = np.maximum(0, 1.0 - distances / 2.0)
            if ref_hist is not None:
                for local_idx, crop in enumerate(crops):
                    color_scores[local_idx] = calculate_color_score(crop, ref_hist)

            frame_candidates = [
                {
                    'bbox': boxes[candidate_idx].tolist(),
                    'scores': [
                        float(confidences[candidate_idx]),
                        float(siamese_scores[local_idx]),
                        float(color_scores[local_idx]),
                    ],
                }
                for local_idx, candidate_idx in enumerate(valid_indices)
            ]
            frames[str(frame_idx)] = frame_candidates
            candidate_count += len(frame_candidates)

        elapsed = time.perf_counter() - video_start
        cache['videos'][video_id] = {
            'max_frame_idx': max_frame_idx,
            'has_siamese': ref_emb is not None,
            'has_color': ref_hist is not None,
            'frames': frames,
            'explicit_imgsz': int(imgsz),
            'processed_frames': max_frame_idx + 1,
            'extraction_elapsed_seconds': elapsed,
            'candidate_count': candidate_count,
        }
        atomic_write_json(cache_file, cache, compressed=True)
        print(
            f'  [{video_id}] imgsz={imgsz} | candidate frames={len(frames)} | '
            f'candidates={candidate_count} | elapsed={elapsed:.1f}s'
        )

    cache['complete'] = all(video_id in cache['videos'] for video_id in video_folders)
    cache['variant_runtime'] = {
        'imgsz': int(imgsz),
        'wall_seconds_this_invocation': time.perf_counter() - variant_start,
        'total_video_seconds': sum(
            float(video.get('extraction_elapsed_seconds', 0.0))
            for video in cache['videos'].values()
        ),
        'total_processed_frames': sum(
            int(video.get('processed_frames', video.get('max_frame_idx', -1) + 1))
            for video in cache['videos'].values()
        ),
        'peak_cuda_memory_mb_this_invocation': (
            torch.cuda.max_memory_allocated() / (1024 ** 2)
            if torch.cuda.is_available() else None
        ),
    }
    runtime = cache['variant_runtime']
    runtime['aggregate_fps'] = (
        runtime['total_processed_frames'] / runtime['total_video_seconds']
        if runtime['total_video_seconds'] > 0 else None
    )
    atomic_write_json(cache_file, cache, compressed=True)
    if not cache['complete']:
        raise RuntimeError(f'Explicit imgsz={imgsz} cache is incomplete')
    del yolo, siam
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return cache


def resolution_identity_scores(result):
    groups = {}
    for video_id, score in result['per_video'].items():
        groups.setdefault(identity_id_from_video_id(video_id), []).append(float(score))
    return {
        identity_id: sum(scores) / len(scores)
        for identity_id, scores in sorted(groups.items())
    }


def profile_sub16_recall(frame_rows, identity_id=None):
    relevant = [
        row for row in frame_rows
        if row['projected_min_side_px'] < 16.0
        and (identity_id is None or row['identity_id'] == identity_id)
    ]
    return (
        sum(row['oracle_iou_ge_05'] for row in relevant) / len(relevant)
        if relevant else None
    )


def resolution_variant_row(name, imgsz, cache, cache_file, result, diagnostics,
                           frame_rows, legacy_result):
    runtime = cache.get('variant_runtime', {})
    global_profile = diagnostics['global']
    return {
        'variant': name,
        'imgsz': '' if imgsz is None else int(imgsz),
        'cache_file': cache_file,
        'mean_st_iou': result['mean_st_iou'],
        'delta_st_iou_vs_legacy': result['mean_st_iou'] - legacy_result['mean_st_iou'],
        'detected_frames': result['detected_frames'],
        'oracle_recall_iou_03': global_profile['oracle_recall_iou_03'],
        'oracle_recall_iou_05': global_profile['oracle_recall_iou_05'],
        'oracle_recall_iou_07': global_profile['oracle_recall_iou_07'],
        'sub16_oracle_recall_iou_05': profile_sub16_recall(frame_rows),
        'no_candidate_frames': global_profile['no_candidate_frames'],
        'mean_candidate_count_on_gt': global_profile['mean_candidate_count'],
        'total_candidate_count': sum(
            sum(len(items) for items in video['frames'].values())
            for video in cache['videos'].values()
        ),
        'total_video_seconds': runtime.get('total_video_seconds'),
        'aggregate_fps': runtime.get('aggregate_fps'),
        'peak_cuda_memory_mb': runtime.get('peak_cuda_memory_mb_this_invocation'),
    }


def resolution_video_rows(variant_name, imgsz, profile_video_rows, result,
                          legacy_result):
    output = []
    for profile_row in profile_video_rows:
        video_id = profile_row['video_id']
        row = {
            'variant': variant_name,
            'imgsz': '' if imgsz is None else int(imgsz),
            'video_id': video_id,
            'identity_id': profile_row['identity_id'],
            'final_st_iou': result['per_video'][video_id],
            'delta_st_iou_vs_legacy': (
                result['per_video'][video_id] - legacy_result['per_video'][video_id]
            ),
        }
        for key in (
            'gt_frames', 'no_candidate_frames', 'mean_candidate_count',
            'mean_oracle_iou', 'oracle_recall_iou_03',
            'oracle_recall_iou_05', 'oracle_recall_iou_07',
        ):
            row[key] = profile_row[key]
        output.append(row)
    return output


def run_resolution_candidate_ab(base_cache, base_signature, gt_map,
                                legacy_result, temporal_config):
    prediction_snapshot = json.dumps(
        legacy_result['predictions'], sort_keys=True, separators=(',', ':')
    )
    video_ids = list(base_signature['video_folders'])
    video_metadata = collect_video_metadata(video_ids)
    detector_architecture = inspect_detector_architecture()

    legacy_frame_rows, legacy_profile_videos, legacy_diagnostics = run_detector_profile(
        base_cache,
        gt_map,
        legacy_result,
        video_metadata=video_metadata,
        detector_architecture=detector_architecture,
    )
    legacy_diagnostics['detector_inference_call'] = {
        'explicit_imgsz_passed_to_yolo_predict': False,
        'imgsz': None,
        'reference_imgsz_for_fixed_cohort_projection': RESOLUTION_PROFILE_REFERENCE_IMGSZ,
        'confidence_threshold': CONFIDENCE_THRESHOLD,
        'augment_tta': USE_TTA,
    }
    variants = {
        'legacy': {
            'imgsz': None,
            'cache': base_cache,
            'cache_file': CANDIDATE_CACHE_FILE,
            'result': legacy_result,
            'frame_rows': legacy_frame_rows,
            'profile_videos': legacy_profile_videos,
            'diagnostics': legacy_diagnostics,
        }
    }

    weights = {
        'yolo': WEIGHT_YOLO,
        'siamese': WEIGHT_SIAMESE,
        'color': WEIGHT_COLOR,
    }
    for imgsz in RESOLUTION_VARIANTS:
        variant_name = f'explicit_{int(imgsz)}'
        cache_file = resolution_cache_file(imgsz)
        signature = build_resolution_cache_signature(base_signature, imgsz)
        variant_cache = load_or_initialize_resolution_cache(signature, cache_file)
        variant_cache = extract_explicit_resolution_cache(
            variant_cache, video_ids, imgsz, cache_file
        )
        result = evaluate_config(
            variant_cache,
            gt_map,
            weights,
            MATCHING_THRESHOLD,
            include_debug=False,
            temporal_config=temporal_config,
        )
        frame_rows, profile_videos, diagnostics = run_detector_profile(
            variant_cache,
            gt_map,
            result,
            video_metadata=video_metadata,
            detector_architecture=detector_architecture,
        )
        diagnostics['detector_inference_call'] = {
            'explicit_imgsz_passed_to_yolo_predict': True,
            'imgsz': int(imgsz),
            'reference_imgsz_for_fixed_cohort_projection': RESOLUTION_PROFILE_REFERENCE_IMGSZ,
            'confidence_threshold': CONFIDENCE_THRESHOLD,
            'augment_tta': USE_TTA,
        }
        variants[variant_name] = {
            'imgsz': int(imgsz),
            'cache': variant_cache,
            'cache_file': cache_file,
            'result': result,
            'frame_rows': frame_rows,
            'profile_videos': profile_videos,
            'diagnostics': diagnostics,
        }

    if 'explicit_640' not in variants or 'explicit_960' not in variants:
        raise RuntimeError('Resolution A/B V1 requires explicit 640 and 960 variants')

    control_result = variants['explicit_640']['result']
    control_mean_abs_delta = abs(control_result['mean_st_iou'] - legacy_result['mean_st_iou'])
    control_max_video_abs_delta = max(
        abs(control_result['per_video'][video_id] - legacy_result['per_video'][video_id])
        for video_id in video_ids
    )
    legacy_global_profile = variants['legacy']['diagnostics']['global']
    control_global_profile = variants['explicit_640']['diagnostics']['global']
    control_candidate_count_equal = (
        sum(
            sum(len(items) for items in video['frames'].values())
            for video in variants['explicit_640']['cache']['videos'].values()
        )
        == sum(
            sum(len(items) for items in video['frames'].values())
            for video in variants['legacy']['cache']['videos'].values()
        )
    )
    control_recall_abs_delta = abs(
        control_global_profile['oracle_recall_iou_05']
        - legacy_global_profile['oracle_recall_iou_05']
    )
    legacy_sub16 = profile_sub16_recall(variants['legacy']['frame_rows'])
    control_sub16 = profile_sub16_recall(variants['explicit_640']['frame_rows'])
    control_sub16_abs_delta = abs(control_sub16 - legacy_sub16)
    control_passed = (
        control_mean_abs_delta <= RESOLUTION_CONTROL_TOLERANCE
        and control_max_video_abs_delta <= RESOLUTION_CONTROL_TOLERANCE
        and control_candidate_count_equal
        and control_recall_abs_delta <= RESOLUTION_CONTROL_TOLERANCE
        and control_sub16_abs_delta <= RESOLUTION_CONTROL_TOLERANCE
    )

    candidate_result = variants['explicit_960']['result']
    control_identity = resolution_identity_scores(control_result)
    candidate_identity = resolution_identity_scores(candidate_result)
    identity_deltas = {
        identity_id: candidate_identity[identity_id] - control_identity[identity_id]
        for identity_id in control_identity
    }
    improved_identity_count = sum(delta > 0 for delta in identity_deltas.values())
    worst_identity_delta = min(identity_deltas.values())
    mean_delta_vs_control = candidate_result['mean_st_iou'] - control_result['mean_st_iou']
    candidate_sub16 = profile_sub16_recall(variants['explicit_960']['frame_rows'])
    sub16_delta = candidate_sub16 - control_sub16
    promotion_passed = (
        control_passed
        and mean_delta_vs_control >= RESOLUTION_MIN_MEAN_ST_IOU_DELTA
        and improved_identity_count >= RESOLUTION_MIN_IMPROVED_IDENTITIES
        and worst_identity_delta >= -RESOLUTION_MAX_WORST_IDENTITY_DROP
        and sub16_delta >= RESOLUTION_MIN_SUB16_RECALL_DELTA
    )

    legacy_unchanged = (
        json.dumps(legacy_result['predictions'], sort_keys=True, separators=(',', ':'))
        == prediction_snapshot
    )
    if not legacy_unchanged:
        raise RuntimeError('Resolution A/B mutated legacy production predictions')

    result_rows, video_rows = [], []
    for variant_name, variant in variants.items():
        result_rows.append(resolution_variant_row(
            variant_name,
            variant['imgsz'],
            variant['cache'],
            variant['cache_file'],
            variant['result'],
            variant['diagnostics'],
            variant['frame_rows'],
            legacy_result,
        ))
        video_rows.extend(resolution_video_rows(
            variant_name,
            variant['imgsz'],
            variant['profile_videos'],
            variant['result'],
            legacy_result,
        ))

    diagnostics = {
        'legacy_predictions_unchanged': legacy_unchanged,
        'fixed_cohort_reference_imgsz': RESOLUTION_PROFILE_REFERENCE_IMGSZ,
        'control': {
            'passed': control_passed,
            'tolerance': RESOLUTION_CONTROL_TOLERANCE,
            'mean_st_iou_abs_delta': control_mean_abs_delta,
            'max_per_video_st_iou_abs_delta': control_max_video_abs_delta,
            'total_candidate_count_equal': control_candidate_count_equal,
            'oracle_recall_iou_05_abs_delta': control_recall_abs_delta,
            'sub16_recall_iou_05_abs_delta': control_sub16_abs_delta,
        },
        'promotion': {
            'passed': promotion_passed,
            'candidate_variant': 'explicit_960',
            'baseline_variant': 'explicit_640',
            'mean_st_iou_delta': mean_delta_vs_control,
            'identity_deltas': identity_deltas,
            'improved_identity_count': improved_identity_count,
            'worst_identity_delta': worst_identity_delta,
            'sub16_recall_iou_05_control': control_sub16,
            'sub16_recall_iou_05_candidate': candidate_sub16,
            'sub16_recall_iou_05_delta': sub16_delta,
            'criteria': {
                'minimum_mean_st_iou_delta': RESOLUTION_MIN_MEAN_ST_IOU_DELTA,
                'minimum_improved_identities': RESOLUTION_MIN_IMPROVED_IDENTITIES,
                'maximum_worst_identity_drop': RESOLUTION_MAX_WORST_IDENTITY_DROP,
                'minimum_sub16_recall_iou_05_delta': RESOLUTION_MIN_SUB16_RECALL_DELTA,
            },
        },
        'detector_architecture': detector_architecture,
        'variants': {
            name: {
                'imgsz': variant['imgsz'],
                'cache_signature': variant['cache']['signature'],
                'cache_file': variant['cache_file'],
                'runtime': variant['cache'].get('variant_runtime'),
                'mean_st_iou': variant['result']['mean_st_iou'],
                'detected_frames': variant['result']['detected_frames'],
                'per_video': variant['result']['per_video'],
                'global_candidate_profile': variant['diagnostics']['global'],
                'by_size_bin': variant['diagnostics']['by_size_bin'],
                'by_identity': variant['diagnostics']['by_identity'],
                'sub16_recall_iou_05_by_identity': {
                    identity_id: profile_sub16_recall(variant['frame_rows'], identity_id)
                    for identity_id in sorted({
                        row['identity_id'] for row in variant['frame_rows']
                    })
                },
            }
            for name, variant in variants.items()
        },
    }
    return result_rows, video_rows, diagnostics, variants

# --- DETECTOR PROFILE V1: read-only candidate-generation diagnostics ---

def detector_profile_size_bin(projected_min_side_px):
    value = float(projected_min_side_px)
    for name, lower, upper in DETECTOR_PROFILE_SIZE_BINS:
        if value >= lower and (upper is None or value < upper):
            return name
    raise RuntimeError(f'No detector size bin for value: {value}')


def collect_video_metadata(video_ids):
    metadata = {}
    for video_id in video_ids:
        video_path = os.path.join(TEST_DATA_DIR, video_id, 'drone_video.mp4')
        cap = cv2.VideoCapture(video_path)
        if not cap.isOpened():
            cap.release()
            raise RuntimeError(f'Cannot open video for detector profiling: {video_path}')
        width = int(round(cap.get(cv2.CAP_PROP_FRAME_WIDTH)))
        height = int(round(cap.get(cv2.CAP_PROP_FRAME_HEIGHT)))
        frame_count = int(round(cap.get(cv2.CAP_PROP_FRAME_COUNT)))
        fps = float(cap.get(cv2.CAP_PROP_FPS))
        cap.release()
        if width <= 0 or height <= 0 or frame_count <= 0:
            raise RuntimeError(
                f'Invalid video metadata for {video_id}: '
                f'{width}x{height}, frames={frame_count}'
            )
        metadata[video_id] = {
            'width': width,
            'height': height,
            'frame_count': frame_count,
            'fps': fps,
        }
    return metadata


def inspect_detector_architecture(model_path=None):
    selected_model_path = model_path or YOLO_MODEL_PATH
    detector = YOLO(selected_model_path)
    model = getattr(detector, 'model', None)
    raw_stride = getattr(model, 'stride', [])
    if hasattr(raw_stride, 'detach'):
        raw_stride = raw_stride.detach().cpu().tolist()
    elif hasattr(raw_stride, 'tolist'):
        raw_stride = raw_stride.tolist()
    elif isinstance(raw_stride, (int, float)):
        raw_stride = [raw_stride]
    strides = sorted({float(value) for value in (raw_stride or [])})
    yaml_config = getattr(model, 'yaml', {}) or {}
    layers = getattr(model, 'model', [])
    head_type = type(layers[-1]).__name__ if len(layers) else None
    minimum_stride = min(strides) if strides else None
    return {
        'checkpoint_path': selected_model_path,
        'checkpoint_sha256': file_sha256(selected_model_path),
        'ultralytics_version_pinned_by_notebook': '8.3.221',
        'task': getattr(detector, 'task', None),
        'model_class': type(model).__name__ if model is not None else None,
        'head_type': head_type,
        'layer_count': len(layers),
        'detection_strides': strides,
        'minimum_detection_stride': minimum_stride,
        'has_p2_stride_4_or_finer': bool(minimum_stride is not None and minimum_stride <= 4.0),
        'has_p3_stride_8_or_finer': bool(minimum_stride is not None and minimum_stride <= 8.0),
        'yaml_nc': yaml_config.get('nc'),
        'yaml_scale': yaml_config.get('scale'),
    }


def aggregate_detector_profile_rows(rows):
    count = len(rows)
    safe_count = max(1, count)
    output = {
        'gt_frames': count,
        'no_candidate_frames': sum(row['candidate_count'] == 0 for row in rows),
        'mean_candidate_count': sum(row['candidate_count'] for row in rows) / safe_count,
        'mean_oracle_iou': sum(row['oracle_iou'] for row in rows) / safe_count,
        'mean_projected_min_side_px': (
            sum(row['projected_min_side_px'] for row in rows) / safe_count
        ),
        'mean_relative_area': sum(row['relative_area'] for row in rows) / safe_count,
    }
    output['no_candidate_rate'] = output['no_candidate_frames'] / safe_count
    for threshold in DETECTOR_PROFILE_IOU_THRESHOLDS:
        suffix = str(threshold).replace('.', '')
        recalled = sum(row[f'oracle_iou_ge_{suffix}'] for row in rows)
        output[f'oracle_recall_iou_{suffix}'] = recalled / safe_count
        output[f'localization_miss_iou_{suffix}'] = count - recalled
    return output


def group_detector_profile_rows(rows, key):
    groups = {}
    for row in rows:
        groups.setdefault(row[key], []).append(row)
    return {
        group_name: aggregate_detector_profile_rows(group_rows)
        for group_name, group_rows in sorted(groups.items())
    }


def detector_profile_video_rows(frame_rows, cache, video_metadata):
    rows_by_video = {}
    for row in frame_rows:
        rows_by_video.setdefault(row['video_id'], []).append(row)
    output = []
    for video_id in cache['signature']['video_folders']:
        video_cache = cache['videos'][video_id]
        metadata = video_metadata[video_id]
        gt_rows = rows_by_video.get(video_id, [])
        aggregate = aggregate_detector_profile_rows(gt_rows)
        candidate_count = sum(
            len(candidates) for candidates in video_cache['frames'].values()
        )
        row = {
            'video_id': video_id,
            'identity_id': identity_id_from_video_id(video_id),
            'width': metadata['width'],
            'height': metadata['height'],
            'fps': metadata['fps'],
            'video_frame_count': metadata['frame_count'],
            'cache_max_frame_idx': video_cache['max_frame_idx'],
            'candidate_frames': len(video_cache['frames']),
            'candidate_count': candidate_count,
            'candidates_per_video_frame': candidate_count / max(1, metadata['frame_count']),
        }
        row.update(aggregate)
        output.append(row)
    return output


def run_detector_profile(cache, gt_map, production_result,
                         video_metadata=None, detector_architecture=None):
    prediction_snapshot = json.dumps(
        production_result['predictions'], sort_keys=True, separators=(',', ':')
    )
    video_ids = list(cache['signature']['video_folders'])
    if video_metadata is None:
        video_metadata = collect_video_metadata(video_ids)
    if detector_architecture is None:
        detector_architecture = inspect_detector_architecture()

    frame_rows = []
    for video_id in video_ids:
        metadata = video_metadata[video_id]
        frame_width = metadata['width']
        frame_height = metadata['height']
        scale_to_reference = min(
            DETECTOR_PROFILE_REFERENCE_IMGSZ / frame_width,
            DETECTOR_PROFILE_REFERENCE_IMGSZ / frame_height,
        )
        video_cache = cache['videos'][video_id]
        for frame_idx, gt_bbox in sorted(gt_map.get(video_id, {}).items()):
            x1, y1, x2, y2 = map(float, gt_bbox)
            bbox_width = max(0.0, x2 - x1)
            bbox_height = max(0.0, y2 - y1)
            bbox_area = bbox_width * bbox_height
            min_side = min(bbox_width, bbox_height)
            projected_min_side = min_side * scale_to_reference
            candidates = video_cache['frames'].get(str(frame_idx), [])
            oracle_idx, oracle_iou = oracle_candidate_for_gt(candidates, gt_bbox)
            oracle_yolo = (
                float(candidates[oracle_idx]['scores'][0]) if oracle_idx is not None else None
            )
            row = {
                'video_id': video_id,
                'identity_id': identity_id_from_video_id(video_id),
                'frame': int(frame_idx),
                'frame_width': frame_width,
                'frame_height': frame_height,
                'bbox_width': bbox_width,
                'bbox_height': bbox_height,
                'bbox_area': bbox_area,
                'min_side_px': min_side,
                'sqrt_area_px': math.sqrt(bbox_area),
                'relative_area': bbox_area / max(1.0, frame_width * frame_height),
                'projected_min_side_px': projected_min_side,
                'size_bin': detector_profile_size_bin(projected_min_side),
                'candidate_count': len(candidates),
                'oracle_candidate_idx': oracle_idx,
                'oracle_iou': oracle_iou,
                'oracle_yolo_confidence': oracle_yolo,
            }
            for threshold in DETECTOR_PROFILE_IOU_THRESHOLDS:
                suffix = str(threshold).replace('.', '')
                row[f'oracle_iou_ge_{suffix}'] = int(oracle_iou >= threshold)
            frame_rows.append(row)

    expected_gt_frames = sum(len(gt_map.get(video_id, {})) for video_id in video_ids)
    if len(frame_rows) != expected_gt_frames:
        raise RuntimeError(
            f'Detector profile GT coverage mismatch: {len(frame_rows)} vs {expected_gt_frames}'
        )

    final_by_video = prediction_frame_maps(production_result['predictions'])
    metric_parity = True
    for video_id in video_ids:
        actual = compute_st_iou_video(
            gt_map.get(video_id, {}), final_by_video.get(video_id, {})
        )
        expected = production_result['per_video'][video_id]
        if abs(actual - expected) > 1e-12:
            metric_parity = False
            raise RuntimeError(
                f'Detector profile metric mismatch for {video_id}: {actual} vs {expected}'
            )

    predictions_unchanged = (
        json.dumps(production_result['predictions'], sort_keys=True, separators=(',', ':'))
        == prediction_snapshot
    )
    if not predictions_unchanged:
        raise RuntimeError('Detector profile mutated production predictions')

    video_rows = detector_profile_video_rows(frame_rows, cache, video_metadata)
    global_profile = aggregate_detector_profile_rows(frame_rows)
    global_profile['projected_lt_16_frames'] = sum(
        row['projected_min_side_px'] < 16.0 for row in frame_rows
    )
    global_profile['projected_lt_16_rate'] = (
        global_profile['projected_lt_16_frames'] / max(1, len(frame_rows))
    )
    global_profile['projected_lt_32_frames'] = sum(
        row['projected_min_side_px'] < 32.0 for row in frame_rows
    )
    global_profile['projected_lt_32_rate'] = (
        global_profile['projected_lt_32_frames'] / max(1, len(frame_rows))
    )
    diagnostics = {
        'production_predictions_unchanged': predictions_unchanged,
        'production_per_video_metric_exact_parity': metric_parity,
        'gt_frame_coverage_exact': len(frame_rows) == expected_gt_frames,
        'detector_inference_call': {
            'explicit_imgsz_passed_to_yolo_predict': False,
            'reference_imgsz_for_size_projection_only': DETECTOR_PROFILE_REFERENCE_IMGSZ,
            'warning': (
                'The current production call omits imgsz. The 640 projection is diagnostic '
                'and must not be interpreted as runtime proof until the next A/B run locks imgsz.'
            ),
            'confidence_threshold': CONFIDENCE_THRESHOLD,
            'augment_tta': USE_TTA,
        },
        'detector_architecture': detector_architecture,
        'video_metadata': video_metadata,
        'global': global_profile,
        'by_size_bin': group_detector_profile_rows(frame_rows, 'size_bin'),
        'by_identity': group_detector_profile_rows(frame_rows, 'identity_id'),
    }
    return frame_rows, video_rows, diagnostics


# --- MOTION DISTRIBUTION DIAGNOSTICS V1: read-only GT/cache profiling ---

def linear_percentile(values, quantile):
    if not values:
        return None
    if not 0.0 <= quantile <= 1.0:
        raise ValueError('quantile must be within [0, 1]')
    ordered = sorted(float(value) for value in values)
    position = quantile * (len(ordered) - 1)
    lower = int(math.floor(position))
    upper = int(math.ceil(position))
    if lower == upper:
        return ordered[lower]
    alpha = position - lower
    return ordered[lower] * (1.0 - alpha) + ordered[upper] * alpha


def motion_pair_measurements(previous_bbox, current_bbox):
    center_speed, log_scale_speed = gap_motion_metrics(
        previous_bbox, current_bbox, 1
    )
    previous_center = bbox_center(previous_bbox)
    current_center = bbox_center(current_bbox)
    return {
        'center_speed': center_speed,
        'log_scale_speed': log_scale_speed,
        'center_displacement_px': math.hypot(
            current_center[0] - previous_center[0],
            current_center[1] - previous_center[1],
        ),
    }


def add_motion_stage_fields(row, stage, previous_item, current_item,
                            previous_gt, current_gt):
    prefix = f'{stage}__'
    field_defaults = {
        'pair_present': 0,
        'center_speed': None,
        'log_scale_speed': None,
        'center_displacement_px': None,
        'min_endpoint_iou': None,
        'min_fusion_score': None,
        'abs_center_speed_error_vs_gt': None,
    }
    for threshold in MOTION_PROFILE_GATE_THRESHOLDS:
        suffix = str(threshold).replace('.', '')
        field_defaults[f'exceeds_gate_{suffix}'] = None
        field_defaults[f'good_pair_exceeds_gate_{suffix}'] = None
    row.update({prefix + key: value for key, value in field_defaults.items()})
    if previous_item is None or current_item is None:
        return

    previous_bbox = (
        previous_item if stage == 'gt' else previous_item['bbox']
    )
    current_bbox = current_item if stage == 'gt' else current_item['bbox']
    measurements = motion_pair_measurements(previous_bbox, current_bbox)
    row[prefix + 'pair_present'] = 1
    for key, value in measurements.items():
        row[prefix + key] = value
    if stage == 'gt':
        min_endpoint_iou = 1.0
        min_fusion_score = None
        gt_center_speed = measurements['center_speed']
    else:
        min_endpoint_iou = min(
            iou_box(previous_bbox, previous_gt),
            iou_box(current_bbox, current_gt),
        )
        scores = [
            item.get('score') for item in (previous_item, current_item)
            if item.get('score') is not None
        ]
        min_fusion_score = min(scores) if len(scores) == 2 else None
        gt_center_speed = row.get('gt__center_speed')
    row[prefix + 'min_endpoint_iou'] = min_endpoint_iou
    row[prefix + 'min_fusion_score'] = min_fusion_score
    if gt_center_speed is not None:
        row[prefix + 'abs_center_speed_error_vs_gt'] = abs(
            measurements['center_speed'] - gt_center_speed
        )
    good_pair = min_endpoint_iou >= MOTION_PROFILE_GOOD_ENDPOINT_IOU
    for threshold in MOTION_PROFILE_GATE_THRESHOLDS:
        suffix = str(threshold).replace('.', '')
        exceeds = measurements['center_speed'] > threshold
        row[prefix + f'exceeds_gate_{suffix}'] = int(exceeds)
        row[prefix + f'good_pair_exceeds_gate_{suffix}'] = int(
            good_pair and exceeds
        )


def summarize_motion_stage(rows, stage):
    prefix = f'{stage}__'
    present = [row for row in rows if row[prefix + 'pair_present']]
    center_speeds = [row[prefix + 'center_speed'] for row in present]
    log_scale_speeds = [row[prefix + 'log_scale_speed'] for row in present]
    displacements = [row[prefix + 'center_displacement_px'] for row in present]
    endpoint_ious = [
        row[prefix + 'min_endpoint_iou'] for row in present
        if row[prefix + 'min_endpoint_iou'] is not None
    ]
    errors = [
        row[prefix + 'abs_center_speed_error_vs_gt'] for row in present
        if row[prefix + 'abs_center_speed_error_vs_gt'] is not None
    ]
    pair_count = len(present)
    output = {
        'all_gt_adjacent_pairs': len(rows),
        'pair_count': pair_count,
        'pair_coverage': pair_count / max(1, len(rows)),
        'good_endpoint_pair_count': sum(
            value >= MOTION_PROFILE_GOOD_ENDPOINT_IOU for value in endpoint_ious
        ),
        'good_endpoint_pair_rate': sum(
            value >= MOTION_PROFILE_GOOD_ENDPOINT_IOU for value in endpoint_ious
        ) / max(1, len(endpoint_ious)),
        'center_speed_mean': sum(center_speeds) / max(1, pair_count),
        'center_speed_max': max(center_speeds) if center_speeds else None,
        'log_scale_speed_mean': sum(log_scale_speeds) / max(1, pair_count),
        'log_scale_speed_max': max(log_scale_speeds) if log_scale_speeds else None,
        'center_displacement_px_mean': sum(displacements) / max(1, pair_count),
        'center_displacement_px_max': max(displacements) if displacements else None,
        'mean_abs_center_speed_error_vs_gt': (
            sum(errors) / len(errors) if errors else None
        ),
    }
    for quantile in MOTION_PROFILE_PERCENTILES:
        suffix = f'p{int(round(100 * quantile))}'
        output[f'center_speed_{suffix}'] = linear_percentile(
            center_speeds, quantile
        )
        output[f'log_scale_speed_{suffix}'] = linear_percentile(
            log_scale_speeds, quantile
        )
        output[f'center_displacement_px_{suffix}'] = linear_percentile(
            displacements, quantile
        )
    for threshold in MOTION_PROFILE_GATE_THRESHOLDS:
        suffix = str(threshold).replace('.', '')
        exceed_count = sum(
            row[prefix + f'exceeds_gate_{suffix}'] for row in present
        )
        good_rows = [
            row for row in present
            if row[prefix + 'min_endpoint_iou'] is not None
            and row[prefix + 'min_endpoint_iou']
                >= MOTION_PROFILE_GOOD_ENDPOINT_IOU
        ]
        good_exceed_count = sum(
            row[prefix + f'good_pair_exceeds_gate_{suffix}']
            for row in good_rows
        )
        output[f'exceeds_gate_{suffix}_count'] = exceed_count
        output[f'exceeds_gate_{suffix}_rate'] = exceed_count / max(1, pair_count)
        output[f'good_pair_exceeds_gate_{suffix}_count'] = good_exceed_count
        output[f'good_pair_exceeds_gate_{suffix}_rate'] = (
            good_exceed_count / max(1, len(good_rows))
        )
    return output


def motion_group_rows(frame_rows, group_type, group_key=None):
    if group_key is None:
        groups = {'all': frame_rows}
    else:
        groups = {}
        for row in frame_rows:
            groups.setdefault(str(row[group_key]), []).append(row)
    output = []
    for group_name, rows in sorted(groups.items()):
        for stage in MOTION_PROFILE_STAGES:
            output.append({
                'group_type': group_type,
                'group_name': group_name,
                'stage': stage,
                **summarize_motion_stage(rows, stage),
            })
    return output


def build_motion_profile_frame_rows(video_id, video_cache, gt_frames,
                                    final_frames, metadata, weights):
    ranked, _ = rank_video_frames(video_cache, weights, include_debug=False)
    oracle = {}
    has_siamese = bool(video_cache['has_siamese'])
    has_color = bool(video_cache['has_color'])
    for frame_idx, gt_bbox in gt_frames.items():
        candidates = video_cache['frames'].get(str(frame_idx), [])
        oracle_idx, oracle_iou = oracle_candidate_for_gt(candidates, gt_bbox)
        if oracle_idx is None:
            continue
        candidate = candidates[oracle_idx]
        oracle[frame_idx] = {
            'bbox': candidate['bbox'],
            'score': fuse_candidate_score(
                candidate['scores'], weights, has_siamese, has_color
            ),
            'oracle_iou': oracle_iou,
            'candidate_idx': oracle_idx,
        }
    final_items = {
        frame_idx: {'bbox': bbox, 'score': None}
        for frame_idx, bbox in final_frames.items()
    }
    scale_to_reference = min(
        MOTION_PROFILE_REFERENCE_IMGSZ / metadata['width'],
        MOTION_PROFILE_REFERENCE_IMGSZ / metadata['height'],
    )
    rows = []
    ordered_frames = sorted(gt_frames)
    for previous_frame, current_frame in zip(ordered_frames, ordered_frames[1:]):
        if current_frame != previous_frame + 1:
            continue
        previous_gt = gt_frames[previous_frame]
        current_gt = gt_frames[current_frame]
        previous_min_side = min(
            previous_gt[2] - previous_gt[0], previous_gt[3] - previous_gt[1]
        )
        current_min_side = min(
            current_gt[2] - current_gt[0], current_gt[3] - current_gt[1]
        )
        mean_min_side = 0.5 * (previous_min_side + current_min_side)
        projected_min_side = mean_min_side * scale_to_reference
        row = {
            'video_id': video_id,
            'identity_id': identity_id_from_video_id(video_id),
            'previous_frame': previous_frame,
            'current_frame': current_frame,
            'frame_step': 1,
            'mean_gt_min_side_px': mean_min_side,
            'mean_gt_projected_min_side_px': projected_min_side,
            'size_bin': detector_profile_size_bin(projected_min_side),
        }
        add_motion_stage_fields(
            row, 'gt', previous_gt, current_gt, previous_gt, current_gt
        )
        add_motion_stage_fields(
            row, 'selected_top1', ranked.get(previous_frame),
            ranked.get(current_frame), previous_gt, current_gt
        )
        add_motion_stage_fields(
            row, 'oracle_candidate', oracle.get(previous_frame),
            oracle.get(current_frame), previous_gt, current_gt
        )
        add_motion_stage_fields(
            row, 'production_final', final_items.get(previous_frame),
            final_items.get(current_frame), previous_gt, current_gt
        )
        rows.append(row)
    return rows


def run_motion_distribution_diagnostics(cache, gt_map, weights,
                                        production_result, video_metadata=None):
    prediction_snapshot = json.dumps(
        production_result['predictions'], sort_keys=True, separators=(',', ':')
    )
    video_ids = list(cache['signature']['video_folders'])
    if video_metadata is None:
        video_metadata = collect_video_metadata(video_ids)
    final_by_video = prediction_frame_maps(production_result['predictions'])
    frame_rows = []
    for video_id in video_ids:
        actual_score = compute_st_iou_video(
            gt_map.get(video_id, {}), final_by_video.get(video_id, {})
        )
        expected_score = production_result['per_video'][video_id]
        if abs(actual_score - expected_score) > 1e-12:
            raise RuntimeError(
                f'Motion profile production metric mismatch for {video_id}'
            )
        frame_rows.extend(build_motion_profile_frame_rows(
            video_id, cache['videos'][video_id], gt_map.get(video_id, {}),
            final_by_video.get(video_id, {}), video_metadata[video_id], weights,
        ))
    if json.dumps(
        production_result['predictions'], sort_keys=True, separators=(',', ':')
    ) != prediction_snapshot:
        raise RuntimeError('Motion profile mutated production predictions')

    expected_pairs = sum(
        sum(current == previous + 1 for previous, current in zip(
            sorted(gt_map.get(video_id, {})),
            sorted(gt_map.get(video_id, {}))[1:],
        ))
        for video_id in video_ids
    )
    if len(frame_rows) != expected_pairs:
        raise RuntimeError(
            f'Motion profile adjacent-pair mismatch: {len(frame_rows)} vs {expected_pairs}'
        )
    group_rows = []
    group_rows.extend(motion_group_rows(frame_rows, 'global'))
    group_rows.extend(motion_group_rows(frame_rows, 'size_bin', 'size_bin'))
    group_rows.extend(motion_group_rows(frame_rows, 'identity', 'identity_id'))
    video_rows = motion_group_rows(frame_rows, 'video', 'video_id')
    group_rows.extend(video_rows)

    def nested_group(group_type):
        return {
            row['group_name']: {
                stage_row['stage']: {
                    key: value for key, value in stage_row.items()
                    if key not in ('group_type', 'group_name', 'stage')
                }
                for stage_row in group_rows
                if stage_row['group_type'] == group_type
                and stage_row['group_name'] == row['group_name']
            }
            for row in group_rows if row['group_type'] == group_type
        }

    diagnostics = {
        'production_predictions_unchanged': True,
        'production_per_video_metric_exact_parity': True,
        'adjacent_gt_pair_coverage_exact': len(frame_rows) == expected_pairs,
        'adjacent_gt_pair_count': expected_pairs,
        'motion_definition': {
            'center_speed': (
                'center displacement / mean sqrt bbox area for consecutive frames'
            ),
            'log_scale_speed': 'abs(log(scale2 / scale1)) for consecutive frames',
            'size_bin': (
                f'mean GT min side projected to {MOTION_PROFILE_REFERENCE_IMGSZ}px '
                'with aspect-ratio-preserving scale'
            ),
            'good_candidate_pair': (
                f'both endpoint bboxes have IoU >= {MOTION_PROFILE_GOOD_ENDPOINT_IOU}'
            ),
        },
        'gate_thresholds': MOTION_PROFILE_GATE_THRESHOLDS,
        'global': nested_group('global')['all'],
        'by_size_bin': nested_group('size_bin'),
        'by_identity': nested_group('identity'),
        'by_video': nested_group('video'),
        'video_metadata': video_metadata,
    }
    return frame_rows, group_rows, video_rows, diagnostics


# --- SPATIAL LOCALIZATION HEADROOM V1: diagnostic counterfactuals only ---

def spatial_bbox_geometry(box):
    values = [float(value) for value in box]
    if len(values) != 4 or not all(math.isfinite(value) for value in values):
        raise ValueError(f'Invalid diagnostic bbox: {box}')
    x1, y1, x2, y2 = values
    if x2 <= x1 or y2 <= y1:
        raise ValueError(f'Degenerate diagnostic bbox: {box}')
    return (x1 + x2) / 2, (y1 + y2) / 2, x2 - x1, y2 - y1


def spatial_box_from_geometry(cx, cy, width, height):
    return [cx - width / 2, cy - height / 2,
            cx + width / 2, cy + height / 2]


def build_spatial_headroom_rows(video_id, video_cache, gt_frames, final_frames,
                                metadata, video_count):
    union = sorted(set(gt_frames) | set(final_frames))
    denominator = len(union)
    width, height = float(metadata['width']), float(metadata['height'])
    if width <= 0 or height <= 0:
        raise ValueError(f'Invalid spatial metadata for {video_id}')
    projection = SPATIAL_HEADROOM_REFERENCE_IMGSZ / max(width, height)
    rows = []
    previous = None
    for frame_idx in union:
        gt = gt_frames.get(frame_idx)
        pred = final_frames.get(frame_idx)
        shared = gt is not None and pred is not None
        iou = iou_box(gt, pred) if shared else 0.0
        candidates = video_cache['frames'].get(str(frame_idx), [])
        cohort = ('shared_good_overlap' if iou >= SPATIAL_HEADROOM_GOOD_IOU
                  else 'shared_low_overlap') if shared else (
                      'missed_gt' if gt is not None else 'background_prediction')
        row = {
            'video_id': video_id, 'identity_id': identity_id_from_video_id(video_id),
            'frame': int(frame_idx), 'has_gt': int(gt is not None),
            'has_prediction': int(pred is not None), 'cohort': cohort,
            'union_frame_count': denominator, 'size_bin': 'no_gt',
            'gt_projected_min_side_px': None, 'production_iou': iou,
            'candidate_count': len(candidates), 'best_candidate_iou': None,
            'best_candidate_index': None, 'local_candidate_iou': None,
            'center_dx_px': None, 'center_dy_px': None,
            'center_error_gt_scale': None, 'log_width_ratio': None,
            'log_height_ratio': None, 'center_only_iou': None, 'size_only_iou': None,
            'adjacent_good_overlap_pair': 0, 'center_residual_jitter': None,
            'log_size_residual_jitter': None, 'gt_center_speed': None,
        }
        for prefix, box in [('gt', gt), ('prediction', pred)]:
            for key, value in zip(('x1', 'y1', 'x2', 'y2'), box or [None] * 4):
                row[f'{prefix}_{key}'] = value
        if gt is not None:
            gx, gy, gw, gh = spatial_bbox_geometry(gt)
            row['gt_projected_min_side_px'] = min(gw, gh) * projection
            row['size_bin'] = detector_profile_size_bin(row['gt_projected_min_side_px'])
        if pred is not None:
            px, py, pw, ph = spatial_bbox_geometry(pred)
        # Every scenario keeps the original prediction frame set. Missed GT and
        # background predictions contribute zero numerator but retain their denominator.
        scenario_ious = {
            'perfect_shared': 1.0 if shared else 0.0,
            'perfect_good_overlap': iou,
            'candidate_keep_best': iou,
            'local_candidate_keep_best': iou,
            'center_only_good_overlap': iou,
            'size_only_good_overlap': iou,
        }
        if shared:
            dx, dy = px - gx, py - gy
            row.update({
                'center_dx_px': dx, 'center_dy_px': dy,
                'center_error_gt_scale': math.hypot(dx, dy) / math.sqrt(gw * gh),
                'log_width_ratio': math.log(pw / gw),
                'log_height_ratio': math.log(ph / gh),
                'center_only_iou': iou_box(spatial_box_from_geometry(gx, gy, pw, ph), gt),
                'size_only_iou': iou_box(spatial_box_from_geometry(px, py, gw, gh), gt),
            })
            best_iou, local_iou, best_idx = 0.0, 0.0, None
            for idx, candidate in enumerate(candidates):
                # Match production output's coordinate rounding. Never change cache.
                box = [int(round(value)) for value in candidate['bbox']]
                candidate_iou = iou_box(box, gt)
                if candidate_iou > best_iou:
                    best_iou, best_idx = candidate_iou, idx
                if iou_box(box, pred) >= SPATIAL_HEADROOM_LOCAL_OVERLAP:
                    local_iou = max(local_iou, candidate_iou)
            row.update({'best_candidate_iou': best_iou,
                        'best_candidate_index': best_idx, 'local_candidate_iou': local_iou})
            scenario_ious['candidate_keep_best'] = max(iou, best_iou)
            scenario_ious['local_candidate_keep_best'] = max(iou, local_iou)
            if cohort == 'shared_good_overlap':
                scenario_ious.update({
                    'perfect_good_overlap': 1.0,
                    'center_only_good_overlap': row['center_only_iou'],
                    'size_only_good_overlap': row['size_only_iou'],
                })
                if (previous is not None and previous['frame'] == frame_idx - 1
                        and previous['cohort'] == 'shared_good_overlap'):
                    prev_gt = [previous[f'gt_{key}'] for key in ('x1', 'y1', 'x2', 'y2')]
                    pgx, pgy, pgw, pgh = spatial_bbox_geometry(prev_gt)
                    scale = (math.sqrt(gw * gh) + math.sqrt(pgw * pgh)) / 2
                    row.update({
                        'adjacent_good_overlap_pair': 1,
                        'center_residual_jitter': math.hypot(
                            dx - previous['center_dx_px'], dy - previous['center_dy_px']) / scale,
                        'log_size_residual_jitter': math.hypot(
                            row['log_width_ratio'] - previous['log_width_ratio'],
                            row['log_height_ratio'] - previous['log_height_ratio']),
                        'gt_center_speed': math.hypot(gx - pgx, gy - pgy) / scale,
                    })
        for name, value in scenario_ious.items():
            row[f'{name}_iou'] = value
            row[f'{name}_macro_delta_contribution'] = (value - iou) / denominator / video_count
        rows.append(row)
        previous = row
    return rows


def summarize_spatial_rows(rows):
    shared = [row for row in rows if row['has_gt'] and row['has_prediction']]
    good = [row for row in shared if row['cohort'] == 'shared_good_overlap']
    result = {
        'row_count': len(rows),
        'gt_frame_count': sum(row['has_gt'] for row in rows),
        'prediction_frame_count': sum(row['has_prediction'] for row in rows),
        'shared_frame_count': len(shared), 'good_overlap_frame_count': len(good),
        'low_overlap_frame_count': len(shared) - len(good),
        'missed_gt_count': sum(row['cohort'] == 'missed_gt' for row in rows),
        'background_prediction_count': sum(row['cohort'] == 'background_prediction' for row in rows),
        'mean_iou_on_shared': sum(row['production_iou'] for row in shared) / len(shared) if shared else None,
        'mean_iou_on_good_overlap': sum(row['production_iou'] for row in good) / len(good) if good else None,
        'adjacent_good_overlap_pair_count': sum(row['adjacent_good_overlap_pair'] for row in rows),
    }
    for field in ('center_error_gt_scale', 'log_width_ratio', 'log_height_ratio',
                  'center_residual_jitter', 'log_size_residual_jitter', 'gt_center_speed'):
        values = [row[field] for row in good if row[field] is not None]
        for suffix, q in [('p50', 0.5), ('p90', 0.9), ('p95', 0.95)]:
            result[f'good_overlap_{field}_{suffix}'] = linear_percentile(values, q)
    for name in SPATIAL_HEADROOM_SCENARIOS:
        result[f'{name}_macro_delta_contribution'] = sum(
            row[f'{name}_macro_delta_contribution'] for row in rows)
    return result


def run_spatial_headroom_diagnostics(cache, gt_map, production_result, video_metadata=None):
    snapshot = json.dumps(production_result['predictions'], sort_keys=True, separators=(',', ':'))
    video_ids = sorted(cache['signature']['video_folders'])
    if not video_ids or len(video_ids) != len(set(video_ids)):
        raise ValueError('Spatial diagnostic requires unique, non-empty video ids')
    final_by_video = prediction_frame_maps(production_result['predictions'])
    if set(final_by_video) != set(video_ids) or set(production_result['per_video']) != set(video_ids):
        raise ValueError('Spatial diagnostic production/cache video sets differ')
    if any(video_id not in gt_map for video_id in video_ids):
        raise ValueError('Spatial diagnostic requires GT for every video')
    if video_metadata is None:
        video_metadata = collect_video_metadata(video_ids)
    frame_rows, video_rows = [], []
    for video_id in video_ids:
        gt, pred = gt_map[video_id], final_by_video[video_id]
        actual = compute_st_iou_video(gt, pred)
        if not math.isfinite(actual) or abs(actual - production_result['per_video'][video_id]) > 1e-12:
            raise RuntimeError(f'Spatial diagnostic production metric mismatch: {video_id}')
        rows = build_spatial_headroom_rows(
            video_id, cache['videos'][video_id], gt, pred, video_metadata[video_id], len(video_ids))
        denominator = len(set(gt) | set(pred))
        row = {'video_id': video_id, 'identity_id': identity_id_from_video_id(video_id),
               'production_st_iou': actual, 'union_frame_count': denominator,
               **summarize_spatial_rows(rows)}
        if abs(sum(item['production_iou'] for item in rows) / max(1, denominator) - actual) > 1e-12:
            raise RuntimeError('Spatial diagnostic numerator reconciliation failed')
        for name in SPATIAL_HEADROOM_SCENARIOS:
            score = sum(item[f'{name}_iou'] for item in rows) / max(1, denominator)
            row[f'{name}_st_iou'] = score
            row[f'{name}_delta'] = score - actual
        if not (actual <= row['perfect_good_overlap_st_iou'] + 1e-12
                <= row['perfect_shared_st_iou'] + 2e-12):
            raise RuntimeError('Spatial diagnostic upper bounds are inconsistent')
        frame_rows.extend(rows)
        video_rows.append(row)
    production_mean = sum(row['production_st_iou'] for row in video_rows) / len(video_rows)
    if abs(production_mean - production_result['mean_st_iou']) > 1e-12:
        raise RuntimeError('Spatial diagnostic macro production metric mismatch')
    expected_frames = sum(len(final_by_video[video_id]) for video_id in video_ids)
    if expected_frames != production_result['detected_frames']:
        raise RuntimeError('Spatial diagnostic production frame count mismatch')
    group_rows = [{'group_type': 'global', 'group_name': 'all', **summarize_spatial_rows(frame_rows)}]
    for field in ('identity_id', 'size_bin', 'cohort'):
        for value in sorted({row[field] for row in frame_rows}):
            group_rows.append({'group_type': field, 'group_name': value,
                               **summarize_spatial_rows([row for row in frame_rows if row[field] == value])})
    scenarios = {}
    for name in SPATIAL_HEADROOM_SCENARIOS:
        mean_score = sum(row[f'{name}_st_iou'] for row in video_rows) / len(video_rows)
        identity_deltas = {}
        for identity in sorted({row['identity_id'] for row in video_rows}):
            values = [row[f'{name}_delta'] for row in video_rows if row['identity_id'] == identity]
            identity_deltas[identity] = sum(values) / len(values)
        scenarios[name] = {
            'mean_st_iou': mean_score, 'macro_delta': mean_score - production_mean,
            'per_identity_delta_descriptive_only': identity_deltas,
        }
        if abs(scenarios[name]['macro_delta'] - group_rows[0][f'{name}_macro_delta_contribution']) > 1e-12:
            raise RuntimeError('Spatial diagnostic macro contribution reconciliation failed')
    if json.dumps(production_result['predictions'], sort_keys=True, separators=(',', ':')) != snapshot:
        raise RuntimeError('Spatial diagnostic mutated production predictions')
    refiner_ceiling = scenarios['perfect_good_overlap']
    qualifying_identities = [
        identity for identity, delta in
        refiner_ceiling['per_identity_delta_descriptive_only'].items()
        if delta >= SPATIAL_HEADROOM_MIN_IDENTITY_CEILING_DELTA
    ]
    refiner_ab_decision = {
        'basis': (
            'perfect_good_overlap is a conservative spatial ceiling: only '
            'production/GT shared frames with IoU >= 0.5 may become perfect; '
            'the temporal support and every other bbox remain fixed'
        ),
        'minimum_macro_ceiling_delta': SPATIAL_HEADROOM_MIN_REFINER_CEILING_DELTA,
        'minimum_identity_ceiling_delta': SPATIAL_HEADROOM_MIN_IDENTITY_CEILING_DELTA,
        'minimum_qualifying_identities': SPATIAL_HEADROOM_MIN_QUALIFYING_IDENTITIES,
        'observed_macro_ceiling_delta': refiner_ceiling['macro_delta'],
        'qualifying_identities': qualifying_identities,
        'qualifying_identity_count': len(qualifying_identities),
        'run_alpha_refine_ab': (
            refiner_ceiling['macro_delta']
                >= SPATIAL_HEADROOM_MIN_REFINER_CEILING_DELTA
            and len(qualifying_identities)
                >= SPATIAL_HEADROOM_MIN_QUALIFYING_IDENTITIES
        ),
        'scope_if_passed': (
            'Train on train-set GT with bbox jitter; evaluate one global refiner '
            'while keeping production frame support fixed'
        ),
    }
    diagnostics = {
        'production_predictions_unchanged': True,
        'production_metric_parity': True, 'fixed_prediction_frame_count': expected_frames,
        'union_frame_rows': len(frame_rows), 'macro_aggregation': 'equal weight per video',
        'global': group_rows[0], 'scenarios': scenarios,
        'refiner_ab_decision': refiner_ab_decision,
        'video_metadata': video_metadata,
        'interpretation': {
            'perfect_shared': 'Exact temporal IoU upper bound with fixed prediction support and arbitrary perfect boxes on GT intersection; includes wrong-object correction.',
            'perfect_good_overlap': 'Perfect boxes only where production IoU >= 0.5; overlap is a proxy, not an identity label. GT eligibility is unavailable at inference.',
            'candidate_keep_best': 'GT chooses max(production IoU, rounded cached-candidate IoU) only on shared frames; may switch object. Not a refiner upper bound.',
            'local_candidate_keep_best': 'Same oracle but candidate must overlap production bbox by IoU >= 0.5; proximity does not guarantee same identity.',
            'center_only_good_overlap': 'Use GT center with production width/height on good-overlap frames. Continuous diagnostic coordinates; not a deployable prediction.',
            'size_only_good_overlap': 'Use GT width/height with production center on good-overlap frames. Signed gain can be negative; center/size gains are not additive.',
            'jitter': 'Change in prediction-minus-GT error only across consecutive good-overlap frames. GT annotation noise and cuts can affect it; no smoothing or camera-cut inference is performed.',
            'size_projection': 'GT minimum side projected to 640 with aspect ratio preserved; not original pixel size.',
            'selection': 'Diagnostic only. No hyperparameter selection, model promotion, or held-out generalization claim.',
        },
    }
    return frame_rows, group_rows, video_rows, diagnostics



# --- ANCHOR-PRESERVING RESIDUAL RECOVERY V2: cache replay only ---

def residual_recovery_config_grid():
    return [
        {
            'top_k': RESIDUAL_RECOVERY_TOP_K,
            'weak_threshold': weak_threshold,
            'max_anchor_gap': max_anchor_gap,
            'max_center_speed': max_center_speed,
            'max_candidate_gap': RESIDUAL_RECOVERY_MAX_CANDIDATE_GAP,
            'min_recovered_frames': RESIDUAL_RECOVERY_MIN_RECOVERED_FRAMES,
            'min_gap_coverage': RESIDUAL_RECOVERY_MIN_GAP_COVERAGE,
        }
        for weak_threshold in RESIDUAL_RECOVERY_WEAK_THRESHOLD_GRID
        for max_anchor_gap in RESIDUAL_RECOVERY_MAX_ANCHOR_GAP_GRID
        for max_center_speed in RESIDUAL_RECOVERY_MAX_CENTER_SPEED_GRID
    ]


def residual_recovery_config_key(config):
    return (
        int(config['top_k']), float(config['weak_threshold']),
        int(config['max_anchor_gap']), float(config['max_center_speed']),
        int(config['max_candidate_gap']), int(config['min_recovered_frames']),
        float(config['min_gap_coverage']),
    )


def recover_candidate_chain_between_anchors(candidates_by_frame, left_frame,
                                            left_anchor, right_frame,
                                            right_anchor, config):
    """Find a dense motion-consistent DAG path; never modifies either anchor."""
    gap_size = right_frame - left_frame - 1
    empty = {
        'gap_size': gap_size, 'eligible_candidate_count': 0,
        'path_length': 0, 'coverage': 0.0,
    }
    if gap_size <= PRODUCTION_TEMPORAL_CONFIG['max_gap']:
        return {}, {**empty, 'reason': 'already_handled_by_production_gap_fill'}
    if gap_size > int(config['max_anchor_gap']):
        return {}, {**empty, 'reason': 'anchor_gap_too_long'}
    steps = right_frame - left_frame
    if not gap_is_plausible(
        left_anchor['bbox'], right_anchor['bbox'], steps,
        max_center_speed=config['max_center_speed'],
        max_log_scale_speed=None,
    ):
        return {}, {**empty, 'reason': 'scene_discontinuity_reset'}

    weak_threshold = float(config['weak_threshold'])
    max_step = int(config['max_candidate_gap']) + 1
    nodes = []
    eligible_count = 0
    for frame_idx in range(left_frame + 1, right_frame):
        for candidate in candidates_by_frame.get(frame_idx, []):
            if candidate['score'] < weak_threshold:
                continue
            eligible_count += 1
            best = None
            if frame_idx - left_frame <= max_step and gap_is_plausible(
                left_anchor['bbox'], candidate['bbox'], frame_idx - left_frame,
                max_center_speed=config['max_center_speed'],
                max_log_scale_speed=None,
            ):
                best = {
                    'length': 1,
                    'score_sum': candidate['score'] - weak_threshold,
                    'parent': None,
                }
            for previous_idx, previous in enumerate(nodes):
                frame_step = frame_idx - previous['frame']
                if not 0 < frame_step <= max_step:
                    continue
                if not gap_is_plausible(
                    previous['candidate']['bbox'], candidate['bbox'], frame_step,
                    max_center_speed=config['max_center_speed'],
                    max_log_scale_speed=None,
                ):
                    continue
                proposal = {
                    'length': previous['length'] + 1,
                    'score_sum': (
                        previous['score_sum'] + candidate['score'] - weak_threshold
                    ),
                    'parent': previous_idx,
                }
                if best is None or (
                    proposal['length'], proposal['score_sum']
                ) > (best['length'], best['score_sum']):
                    best = proposal
            if best is not None:
                nodes.append({
                    'frame': frame_idx, 'candidate': candidate, **best,
                })

    terminal_indices = []
    for node_idx, node in enumerate(nodes):
        frame_step = right_frame - node['frame']
        if 0 < frame_step <= max_step and gap_is_plausible(
            node['candidate']['bbox'], right_anchor['bbox'], frame_step,
            max_center_speed=config['max_center_speed'],
            max_log_scale_speed=None,
        ):
            terminal_indices.append(node_idx)
    if not terminal_indices:
        return {}, {
            **empty, 'eligible_candidate_count': eligible_count,
            'reason': 'no_anchor_to_anchor_path',
        }
    best_idx = max(
        terminal_indices,
        key=lambda idx: (nodes[idx]['length'], nodes[idx]['score_sum'], -idx),
    )
    path = []
    while best_idx is not None:
        path.append(nodes[best_idx])
        best_idx = nodes[best_idx]['parent']
    path.reverse()
    coverage = len(path) / max(1, gap_size)
    if (
        len(path) < int(config['min_recovered_frames'])
        or coverage < float(config['min_gap_coverage'])
    ):
        return {}, {
            **empty, 'eligible_candidate_count': eligible_count,
            'path_length': len(path), 'coverage': coverage,
            'reason': 'insufficient_path_coverage',
        }
    return {
        node['frame']: node['candidate'] for node in path
    }, {
        **empty, 'eligible_candidate_count': eligible_count,
        'path_length': len(path), 'coverage': coverage,
        'reason': 'recovered',
    }


def build_residual_recovery_for_video(video_cache, final_frames, weights,
                                      matching_threshold, config):
    diagnostics = {
        'strong_anchor_count': 0, 'candidate_gap_count': 0,
        'recovered_gap_count': 0, 'recovered_frame_count': 0,
        'scene_discontinuity_reset_count': 0,
        'reason_counts': {},
    }
    # weak==strong is a guaranteed no-op control independent of cache contents.
    if float(config['weak_threshold']) >= float(matching_threshold):
        diagnostics['reason_counts']['exact_noop_control'] = 1
        return {}, diagnostics
    candidates_by_frame = rank_top_k_video_frames(
        video_cache, weights, top_k=config['top_k']
    )
    threshold = (
        matching_threshold if video_cache['has_siamese'] else YOLO_ONLY_THRESHOLD
    )
    anchors = {
        frame_idx: candidates[0]
        for frame_idx, candidates in candidates_by_frame.items()
        if candidates and candidates[0]['score'] >= threshold
        and frame_idx in final_frames
    }
    anchor_frames = sorted(anchors)
    diagnostics['strong_anchor_count'] = len(anchor_frames)
    recovered = {}
    for left_frame, right_frame in zip(anchor_frames, anchor_frames[1:]):
        gap_size = right_frame - left_frame - 1
        if gap_size <= PRODUCTION_TEMPORAL_CONFIG['max_gap']:
            continue
        diagnostics['candidate_gap_count'] += 1
        gap_recovery, gap_diagnostics = recover_candidate_chain_between_anchors(
            candidates_by_frame, left_frame, anchors[left_frame],
            right_frame, anchors[right_frame], config,
        )
        reason = gap_diagnostics['reason']
        diagnostics['reason_counts'][reason] = (
            diagnostics['reason_counts'].get(reason, 0) + 1
        )
        if reason == 'scene_discontinuity_reset':
            diagnostics['scene_discontinuity_reset_count'] += 1
        if gap_recovery:
            diagnostics['recovered_gap_count'] += 1
            for frame_idx, candidate in gap_recovery.items():
                if frame_idx not in final_frames:
                    recovered[frame_idx] = candidate
    diagnostics['recovered_frame_count'] = len(recovered)
    return recovered, diagnostics


def render_anchor_preserving_residual(production_result, recovered_by_video,
                                      gt_map):
    predictions = json.loads(json.dumps(production_result['predictions']))
    prediction_by_video = {
        item['video_id']: item for item in predictions
    }
    per_video, detected_frames = {}, 0
    for video_id in sorted(production_result['per_video']):
        prediction = prediction_by_video[video_id]
        recovered = recovered_by_video.get(video_id, {})
        if recovered:
            if not prediction.get('detections'):
                prediction['detections'] = [{'bboxes': []}]
            bboxes = prediction['detections'][0].setdefault('bboxes', [])
            existing_frames = {int(item['frame']) for item in bboxes}
            for frame_idx, candidate in recovered.items():
                if frame_idx in existing_frames:
                    continue
                x1, y1, x2, y2 = candidate['bbox']
                bboxes.append({
                    'frame': int(frame_idx),
                    'x1': int(round(x1)), 'y1': int(round(y1)),
                    'x2': int(round(x2)), 'y2': int(round(y2)),
                })
            bboxes.sort(key=lambda item: int(item['frame']))
        frame_map = prediction_frame_maps([prediction])[video_id]
        per_video[video_id] = compute_st_iou_video(
            gt_map.get(video_id, {}), frame_map
        )
        detected_frames += len(frame_map)
    return {
        'mean_st_iou': sum(per_video.values()) / max(1, len(per_video)),
        'per_video': per_video,
        'detected_frames': detected_frames,
        'predictions': predictions,
    }


def evaluate_residual_recovery_config(cache, gt_map, weights,
                                      matching_threshold, production_result,
                                      config):
    start_time = time.perf_counter()
    final_by_video = prediction_frame_maps(production_result['predictions'])
    recovered_by_video, per_video_diagnostics = {}, {}
    for video_id in cache['signature']['video_folders']:
        recovered_by_video[video_id], per_video_diagnostics[video_id] = (
            build_residual_recovery_for_video(
                cache['videos'][video_id], final_by_video.get(video_id, {}),
                weights, matching_threshold, config,
            )
        )
    result = render_anchor_preserving_residual(
        production_result, recovered_by_video, gt_map
    )
    elapsed = max(1e-12, time.perf_counter() - start_time)
    total_frames = sum(
        max(0, int(video['max_frame_idx']) + 1)
        for video in cache['videos'].values()
    )
    result['residual_diagnostics'] = {
        'elapsed_seconds': elapsed,
        'replay_fps': total_frames / elapsed,
        'recovered_frame_count': sum(
            item['recovered_frame_count']
            for item in per_video_diagnostics.values()
        ),
        'recovered_gap_count': sum(
            item['recovered_gap_count']
            for item in per_video_diagnostics.values()
        ),
        'scene_discontinuity_reset_count': sum(
            item['scene_discontinuity_reset_count']
            for item in per_video_diagnostics.values()
        ),
        'per_video': per_video_diagnostics,
    }
    return result


def residual_recovery_result_row(config, result):
    diagnostics = result['residual_diagnostics']
    row = {
        **config,
        'mean_st_iou': result['mean_st_iou'],
        'detected_frames': result['detected_frames'],
        'replay_fps': diagnostics['replay_fps'],
        'recovered_frame_count': diagnostics['recovered_frame_count'],
        'recovered_gap_count': diagnostics['recovered_gap_count'],
        'scene_discontinuity_reset_count': (
            diagnostics['scene_discontinuity_reset_count']
        ),
    }
    for video_id, score in sorted(result['per_video'].items()):
        row[f'st_iou__{video_id}'] = score
    return row


def residual_recovery_config_from_row(row):
    return {
        key: row[key] for key in (
            'top_k', 'weak_threshold', 'max_anchor_gap', 'max_center_speed',
            'max_candidate_gap', 'min_recovered_frames', 'min_gap_coverage',
        )
    }


def select_residual_recovery_row(rows, selection_video_ids):
    def key(row):
        # Exact ties prefer the no-op/higher threshold and smaller search extent.
        return (
            mean_row_score(row, selection_video_ids),
            float(row['weak_threshold']), -int(row['max_anchor_gap']),
            -float(row['max_center_speed']),
        )
    return max(rows, key=key)


def select_voted_residual_recovery_config(fold_rows):
    votes = {}
    for row in fold_rows:
        key = residual_recovery_config_key(
            residual_recovery_config_from_row(row)
        )
        votes[key] = votes.get(key, 0) + 1
    best_key = max(
        votes,
        key=lambda key: (votes[key], key[1], -key[2], -key[3]),
    )
    return {
        'top_k': best_key[0], 'weak_threshold': best_key[1],
        'max_anchor_gap': best_key[2], 'max_center_speed': best_key[3],
        'max_candidate_gap': best_key[4],
        'min_recovered_frames': best_key[5],
        'min_gap_coverage': best_key[6],
    }, votes[best_key]


def run_residual_recovery_identity_cv(cache, gt_map, weights,
                                      matching_threshold, production_result):
    prediction_snapshot = json.dumps(
        production_result['predictions'], sort_keys=True, separators=(',', ':')
    )
    rows, results_by_config = [], {}
    for config in tqdm(
        residual_recovery_config_grid(), desc='Anchor-preserving residual grid'
    ):
        result = evaluate_residual_recovery_config(
            cache, gt_map, weights, matching_threshold, production_result, config
        )
        rows.append(residual_recovery_result_row(config, result))
        results_by_config[residual_recovery_config_key(config)] = result
    control_config = next(
        config for config in residual_recovery_config_grid()
        if config['weak_threshold'] >= matching_threshold
    )
    control = results_by_config[residual_recovery_config_key(control_config)]
    if control['predictions'] != production_result['predictions']:
        raise RuntimeError('Residual recovery no-op control is not exact production')
    if json.dumps(
        production_result['predictions'], sort_keys=True, separators=(',', ':')
    ) != prediction_snapshot:
        raise RuntimeError('Residual recovery mutated production predictions')

    identity_folds = build_identity_folds(cache['signature']['video_folders'])
    all_video_ids = sorted(cache['signature']['video_folders'])
    fold_rows = []
    for held_identity, held_video_ids in identity_folds.items():
        selection_video_ids = [
            video_id for video_id in all_video_ids if video_id not in held_video_ids
        ]
        selected = select_residual_recovery_row(rows, selection_video_ids)
        heldout_score = mean_row_score(selected, held_video_ids)
        baseline_score = sum(
            production_result['per_video'][video_id]
            for video_id in held_video_ids
        ) / len(held_video_ids)
        fold_rows.append({
            'heldout_identity': held_identity,
            'heldout_videos': '|'.join(held_video_ids),
            'selection_videos': '|'.join(selection_video_ids),
            **residual_recovery_config_from_row(selected),
            'selection_mean_st_iou': mean_row_score(selected, selection_video_ids),
            'heldout_mean_st_iou': heldout_score,
            'baseline_heldout_mean_st_iou': baseline_score,
            'heldout_delta': heldout_score - baseline_score,
        })
    recommended_config, recommendation_votes = (
        select_voted_residual_recovery_config(fold_rows)
    )
    recommended_result = results_by_config[
        residual_recovery_config_key(recommended_config)
    ]
    mean_cv_score = sum(
        row['heldout_mean_st_iou'] for row in fold_rows
    ) / len(fold_rows)
    mean_cv_baseline = sum(
        row['baseline_heldout_mean_st_iou'] for row in fold_rows
    ) / len(fold_rows)
    deltas = [row['heldout_delta'] for row in fold_rows]
    full_delta = (
        recommended_result['mean_st_iou'] - production_result['mean_st_iou']
    )
    replay_fps = recommended_result['residual_diagnostics']['replay_fps']
    is_enabled = recommended_config['weak_threshold'] < matching_threshold
    promotion_passed = (
        is_enabled
        and mean_cv_score - mean_cv_baseline >= RESIDUAL_RECOVERY_MIN_MEAN_CV_DELTA
        and sum(delta > 0 for delta in deltas)
            >= RESIDUAL_RECOVERY_MIN_IMPROVED_IDENTITIES
        and min(deltas) >= -RESIDUAL_RECOVERY_MAX_WORST_IDENTITY_DROP
        and recommendation_votes >= RESIDUAL_RECOVERY_MIN_CONFIG_VOTES
        and full_delta >= RESIDUAL_RECOVERY_MIN_MEAN_CV_DELTA
        and replay_fps >= RESIDUAL_RECOVERY_MIN_REPLAY_FPS
    )
    diagnostics = {
        'exact_noop_control_prediction_parity': True,
        'production_predictions_untouched': True,
        'identity_folds': identity_folds,
        'mean_cv_st_iou': mean_cv_score,
        'mean_cv_baseline_st_iou': mean_cv_baseline,
        'mean_cv_delta': mean_cv_score - mean_cv_baseline,
        'improved_identity_count': sum(delta > 0 for delta in deltas),
        'worst_identity_delta': min(deltas),
        'recommended_config': recommended_config,
        'recommendation_votes': recommendation_votes,
        'recommended_full_public_delta': full_delta,
        'recommended_replay_fps': replay_fps,
        'promotion_passed': promotion_passed,
    }
    video_rows = [
        {
            'video_id': video_id,
            'identity_id': identity_id_from_video_id(video_id),
            'production_st_iou': production_result['per_video'][video_id],
            'residual_st_iou': recommended_result['per_video'][video_id],
            'delta': (
                recommended_result['per_video'][video_id]
                - production_result['per_video'][video_id]
            ),
            **recommended_result['residual_diagnostics']['per_video'][video_id],
        }
        for video_id in all_video_ids
    ]
    return (
        sorted(rows, key=lambda row: row['mean_st_iou'], reverse=True),
        fold_rows, video_rows, diagnostics, recommended_result,
    )

# --- CHECKPOINT-SPECIFIC SCORE CALIBRATION V1: cache replay only ---

def transform_yolo_score(score, power):
    clipped = min(1.0, max(0.0, float(score)))
    return clipped ** float(power)


def fuse_calibrated_candidate_score(scores, weights, yolo_power,
                                    has_siamese=True, has_color=True):
    yolo_score, siamese_score, color_score = map(float, scores)
    yolo_score = transform_yolo_score(yolo_score, yolo_power)
    wy = float(weights['yolo'])
    ws = float(weights['siamese']) if has_siamese else 0.0
    wc = float(weights['color']) if has_color else 0.0
    total = wy + ws + wc
    if total <= 0:
        return yolo_score
    return (wy * yolo_score + ws * siamese_score + wc * color_score) / total


def rank_video_frames_score_calibrated(video_cache, weights, yolo_power):
    ranked = {}
    has_siamese = bool(video_cache['has_siamese'])
    has_color = bool(video_cache['has_color'])
    for frame_key, candidates in video_cache['frames'].items():
        frame_idx = int(frame_key)
        best_candidate, best_score = None, -1.0
        for candidate in candidates:
            score = fuse_calibrated_candidate_score(
                candidate['scores'], weights, yolo_power, has_siamese, has_color
            )
            if score > best_score:
                best_candidate, best_score = candidate, score
        if best_candidate is not None:
            ranked[frame_idx] = {
                'bbox': best_candidate['bbox'], 'score': float(best_score),
            }
    return ranked


def score_calibration_config_grid():
    configs = []
    for yolo_power in SCORE_CAL_YOLO_POWERS:
        for yolo_weight in SCORE_CAL_YOLO_WEIGHTS:
            for color_weight in SCORE_CAL_COLOR_WEIGHTS:
                siamese_weight = round(1.0 - yolo_weight - color_weight, 10)
                if siamese_weight + 1e-12 < SCORE_CAL_MIN_SIAMESE_WEIGHT:
                    continue
                weights = {
                    'yolo': float(yolo_weight),
                    'siamese': float(siamese_weight),
                    'color': float(color_weight),
                }
                for threshold in SCORE_CAL_THRESHOLDS:
                    configs.append({
                        'yolo_power': float(yolo_power),
                        'weights': weights,
                        'matching_threshold': float(threshold),
                    })
    default = {
        'yolo_power': 1.0,
        'weights': {
            'yolo': float(WEIGHT_YOLO),
            'siamese': float(WEIGHT_SIAMESE),
            'color': float(WEIGHT_COLOR),
        },
        'matching_threshold': float(MATCHING_THRESHOLD),
    }
    if score_calibration_config_key(default) not in {
        score_calibration_config_key(config) for config in configs
    }:
        configs.append(default)
    return configs


def score_calibration_config_key(config):
    weights = config['weights']
    return (
        float(config['yolo_power']), float(weights['yolo']),
        float(weights['siamese']), float(weights['color']),
        float(config['matching_threshold']),
    )


def score_calibration_config_from_row(row):
    return {
        'yolo_power': float(row['yolo_power']),
        'weights': {
            'yolo': float(row['weight_yolo']),
            'siamese': float(row['weight_siamese']),
            'color': float(row['weight_color']),
        },
        'matching_threshold': float(row['matching_threshold']),
    }


def score_calibration_distance_from_production(config):
    weights = config['weights']
    return (
        abs(float(config['yolo_power']) - 1.0)
        + abs(float(weights['yolo']) - WEIGHT_YOLO)
        + abs(float(weights['siamese']) - WEIGHT_SIAMESE)
        + abs(float(weights['color']) - WEIGHT_COLOR)
        + abs(float(config['matching_threshold']) - MATCHING_THRESHOLD)
    )


def evaluate_score_calibration_config(cache, gt_map, config, temporal_config):
    ranked = {
        video_id: rank_video_frames_score_calibrated(
            video_cache, config['weights'], config['yolo_power']
        )
        for video_id, video_cache in cache['videos'].items()
    }
    return render_ranked_predictions(
        cache, ranked, gt_map, config['matching_threshold'],
        temporal_config=temporal_config,
    )


def score_calibration_rows(variant_name, cache, gt_map, temporal_config):
    rows = []
    configs = score_calibration_config_grid()
    rank_groups = {}
    for config in configs:
        rank_key = (
            config['yolo_power'], config['weights']['yolo'],
            config['weights']['siamese'], config['weights']['color'],
        )
        rank_groups.setdefault(rank_key, []).append(config['matching_threshold'])
    for rank_key, thresholds in tqdm(
        rank_groups.items(), desc=f'Score calibration {variant_name}'
    ):
        yolo_power, weight_yolo, weight_siamese, weight_color = rank_key
        weights = {
            'yolo': weight_yolo, 'siamese': weight_siamese, 'color': weight_color,
        }
        ranked = {
            video_id: rank_video_frames_score_calibrated(
                video_cache, weights, yolo_power
            )
            for video_id, video_cache in cache['videos'].items()
        }
        for threshold in thresholds:
            result = render_ranked_predictions(
                cache, ranked, gt_map, threshold, temporal_config=temporal_config
            )
            row = {
                'variant': variant_name, 'yolo_power': yolo_power,
                **result_row(weights, threshold, result),
            }
            rows.append(row)
    return rows


def select_score_calibration_row(rows, selection_video_ids):
    def key(row):
        config = score_calibration_config_from_row(row)
        return (
            mean_row_score(row, selection_video_ids),
            -score_calibration_distance_from_production(config),
        )
    return max(rows, key=key)


def select_voted_score_calibration_config(fold_rows):
    vote_counts = {}
    configs = {}
    for fold in fold_rows:
        config = {
            'yolo_power': fold['yolo_power'],
            'weights': {
                'yolo': fold['weight_yolo'],
                'siamese': fold['weight_siamese'],
                'color': fold['weight_color'],
            },
            'matching_threshold': fold['matching_threshold'],
        }
        key = score_calibration_config_key(config)
        vote_counts[key] = vote_counts.get(key, 0) + 1
        configs[key] = config
    best_key = max(
        vote_counts,
        key=lambda key: (
            vote_counts[key],
            -score_calibration_distance_from_production(configs[key]),
        ),
    )
    return configs[best_key], vote_counts[best_key]


def build_score_calibration_cache_signature(base_signature, variant_name):
    signature = dict(base_signature)
    signature['schema_version'] = 6
    signature['yolo_sha256'] = P2_AB_VARIANTS[variant_name]['expected_sha256']
    signature['candidate_generation'] = {
        'mode': 'detector_checkpoint_ab_full_frame',
        'variant': variant_name,
        'imgsz': int(P2_AB_IMGSZ),
        'confidence_threshold': CONFIDENCE_THRESHOLD,
        'augment_tta': bool(USE_TTA),
        'ultralytics_version': '8.3.221',
    }
    return signature


def load_score_calibration_variant_caches(base_cache, base_signature):
    caches = {'production': base_cache}
    cache_files = {'production': CANDIDATE_CACHE_FILE}
    for variant_name in ('p3_control', 'p2_stride4'):
        cache_file = detector_checkpoint_cache_file(variant_name)
        if not os.path.exists(cache_file):
            raise FileNotFoundError(
                f'Missing complete {variant_name} cache: {cache_file}. '
                'Download candidate_scores_<variant>.json.gz from the locked A/B run, '
                'upload it as a Kaggle input, and set P2_AB_PRECOMPUTED_CACHES.'
            )
        cache = read_cache_json(cache_file, f'{variant_name} detector candidate cache')
        expected_signature = build_score_calibration_cache_signature(
            base_signature, variant_name
        )
        if cache.get('signature') != expected_signature:
            mismatch_keys = cache_signature_mismatch_keys(
                cache.get('signature'), expected_signature
            )
            raise RuntimeError(
                f'{variant_name} calibration cache signature mismatch: {mismatch_keys}'
            )
        if not cache.get('complete'):
            raise RuntimeError(f'{variant_name} calibration cache is incomplete')
        caches[variant_name] = cache
        cache_files[variant_name] = cache_file
    return caches, cache_files


def run_checkpoint_score_calibration(base_cache, base_signature, gt_map,
                                     production_result, temporal_config):
    caches, cache_files = load_score_calibration_variant_caches(
        base_cache, base_signature
    )
    default_weights = {
        'yolo': WEIGHT_YOLO, 'siamese': WEIGHT_SIAMESE, 'color': WEIGHT_COLOR,
    }
    locked_results = {'production': production_result}
    for variant_name in ('p3_control', 'p2_stride4'):
        locked_results[variant_name] = evaluate_config(
            caches[variant_name], gt_map, default_weights, MATCHING_THRESHOLD,
            temporal_config=temporal_config,
        )

    all_rows, all_fold_rows = [], []
    variant_diagnostics, recommended_results = {}, {}
    identity_folds = build_identity_folds(base_signature['video_folders'])
    all_video_ids = sorted(base_signature['video_folders'])
    default_config = {
        'yolo_power': 1.0, 'weights': default_weights,
        'matching_threshold': MATCHING_THRESHOLD,
    }
    for variant_name in ('production', 'p3_control', 'p2_stride4'):
        cache = caches[variant_name]
        control_result = evaluate_score_calibration_config(
            cache, gt_map, default_config, temporal_config
        )
        if control_result['predictions'] != locked_results[variant_name]['predictions']:
            raise RuntimeError(
                f'{variant_name} gamma=1 score calibration control lost parity'
            )
        rows = score_calibration_rows(
            variant_name, cache, gt_map, temporal_config
        )
        all_rows.extend(rows)
        variant_fold_rows = []
        for held_identity, held_video_ids in identity_folds.items():
            selection_video_ids = [
                video_id for video_id in all_video_ids
                if video_id not in held_video_ids
            ]
            selected = select_score_calibration_row(rows, selection_video_ids)
            heldout_score = mean_row_score(selected, held_video_ids)
            locked_score = sum(
                locked_results[variant_name]['per_video'][video_id]
                for video_id in held_video_ids
            ) / len(held_video_ids)
            production_score = sum(
                production_result['per_video'][video_id]
                for video_id in held_video_ids
            ) / len(held_video_ids)
            fold = {
                'variant': variant_name,
                'heldout_identity': held_identity,
                'heldout_videos': '|'.join(held_video_ids),
                'selection_videos': '|'.join(selection_video_ids),
                'yolo_power': selected['yolo_power'],
                'weight_yolo': selected['weight_yolo'],
                'weight_siamese': selected['weight_siamese'],
                'weight_color': selected['weight_color'],
                'matching_threshold': selected['matching_threshold'],
                'selection_mean_st_iou': mean_row_score(
                    selected, selection_video_ids
                ),
                'heldout_mean_st_iou': heldout_score,
                'locked_heldout_mean_st_iou': locked_score,
                'production_heldout_mean_st_iou': production_score,
                'heldout_delta_vs_locked': heldout_score - locked_score,
                'heldout_delta_vs_production': heldout_score - production_score,
            }
            variant_fold_rows.append(fold)
            all_fold_rows.append(fold)

        recommended_config, recommendation_votes = (
            select_voted_score_calibration_config(variant_fold_rows)
        )
        recommended_result = evaluate_score_calibration_config(
            cache, gt_map, recommended_config, temporal_config
        )
        recommended_results[variant_name] = recommended_result
        mean_cv_score = sum(
            row['heldout_mean_st_iou'] for row in variant_fold_rows
        ) / len(variant_fold_rows)
        mean_cv_locked = sum(
            row['locked_heldout_mean_st_iou'] for row in variant_fold_rows
        ) / len(variant_fold_rows)
        mean_cv_production = sum(
            row['production_heldout_mean_st_iou'] for row in variant_fold_rows
        ) / len(variant_fold_rows)
        deltas_vs_production = [
            row['heldout_delta_vs_production'] for row in variant_fold_rows
        ]
        improved_identities = sum(delta > 0 for delta in deltas_vs_production)
        worst_identity_delta = min(deltas_vs_production)
        runtime = cache.get('variant_runtime') or {}
        aggregate_fps = runtime.get('aggregate_fps')
        fps_passed = (
            variant_name == 'production'
            or (aggregate_fps is not None
                and aggregate_fps >= SCORE_CAL_MIN_AGGREGATE_FPS)
        )
        full_delta_vs_production = (
            recommended_result['mean_st_iou'] - production_result['mean_st_iou']
        )
        promotion_passed = (
            variant_name != 'production'
            and mean_cv_score - mean_cv_production
                >= SCORE_CAL_MIN_MEAN_CV_DELTA_VS_PRODUCTION
            and improved_identities >= SCORE_CAL_MIN_IMPROVED_IDENTITIES
            and worst_identity_delta >= -SCORE_CAL_MAX_WORST_IDENTITY_DROP
            and recommendation_votes >= SCORE_CAL_MIN_CONFIG_VOTES
            and full_delta_vs_production
                >= SCORE_CAL_MIN_MEAN_CV_DELTA_VS_PRODUCTION
            and fps_passed
        )
        variant_diagnostics[variant_name] = {
            'cache_file': cache_files[variant_name],
            'locked_mean_st_iou': locked_results[variant_name]['mean_st_iou'],
            'mean_cv_st_iou': mean_cv_score,
            'mean_cv_locked_st_iou': mean_cv_locked,
            'mean_cv_production_st_iou': mean_cv_production,
            'mean_cv_delta_vs_locked': mean_cv_score - mean_cv_locked,
            'mean_cv_delta_vs_production': mean_cv_score - mean_cv_production,
            'improved_identity_count_vs_production': improved_identities,
            'worst_identity_delta_vs_production': worst_identity_delta,
            'recommended_config': recommended_config,
            'recommendation_votes': recommendation_votes,
            'recommended_full_public_mean_st_iou': recommended_result['mean_st_iou'],
            'recommended_full_public_delta_vs_production': full_delta_vs_production,
            'aggregate_fps_from_locked_eval': aggregate_fps,
            'fps_passed': fps_passed,
            'promotion_passed': promotion_passed,
        }

    eligible = [
        variant_name for variant_name in ('p3_control', 'p2_stride4')
        if variant_diagnostics[variant_name]['promotion_passed']
    ]
    recommended_variant = (
        max(
            eligible,
            key=lambda name: variant_diagnostics[name][
                'recommended_full_public_mean_st_iou'
            ],
        )
        if eligible else 'production'
    )
    diagnostics = {
        'identity_folds': identity_folds,
        'grid_config_count_per_variant': len(score_calibration_config_grid()),
        'control_prediction_parity': True,
        'recommended_variant': recommended_variant,
        'promotion_passed': recommended_variant != 'production',
        'criteria': {
            'minimum_mean_cv_delta_vs_production': (
                SCORE_CAL_MIN_MEAN_CV_DELTA_VS_PRODUCTION
            ),
            'minimum_improved_identities': SCORE_CAL_MIN_IMPROVED_IDENTITIES,
            'maximum_worst_identity_drop': SCORE_CAL_MAX_WORST_IDENTITY_DROP,
            'minimum_identical_config_votes': SCORE_CAL_MIN_CONFIG_VOTES,
            'minimum_aggregate_fps': SCORE_CAL_MIN_AGGREGATE_FPS,
        },
        'variants': variant_diagnostics,
    }
    rows_sorted = sorted(
        all_rows, key=lambda row: (row['variant'], -row['mean_st_iou'])
    )
    return rows_sorted, all_fold_rows, diagnostics, recommended_results


# ============= [ANCHOR-PRESERVING SPATIAL REFINER V2 — OFFLINE A/B] =============

def spatial_refiner_v2_crop_spec(box, search_factor=None, min_search_side=None):
    search_factor = SPATIAL_REFINER_V2_SEARCH_FACTOR if search_factor is None else float(search_factor)
    min_search_side = SPATIAL_REFINER_V2_MIN_SEARCH_SIDE if min_search_side is None else float(min_search_side)
    x1, y1, x2, y2 = map(float, box)
    if not all(math.isfinite(value) for value in (x1, y1, x2, y2)) or x2 <= x1 or y2 <= y1:
        raise ValueError(f'Invalid production bbox for refiner: {box}')
    center_x, center_y = (x1 + x2) / 2, (y1 + y2) / 2
    side = max(max(x2 - x1, y2 - y1) * search_factor, min_search_side)
    return center_x - side / 2, center_y - side / 2, side


def spatial_refiner_v2_box_from_normalized(normalized_box, crop_spec, image_width, image_height):
    crop_x, crop_y, side = map(float, crop_spec)
    values = [float(value) for value in normalized_box]
    if len(values) != 4 or not all(math.isfinite(value) for value in values):
        return None
    low_x, high_x = sorted((values[0], values[2]))
    low_y, high_y = sorted((values[1], values[3]))
    box = [crop_x + low_x * side, crop_y + low_y * side,
           crop_x + high_x * side, crop_y + high_y * side]
    box = [
        min(max(box[0], 0.0), float(image_width)),
        min(max(box[1], 0.0), float(image_height)),
        min(max(box[2], 0.0), float(image_width)),
        min(max(box[3], 0.0), float(image_height)),
    ]
    if box[2] - box[0] < 1.0 or box[3] - box[1] < 1.0:
        return None
    return box


def spatial_refiner_v2_box_to_normalized(box, crop_spec):
    crop_x, crop_y, side = map(float, crop_spec)
    values = [float(value) for value in box]
    normalized = [
        (values[0] - crop_x) / side,
        (values[1] - crop_y) / side,
        (values[2] - crop_x) / side,
        (values[3] - crop_y) / side,
    ]
    if not all(math.isfinite(value) for value in normalized):
        raise ValueError(f'Invalid normalized coarse bbox: {box}')
    return normalized


def spatial_refiner_v2_plausibility(original, refined, max_center_shift=None,
                                  max_log_scale_change=None):
    if refined is None:
        return False, 'invalid_geometry', {}
    max_center_shift = (SPATIAL_REFINER_V2_MAX_CENTER_SHIFT if max_center_shift is None
                        else float(max_center_shift))
    max_log_scale_change = (SPATIAL_REFINER_V2_MAX_LOG_SCALE_CHANGE
                            if max_log_scale_change is None else float(max_log_scale_change))
    ox, oy, ow, oh = spatial_bbox_geometry(original)
    rx, ry, rw, rh = spatial_bbox_geometry(refined)
    center_shift = math.hypot(rx - ox, ry - oy) / math.sqrt(ow * oh)
    log_scale_change = max(abs(math.log(rw / ow)), abs(math.log(rh / oh)))
    metrics = {'center_shift': center_shift, 'max_abs_log_scale_change': log_scale_change}
    if center_shift > max_center_shift:
        return False, 'center_shift', metrics
    if log_scale_change > max_log_scale_change:
        return False, 'scale_change', metrics
    return True, 'accepted', metrics


def spatial_refiner_v2_image_tensor(image_bgr, size):
    resized = cv2.resize(image_bgr, (int(size), int(size)), interpolation=cv2.INTER_LINEAR)
    rgb = cv2.cvtColor(resized, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
    mean = np.asarray([0.485, 0.456, 0.406], dtype=np.float32)
    std = np.asarray([0.229, 0.224, 0.225], dtype=np.float32)
    rgb = (rgb - mean) / std
    return torch.from_numpy(rgb.transpose(2, 0, 1).copy())


def spatial_refiner_v2_search_crop(frame, production_box):
    crop_spec = spatial_refiner_v2_crop_spec(production_box)
    crop_x, crop_y, side = crop_spec
    scale = SPATIAL_REFINER_V2_SEARCH_SIZE / side
    matrix = np.asarray([[scale, 0.0, -crop_x * scale],
                         [0.0, scale, -crop_y * scale]], dtype=np.float32)
    crop = cv2.warpAffine(
        frame, matrix, (SPATIAL_REFINER_V2_SEARCH_SIZE, SPATIAL_REFINER_V2_SEARCH_SIZE),
        flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_REPLICATE,
    )
    return spatial_refiner_v2_image_tensor(crop, SPATIAL_REFINER_V2_SEARCH_SIZE), crop_spec


def load_spatial_refiner_v2_artifacts():
    for path in (SPATIAL_REFINER_V2_MODEL_PATH, SPATIAL_REFINER_V2_RUN_SUMMARY_PATH):
        if not os.path.isfile(path):
            raise FileNotFoundError(f'Upload Spatial Refiner V2 artifact and update path: {path}')
    with open(SPATIAL_REFINER_V2_RUN_SUMMARY_PATH, encoding='utf-8') as handle:
        training_summary = json.load(handle)
    if training_summary.get('experiment') != 'spatial_refiner_v2_identity_cv':
        raise RuntimeError(f"Unexpected refiner experiment: {training_summary.get('experiment')}")
    preflight = training_summary.get('torchscript_preflight', {})
    if preflight.get('identity_initialization_exact') is not True:
        raise RuntimeError('Refiner V2 summary does not certify identity initialization')
    if training_summary.get('public_test_used_for_training_or_selection') is not False:
        raise RuntimeError('Refiner training summary does not certify public-test isolation')
    if training_summary.get('status') != 'trained_final':
        raise RuntimeError('Refiner identity-CV did not export a final model')
    if not training_summary.get('identity_cv', {}).get('promotion_passed', False):
        raise RuntimeError('Refiner identity-CV promotion gate failed; locked public A/B is blocked')
    expected_hash = training_summary.get('artifacts', {}).get('torchscript_sha256')
    actual_hash = file_sha256(SPATIAL_REFINER_V2_MODEL_PATH)
    if expected_hash != actual_hash:
        raise RuntimeError(f'Refiner TorchScript SHA256 mismatch: expected={expected_hash}, got={actual_hash}')
    preprocessing = training_summary.get('preprocessing', {})
    expected = {
        'search_size': SPATIAL_REFINER_V2_SEARCH_SIZE,
        'reference_size': SPATIAL_REFINER_V2_REFERENCE_SIZE,
        'search_factor': SPATIAL_REFINER_V2_SEARCH_FACTOR,
        'min_search_side': SPATIAL_REFINER_V2_MIN_SEARCH_SIDE,
    }
    for name, value in expected.items():
        actual = preprocessing.get(name)
        if actual is None or not math.isclose(float(actual), float(value), rel_tol=0, abs_tol=1e-12):
            raise RuntimeError(f'Refiner preprocessing mismatch for {name}: {actual} != {value}')
    refiner_device = torch.device('cuda:1' if torch.cuda.device_count() > 1 else DEVICE)
    model = torch.jit.load(SPATIAL_REFINER_V2_MODEL_PATH, map_location=refiner_device).eval()
    return model, refiner_device, training_summary, actual_hash


def load_spatial_refiner_v2_reference(video_id, refiner_device):
    folder = os.path.join(TEST_DATA_DIR, video_id, 'object_images')
    paths = sorted(
        os.path.join(folder, name) for name in os.listdir(folder)
        if name.lower().endswith(('.jpg', '.jpeg', '.png'))
    ) if os.path.isdir(folder) else []
    for path in paths:
        image = cv2.imread(path)
        if image is not None:
            tensor = spatial_refiner_v2_image_tensor(
                image, SPATIAL_REFINER_V2_REFERENCE_SIZE
            ).unsqueeze(0).to(refiner_device)
            return tensor, path
    raise FileNotFoundError(f'No readable refiner reference image for {video_id}')


def replace_spatial_refiner_v2_boxes(production_predictions, refined_maps):
    output = copy.deepcopy(production_predictions)
    before_frames, after_frames = [], []
    for prediction in output:
        video_id = prediction['video_id']
        replacements = refined_maps.get(video_id, {})
        for detection in prediction.get('detections', []):
            for bbox in detection.get('bboxes', []):
                frame = int(bbox['frame'])
                before_frames.append((video_id, frame))
                if frame in replacements:
                    values = replacements[frame]
                    for key, value in zip(('x1', 'y1', 'x2', 'y2'), values):
                        bbox[key] = int(round(value))
                after_frames.append((video_id, frame))
    if before_frames != after_frames or len(set(after_frames)) != len(after_frames):
        raise RuntimeError('Spatial refiner changed or duplicated production frame support')
    return output


def evaluate_prediction_payload(predictions, gt_map):
    frame_maps = prediction_frame_maps(predictions)
    per_video = {
        video_id: compute_st_iou_video(gt_map[video_id], frame_maps.get(video_id, {}))
        for video_id in sorted(gt_map)
    }
    return {
        'predictions': predictions, 'per_video': per_video,
        'mean_st_iou': sum(per_video.values()) / len(per_video) if per_video else 0.0,
        'detected_frames': sum(len(frames) for frames in frame_maps.values()),
    }


def run_spatial_refiner_v2_locked_ab(production_result, gt_map):
    model, refiner_device, training_summary, model_hash = load_spatial_refiner_v2_artifacts()
    production_maps = prediction_frame_maps(production_result['predictions'])
    refined_maps, frame_rows, video_diagnostics = {}, [], {}
    replay_start = time.perf_counter()
    total_model_seconds = 0.0
    for video_id in sorted(production_maps):
        frame_map = production_maps[video_id]
        selected_frames = sorted(frame_map)
        reference_tensor, reference_path = load_spatial_refiner_v2_reference(
            video_id, refiner_device
        )
        video_path = os.path.join(TEST_DATA_DIR, video_id, 'drone_video.mp4')
        capture = cv2.VideoCapture(video_path)
        if not capture.isOpened():
            raise RuntimeError(f'Cannot open video for refiner replay: {video_path}')
        image_width = int(capture.get(cv2.CAP_PROP_FRAME_WIDTH))
        image_height = int(capture.get(cv2.CAP_PROP_FRAME_HEIGHT))
        pending_tensors, pending_metadata = [], []
        replacements, reason_counts = {}, {}

        def flush_batch():
            nonlocal pending_tensors, pending_metadata, total_model_seconds
            if not pending_tensors:
                return
            search_batch = torch.stack(pending_tensors).to(refiner_device, non_blocking=True)
            references = reference_tensor.expand(search_batch.shape[0], -1, -1, -1)
            coarse_batch = torch.tensor(
                [spatial_refiner_v2_box_to_normalized(item[1], item[2])
                 for item in pending_metadata],
                dtype=search_batch.dtype, device=refiner_device,
            )
            if refiner_device.type == 'cuda':
                torch.cuda.synchronize(refiner_device)
            model_start = time.perf_counter()
            with torch.no_grad(), torch.amp.autocast(
                device_type=refiner_device.type, enabled=refiner_device.type == 'cuda'
            ):
                normalized_boxes, _ = model(references, search_batch, coarse_batch)
            if refiner_device.type == 'cuda':
                torch.cuda.synchronize(refiner_device)
            total_model_seconds += time.perf_counter() - model_start
            normalized_boxes = normalized_boxes.detach().float().cpu().numpy()
            for normalized, metadata in zip(normalized_boxes, pending_metadata):
                frame_idx, production_box, crop_spec = metadata
                proposed = spatial_refiner_v2_box_from_normalized(
                    normalized, crop_spec, image_width, image_height
                )
                accepted, reason, geometry = spatial_refiner_v2_plausibility(
                    production_box, proposed
                )
                deployed = (
                    [int(round(value)) for value in proposed]
                    if accepted else list(map(float, production_box))
                )
                replacements[frame_idx] = deployed
                reason_counts[reason] = reason_counts.get(reason, 0) + 1
                gt = gt_map.get(video_id, {}).get(frame_idx)
                frame_rows.append({
                    'video_id': video_id,
                    'identity_id': identity_id_from_video_id(video_id),
                    'frame': frame_idx,
                    'accepted': int(accepted),
                    'reason': reason,
                    'center_shift': geometry.get('center_shift'),
                    'max_abs_log_scale_change': geometry.get('max_abs_log_scale_change'),
                    'production_iou': iou_box(gt, production_box) if gt is not None else None,
                    'refined_iou': iou_box(gt, deployed) if gt is not None else None,
                    **{f'production_{key}': value for key, value in zip(('x1','y1','x2','y2'), production_box)},
                    **{f'refined_{key}': value for key, value in zip(('x1','y1','x2','y2'), deployed)},
                })
            pending_tensors, pending_metadata = [], []

        selected_set = set(selected_frames)
        frame_idx = 0
        while capture.isOpened() and selected_set:
            ok, frame = capture.read()
            if not ok:
                break
            if frame_idx in selected_set:
                tensor, crop_spec = spatial_refiner_v2_search_crop(frame, frame_map[frame_idx])
                pending_tensors.append(tensor)
                pending_metadata.append((frame_idx, frame_map[frame_idx], crop_spec))
                selected_set.remove(frame_idx)
                if len(pending_tensors) >= SPATIAL_REFINER_V2_BATCH_SIZE:
                    flush_batch()
            frame_idx += 1
        flush_batch()
        capture.release()
        if selected_set:
            raise RuntimeError(f'Video ended before production frames: {video_id} {sorted(selected_set)[:5]}')
        if set(replacements) != set(frame_map):
            raise RuntimeError(f'Refiner frame support mismatch for {video_id}')
        refined_maps[video_id] = replacements
        video_diagnostics[video_id] = {
            'reference_path': reference_path,
            'frame_count': len(frame_map),
            'accepted_count': reason_counts.get('accepted', 0),
            'fallback_count': len(frame_map) - reason_counts.get('accepted', 0),
            'reason_counts': reason_counts,
        }
    replay_seconds = time.perf_counter() - replay_start
    predictions = replace_spatial_refiner_v2_boxes(production_result['predictions'], refined_maps)
    refined_result = evaluate_prediction_payload(predictions, gt_map)
    if refined_result['detected_frames'] != production_result['detected_frames']:
        raise RuntimeError('Spatial refiner changed prediction frame count')
    video_rows = []
    for video_id in sorted(refined_result['per_video']):
        production_score = production_result['per_video'][video_id]
        refined_score = refined_result['per_video'][video_id]
        video_rows.append({
            'video_id': video_id, 'identity_id': identity_id_from_video_id(video_id),
            'production_st_iou': production_score, 'refined_st_iou': refined_score,
            'delta': refined_score - production_score,
            **video_diagnostics[video_id],
        })
    identities = sorted({row['identity_id'] for row in video_rows})
    identity_deltas = {
        identity: sum(row['delta'] for row in video_rows if row['identity_id'] == identity)
                  / sum(1 for row in video_rows if row['identity_id'] == identity)
        for identity in identities
    }
    model_fps = len(frame_rows) / total_model_seconds if total_model_seconds > 0 else None
    replay_fps = len(frame_rows) / replay_seconds if replay_seconds > 0 else None
    sequential_fps = (1.0 / (1.0 / SPATIAL_REFINER_V2_BASELINE_REFERENCE_FPS + 1.0 / model_fps)
                      if model_fps else None)
    dual_gpu_pipeline_fps = min(SPATIAL_REFINER_V2_BASELINE_REFERENCE_FPS, model_fps) if model_fps else None
    mean_delta = refined_result['mean_st_iou'] - production_result['mean_st_iou']
    improved_identities = sum(value > 0 for value in identity_deltas.values())
    worst_identity_delta = min(identity_deltas.values())
    quality_passed = (
        mean_delta >= SPATIAL_REFINER_V2_MIN_PUBLIC_DELTA
        and improved_identities >= SPATIAL_REFINER_V2_MIN_IMPROVED_IDENTITIES
        and worst_identity_delta >= -SPATIAL_REFINER_V2_MAX_WORST_IDENTITY_DROP
    )
    diagnostics = {
        'production_predictions_unchanged': True,
        'frame_support_parity': True,
        'torchscript_sha256': model_hash,
        'training_identity_cv': training_summary['identity_cv'],
        'reference_policy': 'first readable reference in lexicographic order; one global policy',
        'processed_frames': len(frame_rows),
        'accepted_frames': sum(row['accepted'] for row in frame_rows),
        'fallback_frames': sum(1 - row['accepted'] for row in frame_rows),
        'replay_seconds_including_decode_and_preprocess': replay_seconds,
        'model_seconds': total_model_seconds,
        'replay_fps': replay_fps,
        'model_only_fps': model_fps,
        'estimated_sequential_pipeline_fps': sequential_fps,
        'estimated_t4x2_pipelined_fps': dual_gpu_pipeline_fps,
        'identity_deltas': identity_deltas,
        'mean_st_iou_delta': mean_delta,
        'improved_identity_count': improved_identities,
        'worst_identity_delta': worst_identity_delta,
        'quality_promotion_passed': quality_passed,
        'speed_estimate_passed': (dual_gpu_pipeline_fps is not None and
                                  dual_gpu_pipeline_fps >= SPATIAL_REFINER_V2_MIN_PIPELINED_FPS),
        'promotion_passed': quality_passed,
        'speed_measurement_descriptive_only': True,
        'speed_caveat': 'T4x2 pipelined FPS is an estimate; confirm with integrated one-pass profiling before production.',
    }
    return refined_result, frame_rows, video_rows, diagnostics


# --- 7. MAIN ---

def print_evaluation(label, result):
    print(f"\n{label} | Mean ST-IoU: {result['mean_st_iou']:.6f} | Frames: {result['detected_frames']}")
    for vid, score in sorted(result['per_video'].items()):
        print(f'  {vid:30s}: {score:.4f}')


def main():
    os.makedirs(CALIBRATION_DIR, exist_ok=True)
    active_experiments = sum([
        bool(RUN_CALIBRATION), bool(RUN_TEMPORAL_CALIBRATION),
        bool(RUN_RERANKING_EXPERIMENT), bool(RUN_ORACLE_DIAGNOSTICS),
        bool(RUN_HYSTERESIS_EXPERIMENT), bool(RUN_MARGIN_DIAGNOSTICS),
        bool(RUN_DETECTOR_PROFILE), bool(RUN_RESOLUTION_AB),
        bool(RUN_P2_DETECTOR_AB), bool(RUN_CHECKPOINT_SCORE_CALIBRATION),
        bool(RUN_HYBRID_SAHI), bool(RUN_SAHI_ADMISSION),
        bool(RUN_TRACKLET_GRAPH_EXPERIMENT),
        bool(RUN_MOTION_PROFILE),
        bool(RUN_RESIDUAL_RECOVERY_EXPERIMENT),
        bool(RUN_SPATIAL_HEADROOM),
        bool(RUN_SPATIAL_REFINER_V2_AB),
    ])
    if active_experiments > 1:
        raise RuntimeError(
            'Run fusion, temporal, reranking, oracle, hysteresis, margin, detector profile, resolution A/B, P2 detector A/B, checkpoint score calibration, SAHI, SAHI admission, tracklet graph, motion profile, residual recovery, spatial headroom, and spatial refiner separately'
        )
    video_folders = list_video_folders()
    signature = build_cache_signature(video_folders)
    cache = load_or_initialize_cache(signature)
    cache = extract_candidate_cache(cache, video_folders)
    gt_map = load_gt_annotations(GT_ANN_PATH)
    missing_gt = [vid for vid in video_folders if vid not in gt_map]
    if active_experiments and missing_gt:
        raise RuntimeError(
            'Calibration requires GT for every video; missing: ' + ', '.join(missing_gt)
        )

    default_weights = {
        'yolo': WEIGHT_YOLO, 'siamese': WEIGHT_SIAMESE, 'color': WEIGHT_COLOR
    }
    # A sweep always starts from the exact legacy temporal control. Production uses
    # the locked gated configuration without changing calibration replay behavior.
    active_temporal_config = (
        None if (RUN_CALIBRATION or RUN_TEMPORAL_CALIBRATION or not USE_TEMPORAL_GATING)
        else PRODUCTION_TEMPORAL_CONFIG
    )
    default_result = evaluate_config(
        cache, gt_map, default_weights, MATCHING_THRESHOLD,
        include_debug=not active_experiments,
        temporal_config=active_temporal_config,
    )
    atomic_write_json(OUTPUT_FILE, default_result['predictions'])
    is_sweep = RUN_CALIBRATION or RUN_TEMPORAL_CALIBRATION
    replay_label = 'CACHE REPLAY - LEGACY TEMPORAL' if is_sweep else 'PRODUCTION CONFIG'
    print_evaluation(replay_label, default_result)

    if RUN_SPATIAL_REFINER_V2_AB:
        refined_result, frame_rows, video_rows, diagnostics = run_spatial_refiner_v2_locked_ab(
            default_result, gt_map
        )
        os.makedirs(SPATIAL_REFINER_V2_OUTPUT_DIR, exist_ok=True)
        for path, rows in [(SPATIAL_REFINER_V2_FRAMES_CSV, frame_rows),
                           (SPATIAL_REFINER_V2_VIDEOS_CSV, video_rows)]:
            if not rows:
                raise RuntimeError(f'Spatial refiner produced no rows for {path}')
            with open(path + '.tmp', 'w', newline='', encoding='utf-8') as handle:
                writer = csv.DictWriter(handle, fieldnames=list(rows[0]))
                writer.writeheader()
                writer.writerows(rows)
            os.replace(path + '.tmp', path)
        atomic_write_json(SPATIAL_REFINER_V2_PREDICTIONS_FILE, refined_result['predictions'])
        summary = {
            'experiment': SPATIAL_REFINER_V2_EXPERIMENT_NAME,
            'status': 'locked_public_ab_not_auto_promoted',
            'selection_protocol': (
                'Training and checkpoint selection use train-set identity-held-out CV only. '
                'Public GT is used once for descriptive locked A/B; no per-object tuning.'
            ),
            'fixed_matching': {
                'weights': default_weights, 'threshold': MATCHING_THRESHOLD,
            },
            'fixed_temporal': active_temporal_config,
            'production_control': {
                'mean_st_iou': default_result['mean_st_iou'],
                'detected_frames': default_result['detected_frames'],
                'per_video': default_result['per_video'],
            },
            'spatial_refiner': {
                'mean_st_iou': refined_result['mean_st_iou'],
                'detected_frames': refined_result['detected_frames'],
                'per_video': refined_result['per_video'],
            },
            'promotion_criteria': {
                'minimum_public_delta': SPATIAL_REFINER_V2_MIN_PUBLIC_DELTA,
                'minimum_improved_identities': SPATIAL_REFINER_V2_MIN_IMPROVED_IDENTITIES,
                'maximum_worst_identity_drop': SPATIAL_REFINER_V2_MAX_WORST_IDENTITY_DROP,
                'speed_is_not_a_promotion_gate': True,
            },
            'diagnostics': diagnostics,
            'outputs': {
                'predictions': SPATIAL_REFINER_V2_PREDICTIONS_FILE,
                'frames': SPATIAL_REFINER_V2_FRAMES_CSV,
                'videos': SPATIAL_REFINER_V2_VIDEOS_CSV,
            },
        }
        atomic_write_json(SPATIAL_REFINER_V2_SUMMARY_FILE, summary)
        print_evaluation('SPATIAL REFINER V1 - LOCKED PUBLIC A/B', refined_result)
        print(f"  Delta: {diagnostics['mean_st_iou_delta']:+.6f}")
        print(f"  Accepted/fallback: {diagnostics['accepted_frames']}/{diagnostics['fallback_frames']}")
        print(f"  Model FPS: {diagnostics['model_only_fps']:.2f} | estimated T4x2 pipeline FPS: {diagnostics['estimated_t4x2_pipelined_fps']:.2f}")
        print(f"  Quality gate: {diagnostics['quality_promotion_passed']} | full promotion gate: {diagnostics['promotion_passed']}")
        print(f'Spatial refiner summary: {SPATIAL_REFINER_V2_SUMMARY_FILE}')
        return

    if RUN_SPATIAL_HEADROOM:
        frame_rows, group_rows, video_rows, diagnostics = run_spatial_headroom_diagnostics(
            cache, gt_map, default_result
        )
        os.makedirs(SPATIAL_HEADROOM_OUTPUT_DIR, exist_ok=True)
        outputs = {}
        for suffix, rows in [('frames', frame_rows), ('groups', group_rows), ('videos', video_rows)]:
            if not rows:
                raise RuntimeError(f'Spatial diagnostic has no {suffix} rows')
            path = os.path.join(SPATIAL_HEADROOM_OUTPUT_DIR,
                                f'{SPATIAL_HEADROOM_EXPERIMENT_NAME}_{suffix}.csv')
            with open(path + '.tmp', 'w', newline='', encoding='utf-8') as handle:
                writer = csv.DictWriter(handle, fieldnames=list(rows[0]))
                writer.writeheader()
                writer.writerows(rows)
            os.replace(path + '.tmp', path)
            outputs[suffix] = path
        summary = {
            'experiment': SPATIAL_HEADROOM_EXPERIMENT_NAME,
            'status': 'diagnostic_only_no_refiner_trained_or_promoted',
            'cache_signature': signature,
            'fixed_matching': {'weights': default_weights, 'threshold': MATCHING_THRESHOLD},
            'fixed_temporal': active_temporal_config,
            'diagnostic_constants': {
                'reference_imgsz': SPATIAL_HEADROOM_REFERENCE_IMGSZ,
                'good_overlap_iou': SPATIAL_HEADROOM_GOOD_IOU,
                'local_candidate_overlap_iou': SPATIAL_HEADROOM_LOCAL_OVERLAP,
                'minimum_refiner_ceiling_delta': SPATIAL_HEADROOM_MIN_REFINER_CEILING_DELTA,
                'minimum_identity_ceiling_delta': SPATIAL_HEADROOM_MIN_IDENTITY_CEILING_DELTA,
                'minimum_qualifying_identities': SPATIAL_HEADROOM_MIN_QUALIFYING_IDENTITIES,
            },
            'production_control': {
                'mean_st_iou': default_result['mean_st_iou'],
                'detected_frames': default_result['detected_frames'],
                'per_video': default_result['per_video'],
            },
            'diagnostics': diagnostics,
            'outputs': outputs,
        }
        summary_path = os.path.join(SPATIAL_HEADROOM_OUTPUT_DIR,
                                    f'{SPATIAL_HEADROOM_EXPERIMENT_NAME}_summary.json')
        atomic_write_json(summary_path, summary)
        print('\nSPATIAL LOCALIZATION HEADROOM V1 - GT ORACLES, NOT MODEL RESULTS')
        for name, result in diagnostics['scenarios'].items():
            print(f"  {name:28s}: ST-IoU={result['mean_st_iou']:.6f} | delta={result['macro_delta']:+.6f}")
        counts = diagnostics['global']
        print(f"  Shared frames={counts['shared_frame_count']} | good overlap={counts['good_overlap_frame_count']} | low overlap={counts['low_overlap_frame_count']}")
        print(f"  Missed GT={counts['missed_gt_count']} | background predictions={counts['background_prediction_count']}")
        decision = diagnostics['refiner_ab_decision']
        print(
            f"  Alpha-Refine A/B gate: {decision['run_alpha_refine_ab']} | "
            f"ceiling delta={decision['observed_macro_ceiling_delta']:+.6f} | "
            f"qualifying identities={decision['qualifying_identity_count']}"
        )
        print('  Per-video good-overlap spatial upper-bound delta:')
        for row in video_rows:
            print(f"    {row['video_id']:30s}: {row['perfect_good_overlap_delta']:+.6f}")
        print(f'Spatial headroom summary: {summary_path}')
        return

    if RUN_RESIDUAL_RECOVERY_EXPERIMENT:
        rows, fold_rows, video_rows, diagnostics, recommended_result = (
            run_residual_recovery_identity_cv(
                cache, gt_map, default_weights, MATCHING_THRESHOLD, default_result
            )
        )
        os.makedirs(RESIDUAL_RECOVERY_OUTPUT_DIR, exist_ok=True)
        for path, output_rows in [
            (RESIDUAL_RECOVERY_RESULTS_CSV, rows),
            (RESIDUAL_RECOVERY_FOLDS_CSV, fold_rows),
            (RESIDUAL_RECOVERY_VIDEO_CSV, video_rows),
        ]:
            with open(path + '.tmp', 'w', newline='', encoding='utf-8') as f:
                writer = csv.DictWriter(f, fieldnames=list(output_rows[0].keys()))
                writer.writeheader()
                writer.writerows(output_rows)
            os.replace(path + '.tmp', path)
        atomic_write_json(
            RESIDUAL_RECOVERY_PREDICTIONS_FILE,
            recommended_result['predictions'],
        )
        summary = {
            'experiment': RESIDUAL_RECOVERY_EXPERIMENT_NAME,
            'status': 'experiment_only_not_auto_promoted',
            'purpose': (
                'Recover dense weak top-K candidate chains only inside unresolved gaps '
                'between immutable production anchors.'
            ),
            'scene_boundary_rule': (
                'Do not bridge anchor gaps whose normalized endpoint motion exceeds the '
                'global max_center_speed. This resets recovery when the camera pans away '
                'or a scene discontinuity moves the target to a different image region.'
            ),
            'production_preservation': (
                'The algorithm deep-copies production predictions, never overwrites or '
                'deletes an existing bbox, and appends only recovered missing frames.'
            ),
            'selection_protocol': (
                'Leave one identity out with both _0/_1 videos held together. Full-public '
                'score is descriptive and no per-object parameter is allowed.'
            ),
            'cache_signature': signature,
            'fixed_matching': {
                'weights': default_weights,
                'strong_threshold': MATCHING_THRESHOLD,
            },
            'fixed_production_temporal': active_temporal_config,
            'search_space': {
                'top_k': RESIDUAL_RECOVERY_TOP_K,
                'weak_threshold': RESIDUAL_RECOVERY_WEAK_THRESHOLD_GRID,
                'max_anchor_gap': RESIDUAL_RECOVERY_MAX_ANCHOR_GAP_GRID,
                'max_center_speed': RESIDUAL_RECOVERY_MAX_CENTER_SPEED_GRID,
                'max_candidate_gap': RESIDUAL_RECOVERY_MAX_CANDIDATE_GAP,
                'min_recovered_frames': RESIDUAL_RECOVERY_MIN_RECOVERED_FRAMES,
                'min_gap_coverage': RESIDUAL_RECOVERY_MIN_GAP_COVERAGE,
                'config_count': len(rows),
            },
            'promotion_criteria': {
                'minimum_mean_cv_delta': RESIDUAL_RECOVERY_MIN_MEAN_CV_DELTA,
                'minimum_improved_identities': RESIDUAL_RECOVERY_MIN_IMPROVED_IDENTITIES,
                'maximum_worst_identity_drop': RESIDUAL_RECOVERY_MAX_WORST_IDENTITY_DROP,
                'minimum_identical_config_votes': RESIDUAL_RECOVERY_MIN_CONFIG_VOTES,
                'minimum_full_public_delta': RESIDUAL_RECOVERY_MIN_MEAN_CV_DELTA,
                'minimum_replay_fps': RESIDUAL_RECOVERY_MIN_REPLAY_FPS,
            },
            'production_control': {
                'mean_st_iou': default_result['mean_st_iou'],
                'detected_frames': default_result['detected_frames'],
                'per_video': default_result['per_video'],
            },
            'cross_validation': diagnostics,
            'fold_results': fold_rows,
            'recommended_full_public_result_descriptive_only': {
                'mean_st_iou': recommended_result['mean_st_iou'],
                'detected_frames': recommended_result['detected_frames'],
                'per_video': recommended_result['per_video'],
                'residual_diagnostics': recommended_result['residual_diagnostics'],
            },
            'top_full_public_results_descriptive_only': rows,
            'outputs': {
                'all_results': RESIDUAL_RECOVERY_RESULTS_CSV,
                'identity_cv_folds': RESIDUAL_RECOVERY_FOLDS_CSV,
                'per_video': RESIDUAL_RECOVERY_VIDEO_CSV,
                'recommended_predictions': RESIDUAL_RECOVERY_PREDICTIONS_FILE,
            },
        }
        atomic_write_json(RESIDUAL_RECOVERY_SUMMARY_FILE, summary)
        print('\nANCHOR-PRESERVING RESIDUAL RECOVERY V2 - IDENTITY CV')
        print(f"  recommended={diagnostics['recommended_config']}")
        print(
            f"  CV delta={diagnostics['mean_cv_delta']:+.6f} | "
            f"improved identities={diagnostics['improved_identity_count']}/"
            f"{len(fold_rows)} | worst delta={diagnostics['worst_identity_delta']:+.6f} | "
            f"votes={diagnostics['recommendation_votes']}"
        )
        print(
            f"  Full-public delta={diagnostics['recommended_full_public_delta']:+.6f} | "
            f"replay FPS={diagnostics['recommended_replay_fps']:.2f}"
        )
        residual = recommended_result['residual_diagnostics']
        print(
            f"  Added frames={residual['recovered_frame_count']} | "
            f"recovered gaps={residual['recovered_gap_count']} | "
            f"scene resets={residual['scene_discontinuity_reset_count']}"
        )
        print(f"  Promotion criteria passed: {diagnostics['promotion_passed']}")
        print_evaluation(
            'RESIDUAL RECOVERY - DESCRIPTIVE FULL PUBLIC', recommended_result
        )
        print(f'Residual summary: {RESIDUAL_RECOVERY_SUMMARY_FILE}')
        print(f'All rows: {RESIDUAL_RECOVERY_RESULTS_CSV}')
        print(f'Identity folds: {RESIDUAL_RECOVERY_FOLDS_CSV}')
        print(f'Per-video diagnostics: {RESIDUAL_RECOVERY_VIDEO_CSV}')
        return

    if RUN_MOTION_PROFILE:
        frame_rows, group_rows, video_rows, diagnostics = (
            run_motion_distribution_diagnostics(
                cache, gt_map, default_weights, default_result
            )
        )
        os.makedirs(MOTION_PROFILE_OUTPUT_DIR, exist_ok=True)
        for path, rows in [
            (MOTION_PROFILE_FRAME_CSV, frame_rows),
            (MOTION_PROFILE_GROUP_CSV, group_rows),
            (MOTION_PROFILE_VIDEO_CSV, video_rows),
        ]:
            if not rows:
                raise RuntimeError(f'Motion profile produced no rows for {path}')
            with open(path + '.tmp', 'w', newline='', encoding='utf-8') as f:
                writer = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
                writer.writeheader()
                writer.writerows(rows)
            os.replace(path + '.tmp', path)
        summary = {
            'experiment': MOTION_PROFILE_EXPERIMENT_NAME,
            'status': 'diagnostic_only_no_inference_behavior_change',
            'purpose': (
                'Measure true and candidate adjacent-frame motion after Candidate '
                'Tracklet Graph V1 over-suppressed strong production detections.'
            ),
            'overfitting_control': (
                'GT is used only for read-only profiling after production predictions '
                'are frozen. No threshold, gate, candidate rank, or bbox is modified.'
            ),
            'cache_signature': signature,
            'fixed_matching': {
                'weights': default_weights,
                'matching_threshold': MATCHING_THRESHOLD,
            },
            'fixed_temporal': active_temporal_config,
            'production_control': {
                'mean_st_iou': default_result['mean_st_iou'],
                'detected_frames': default_result['detected_frames'],
                'per_video': default_result['per_video'],
            },
            'diagnostics': diagnostics,
            'outputs': {
                'adjacent_pair_rows': MOTION_PROFILE_FRAME_CSV,
                'group_profiles': MOTION_PROFILE_GROUP_CSV,
                'per_video_profiles': MOTION_PROFILE_VIDEO_CSV,
            },
        }
        atomic_write_json(MOTION_PROFILE_SUMMARY_FILE, summary)
        global_gt = diagnostics['global']['gt']
        global_selected = diagnostics['global']['selected_top1']
        print('\nMOTION DISTRIBUTION DIAGNOSTICS V1')
        print(
            f"  Adjacent GT pairs: {diagnostics['adjacent_gt_pair_count']} | "
            f"GT speed p50={global_gt['center_speed_p50']:.4f} | "
            f"p90={global_gt['center_speed_p90']:.4f} | "
            f"p95={global_gt['center_speed_p95']:.4f} | "
            f"p99={global_gt['center_speed_p99']:.4f}"
        )
        for threshold in MOTION_PROFILE_GATE_THRESHOLDS:
            suffix = str(threshold).replace('.', '')
            print(
                f"  gate>{threshold}: GT={global_gt[f'exceeds_gate_{suffix}_rate']:.4f} | "
                f"correct top1 pairs falsely rejected="
                f"{global_selected[f'good_pair_exceeds_gate_{suffix}_rate']:.4f}"
            )
        print('  GT speed by projected target size:')
        for size_bin, reports in diagnostics['by_size_bin'].items():
            report = reports['gt']
            print(
                f"    {size_bin:20s}: pairs={report['pair_count']} | "
                f"p90={report['center_speed_p90']:.4f} | "
                f">0.4={report['exceeds_gate_04_rate']:.4f} | "
                f">0.6={report['exceeds_gate_06_rate']:.4f}"
            )
        print(f'Motion profile summary: {MOTION_PROFILE_SUMMARY_FILE}')
        print(f'Adjacent-pair rows: {MOTION_PROFILE_FRAME_CSV}')
        print(f'Group profiles: {MOTION_PROFILE_GROUP_CSV}')
        print(f'Per-video profiles: {MOTION_PROFILE_VIDEO_CSV}')
        return

    if RUN_TRACKLET_GRAPH_EXPERIMENT:
        rows, fold_rows, video_rows, diagnostics, recommended_result = (
            run_tracklet_graph_identity_cv(
                cache, gt_map, default_weights, MATCHING_THRESHOLD,
                default_result, active_temporal_config,
            )
        )
        os.makedirs(TRACKLET_GRAPH_OUTPUT_DIR, exist_ok=True)
        with open(
            TRACKLET_GRAPH_RESULTS_CSV + '.tmp', 'w', newline='', encoding='utf-8'
        ) as f:
            writer = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
            writer.writeheader()
            writer.writerows(rows)
        os.replace(TRACKLET_GRAPH_RESULTS_CSV + '.tmp', TRACKLET_GRAPH_RESULTS_CSV)
        with open(
            TRACKLET_GRAPH_FOLDS_CSV + '.tmp', 'w', newline='', encoding='utf-8'
        ) as f:
            writer = csv.DictWriter(f, fieldnames=list(fold_rows[0].keys()))
            writer.writeheader()
            writer.writerows(fold_rows)
        os.replace(TRACKLET_GRAPH_FOLDS_CSV + '.tmp', TRACKLET_GRAPH_FOLDS_CSV)
        with open(
            TRACKLET_GRAPH_VIDEO_CSV + '.tmp', 'w', newline='', encoding='utf-8'
        ) as f:
            writer = csv.DictWriter(f, fieldnames=list(video_rows[0].keys()))
            writer.writeheader()
            writer.writerows(video_rows)
        os.replace(TRACKLET_GRAPH_VIDEO_CSV + '.tmp', TRACKLET_GRAPH_VIDEO_CSV)
        atomic_write_json(
            TRACKLET_GRAPH_PREDICTIONS_FILE, recommended_result['predictions']
        )
        summary = {
            'experiment': TRACKLET_GRAPH_EXPERIMENT_NAME,
            'status': 'experiment_only_not_auto_promoted',
            'purpose': (
                'Use full-video Viterbi association over top-K cached candidates and an '
                'explicit ABSENT state to rescue coherent weak candidates and suppress '
                'isolated strong false positives.'
            ),
            'selection_protocol': (
                'Leave one identity out with both _0/_1 videos held together. Each fold '
                'selects a single global configuration on the other identities.'
            ),
            'camera_compensation_scope': (
                'Optional cache-only robust median translation from mutual-nearest '
                'candidate centers. This is a proxy ablation, not optical flow or homography.'
            ),
            'overfitting_warning': (
                'Only three identities are available. Full-public score is descriptive; '
                'promotion additionally requires held-out identity stability and config votes.'
            ),
            'cache_signature': signature,
            'fixed_matching': {
                'weights': default_weights,
                'matching_threshold': MATCHING_THRESHOLD,
            },
            'fixed_temporal': active_temporal_config,
            'search_space': {
                'top_k': TRACKLET_GRAPH_TOP_K_GRID,
                'motion_weight': TRACKLET_GRAPH_MOTION_WEIGHT_GRID,
                'state_change_penalty': TRACKLET_GRAPH_STATE_CHANGE_PENALTY_GRID,
                'max_center_speed': TRACKLET_GRAPH_MAX_CENTER_SPEED_GRID,
                'camera_compensation': TRACKLET_GRAPH_CAMERA_COMPENSATION_GRID,
                'emission_scale': TRACKLET_GRAPH_EMISSION_SCALE,
                'config_count': len(rows),
            },
            'promotion_criteria': {
                'minimum_mean_cv_delta': TRACKLET_GRAPH_MIN_MEAN_CV_DELTA,
                'minimum_improved_identities': TRACKLET_GRAPH_MIN_IMPROVED_IDENTITIES,
                'maximum_worst_identity_drop': TRACKLET_GRAPH_MAX_WORST_IDENTITY_DROP,
                'minimum_identical_config_votes': TRACKLET_GRAPH_MIN_CONFIG_VOTES,
                'minimum_full_public_delta': TRACKLET_GRAPH_MIN_MEAN_CV_DELTA,
                'minimum_replay_fps': TRACKLET_GRAPH_MIN_REPLAY_FPS,
            },
            'production_control': {
                'mean_st_iou': default_result['mean_st_iou'],
                'detected_frames': default_result['detected_frames'],
                'per_video': default_result['per_video'],
                'predictions_untouched': diagnostics['production_control_untouched'],
            },
            'cross_validation': diagnostics,
            'fold_results': fold_rows,
            'recommended_full_public_result_descriptive_only': {
                'mean_st_iou': recommended_result['mean_st_iou'],
                'detected_frames': recommended_result['detected_frames'],
                'per_video': recommended_result['per_video'],
                'graph_diagnostics': recommended_result['graph_diagnostics'],
            },
            'top_full_public_results_descriptive_only': rows[:10],
            'outputs': {
                'all_results': TRACKLET_GRAPH_RESULTS_CSV,
                'identity_cv_folds': TRACKLET_GRAPH_FOLDS_CSV,
                'per_video': TRACKLET_GRAPH_VIDEO_CSV,
                'recommended_predictions': TRACKLET_GRAPH_PREDICTIONS_FILE,
            },
        }
        atomic_write_json(TRACKLET_GRAPH_SUMMARY_FILE, summary)
        print('\nCANDIDATE TRACKLET GRAPH V1 - IDENTITY CV')
        print(f"  recommended={diagnostics['recommended_config']}")
        print(
            f"  CV delta={diagnostics['mean_cv_delta']:+.6f} | "
            f"improved identities={diagnostics['improved_identity_count']}/"
            f"{len(fold_rows)} | worst delta={diagnostics['worst_identity_delta']:+.6f} | "
            f"votes={diagnostics['recommendation_votes']}"
        )
        print(
            f"  Full-public delta={diagnostics['recommended_full_public_delta']:+.6f} | "
            f"graph replay FPS={diagnostics['recommended_replay_fps']:.2f}"
        )
        print(f"  Promotion criteria passed: {diagnostics['promotion_passed']}")
        graph_diag = recommended_result['graph_diagnostics']
        print(
            f"  Weak frames rescued={graph_diag['rescued_below_threshold']} | "
            f"strong top-1 frames suppressed={graph_diag['suppressed_above_threshold']}"
        )
        print_evaluation(
            'TRACKLET GRAPH - DESCRIPTIVE FULL PUBLIC', recommended_result
        )
        print(f'Tracklet graph summary: {TRACKLET_GRAPH_SUMMARY_FILE}')
        print(f'All graph rows: {TRACKLET_GRAPH_RESULTS_CSV}')
        print(f'Identity folds: {TRACKLET_GRAPH_FOLDS_CSV}')
        print(f'Per-video diagnostics: {TRACKLET_GRAPH_VIDEO_CSV}')
        return

    if RUN_CHECKPOINT_SCORE_CALIBRATION:
        rows, fold_rows, diagnostics, recommended_results = (
            run_checkpoint_score_calibration(
                cache, signature, gt_map, default_result, active_temporal_config
            )
        )
        os.makedirs(SCORE_CAL_OUTPUT_DIR, exist_ok=True)
        with open(SCORE_CAL_RESULTS_CSV + '.tmp', 'w', newline='', encoding='utf-8') as f:
            writer = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
            writer.writeheader()
            writer.writerows(rows)
        os.replace(SCORE_CAL_RESULTS_CSV + '.tmp', SCORE_CAL_RESULTS_CSV)
        with open(SCORE_CAL_FOLDS_CSV + '.tmp', 'w', newline='', encoding='utf-8') as f:
            writer = csv.DictWriter(f, fieldnames=list(fold_rows[0].keys()))
            writer.writeheader()
            writer.writerows(fold_rows)
        os.replace(SCORE_CAL_FOLDS_CSV + '.tmp', SCORE_CAL_FOLDS_CSV)
        prediction_files = {}
        for variant_name, result in recommended_results.items():
            path = os.path.join(
                SCORE_CAL_OUTPUT_DIR, f'predictions_{variant_name}_recommended.json'
            )
            atomic_write_json(path, result['predictions'])
            prediction_files[variant_name] = path
        summary = {
            'experiment': SCORE_CAL_EXPERIMENT_NAME,
            'status': 'identity_cv_calibration_not_auto_promoted',
            'purpose': (
                'Calibrate checkpoint-specific global score transfer using cached '
                'production, P3, and P2 candidates with temporal processing locked.'
            ),
            'overfitting_control': (
                'Leave one identity out with both _0/_1 videos held together. '
                'No per-object parameter, temporal search, or candidate regeneration.'
            ),
            'score_transform': 'clip(yolo_conf,0,1) ** yolo_power',
            'fixed_temporal': active_temporal_config,
            'grid': {
                'yolo_powers': SCORE_CAL_YOLO_POWERS,
                'yolo_weights': SCORE_CAL_YOLO_WEIGHTS,
                'color_weights': SCORE_CAL_COLOR_WEIGHTS,
                'thresholds': SCORE_CAL_THRESHOLDS,
                'minimum_siamese_weight': SCORE_CAL_MIN_SIAMESE_WEIGHT,
            },
            'production_control': {
                'weights': default_weights,
                'matching_threshold': MATCHING_THRESHOLD,
                'mean_st_iou': default_result['mean_st_iou'],
                'per_video': default_result['per_video'],
            },
            'diagnostics': diagnostics,
            'outputs': {
                'all_grid_results': SCORE_CAL_RESULTS_CSV,
                'identity_cv_folds': SCORE_CAL_FOLDS_CSV,
                'recommended_predictions': prediction_files,
            },
        }
        atomic_write_json(SCORE_CAL_SUMMARY_FILE, summary)
        print('\nCHECKPOINT-SPECIFIC SCORE CALIBRATION V1 - IDENTITY CV')
        for variant_name in ('production', 'p3_control', 'p2_stride4'):
            item = diagnostics['variants'][variant_name]
            print(
                f"  {variant_name:12s}: CV={item['mean_cv_st_iou']:.6f} | "
                f"vs locked={item['mean_cv_delta_vs_locked']:+.6f} | "
                f"vs production={item['mean_cv_delta_vs_production']:+.6f} | "
                f"votes={item['recommendation_votes']} | "
                f"promote={item['promotion_passed']}"
            )
            print(f"    recommended={item['recommended_config']}")
        print(
            f"  Recommended variant={diagnostics['recommended_variant']} | "
            f"promotion passed={diagnostics['promotion_passed']}"
        )
        print(f'Score calibration summary: {SCORE_CAL_SUMMARY_FILE}')
        print(f'Grid results: {SCORE_CAL_RESULTS_CSV}')
        print(f'Identity folds: {SCORE_CAL_FOLDS_CSV}')
        return

    if RUN_P2_DETECTOR_AB:
        result_rows, video_rows, diagnostics, variants = run_detector_checkpoint_ab(
            cache, signature, gt_map, default_result, active_temporal_config
        )
        os.makedirs(P2_AB_OUTPUT_DIR, exist_ok=True)
        with open(P2_AB_RESULTS_CSV + '.tmp', 'w', newline='', encoding='utf-8') as f:
            writer = csv.DictWriter(f, fieldnames=list(result_rows[0].keys()))
            writer.writeheader()
            writer.writerows(result_rows)
        os.replace(P2_AB_RESULTS_CSV + '.tmp', P2_AB_RESULTS_CSV)
        with open(P2_AB_VIDEO_CSV + '.tmp', 'w', newline='', encoding='utf-8') as f:
            writer = csv.DictWriter(f, fieldnames=list(video_rows[0].keys()))
            writer.writeheader()
            writer.writerows(video_rows)
        os.replace(P2_AB_VIDEO_CSV + '.tmp', P2_AB_VIDEO_CSV)
        prediction_files = {}
        for variant_name in ('p3_control', 'p2_stride4'):
            prediction_path = os.path.join(
                P2_AB_OUTPUT_DIR, f'predictions_{variant_name}.json'
            )
            atomic_write_json(prediction_path, variants[variant_name]['result']['predictions'])
            prediction_files[variant_name] = prediction_path
        summary = {
            'experiment': P2_AB_EXPERIMENT_NAME,
            'status': 'locked_evaluation_not_auto_promoted',
            'purpose': (
                'Compare the production detector, paired retrained P3 control, and P2/4 '
                'detector with identical downstream matching and temporal processing.'
            ),
            'overfitting_control': (
                'Three checkpoints and promotion gates were fixed before evaluation. '
                'No per-object rule, fusion calibration, threshold search, or temporal tuning.'
            ),
            'fixed_matching': {
                'weights': default_weights,
                'matching_threshold': MATCHING_THRESHOLD,
                'confidence_threshold': CONFIDENCE_THRESHOLD,
                'imgsz': P2_AB_IMGSZ,
                'augment_tta': USE_TTA,
            },
            'fixed_temporal': active_temporal_config,
            'production_control': {
                'checkpoint_sha256': file_sha256(YOLO_MODEL_PATH),
                'mean_st_iou': default_result['mean_st_iou'],
                'detected_frames': default_result['detected_frames'],
                'per_video': default_result['per_video'],
            },
            'diagnostics': diagnostics,
            'results': result_rows,
            'outputs': {
                'variant_results': P2_AB_RESULTS_CSV,
                'per_video_results': P2_AB_VIDEO_CSV,
                'predictions': prediction_files,
            },
        }
        atomic_write_json(P2_AB_SUMMARY_FILE, summary)
        print('\nYOLO P2 DETECTOR LOCKED CHECKPOINT A/B V1')
        for row in result_rows:
            print(
                f"  {row['variant']:12s}: ST-IoU={row['mean_st_iou']:.6f} | "
                f"delta={row['delta_st_iou_vs_legacy']:+.6f} | "
                f"R@0.5={row['oracle_recall_iou_05']:.4f} | "
                f"sub16={row['sub16_oracle_recall_iou_05']:.4f} | "
                f"FPS={row['aggregate_fps']}"
            )
        print(
            f"  Recommended={diagnostics['recommended_variant']} | "
            f"promotion passed={diagnostics['promotion_passed']}"
        )
        causal = diagnostics['p2_vs_paired_p3']
        print(
            f"  P2 vs paired P3: ST-IoU={causal['mean_st_iou_delta']:+.6f} | "
            f"sub16={causal['sub16_recall_iou_05_delta']:+.6f}"
        )
        print(f'P2 detector A/B summary: {P2_AB_SUMMARY_FILE}')
        print(f'Variant results: {P2_AB_RESULTS_CSV}')
        print(f'Per-video results: {P2_AB_VIDEO_CSV}')
        return

    if RUN_SAHI_ADMISSION:
        tile_signature = build_sahi_tile_cache_signature(signature)
        tile_cache = load_or_initialize_sahi_tile_cache(tile_signature)
        tile_cache = extract_sahi_tile_cache(tile_cache, video_folders)
        result_rows, video_rows, fold_rows, diagnostics, policy_results = (
            run_sahi_admission_experiment(
                cache, tile_cache, gt_map, default_result, active_temporal_config
            )
        )
        os.makedirs(SAHI_ADMISSION_OUTPUT_DIR, exist_ok=True)
        with open(SAHI_ADMISSION_RESULTS_CSV + '.tmp', 'w', newline='', encoding='utf-8') as f:
            writer = csv.DictWriter(f, fieldnames=list(result_rows[0].keys()))
            writer.writeheader()
            writer.writerows(result_rows)
        os.replace(SAHI_ADMISSION_RESULTS_CSV + '.tmp', SAHI_ADMISSION_RESULTS_CSV)
        with open(SAHI_ADMISSION_VIDEO_CSV + '.tmp', 'w', newline='', encoding='utf-8') as f:
            writer = csv.DictWriter(f, fieldnames=list(video_rows[0].keys()))
            writer.writeheader()
            writer.writerows(video_rows)
        os.replace(SAHI_ADMISSION_VIDEO_CSV + '.tmp', SAHI_ADMISSION_VIDEO_CSV)
        with open(SAHI_ADMISSION_FOLDS_CSV + '.tmp', 'w', newline='', encoding='utf-8') as f:
            writer = csv.DictWriter(f, fieldnames=list(fold_rows[0].keys()))
            writer.writeheader()
            writer.writerows(fold_rows)
        os.replace(SAHI_ADMISSION_FOLDS_CSV + '.tmp', SAHI_ADMISSION_FOLDS_CSV)
        prediction_files = {}
        for policy in SAHI_ADMISSION_POLICIES:
            prediction_path = os.path.join(
                SAHI_ADMISSION_OUTPUT_DIR, f'predictions_{policy}.json'
            )
            atomic_write_json(prediction_path, policy_results[policy]['predictions'])
            prediction_files[policy] = prediction_path
        summary = {
            'experiment': SAHI_ADMISSION_EXPERIMENT_NAME,
            'status': 'experiment_only_not_auto_promoted',
            'purpose': (
                'Replay two fixed source-admission policies that preserve strong legacy '
                'decisions while using the precomputed SAHI tile cache for rescue.'
            ),
            'selection_protocol': (
                'Leave one identity out. Each fold selects legacy, missing-only, or '
                'weak-legacy rescue using only the other two identities.'
            ),
            'overfitting_control': (
                'Policies and the 0.54 strong-legacy threshold are fixed before evaluation. '
                'No object-specific threshold, penalty, or detector inference is tuned.'
            ),
            'fixed_matching': {
                'weights': default_weights,
                'matching_threshold': MATCHING_THRESHOLD,
            },
            'fixed_temporal': active_temporal_config,
            'tile_cache_file': SAHI_TILE_CACHE_FILE,
            'policies': {
                'legacy_only': 'production control',
                'missing_legacy_only': 'admit tiles only when no legacy candidate exists',
                'weak_legacy_rescue': (
                    'admit tiles when no legacy candidate exists or best legacy fused '
                    'score is below the locked production threshold 0.54'
                ),
            },
            'production_control': {
                'mean_st_iou': default_result['mean_st_iou'],
                'detected_frames': default_result['detected_frames'],
                'per_video': default_result['per_video'],
            },
            'diagnostics': diagnostics,
            'results': result_rows,
            'folds': fold_rows,
            'outputs': {
                'policy_results': SAHI_ADMISSION_RESULTS_CSV,
                'per_video_results': SAHI_ADMISSION_VIDEO_CSV,
                'identity_folds': SAHI_ADMISSION_FOLDS_CSV,
                'predictions': prediction_files,
            },
        }
        atomic_write_json(SAHI_ADMISSION_SUMMARY_FILE, summary)
        print('\nSAHI SOURCE ADMISSION V1')
        for row in result_rows:
            print(
                f"  {row['policy']:22s}: ST-IoU={row['mean_st_iou']:.6f} | "
                f"delta={row['delta_st_iou_vs_legacy']:+.6f} | "
                f"tile accepted={row['accepted_tile_frames']} | "
                f"frames={row['detected_frames']}"
            )
        cv = diagnostics['cross_validation']
        print(
            f"  Recommended descriptive policy: "
            f"{cv['recommended_policy_descriptive_full_public']}"
        )
        print(
            f"  Identity-CV promotion passed: {cv['promotion_passed']} | "
            f"CV delta={cv['mean_cv_delta']:+.6f} | "
            f"improved held-out identities={cv['improved_heldout_identities']}/3 | "
            f"worst held-out delta={cv['worst_heldout_delta']:+.6f} | "
            f"policy counts={cv['selected_policy_counts']}"
        )
        print(f'SAHI admission summary: {SAHI_ADMISSION_SUMMARY_FILE}')
        print(f'Policy results: {SAHI_ADMISSION_RESULTS_CSV}')
        print(f'Per-video results: {SAHI_ADMISSION_VIDEO_CSV}')
        print(f'Identity folds: {SAHI_ADMISSION_FOLDS_CSV}')
        return

    if RUN_HYBRID_SAHI:
        result_rows, video_rows, diagnostics, hybrid_result = run_hybrid_sahi_experiment(
            cache, signature, gt_map, default_result, active_temporal_config
        )
        os.makedirs(SAHI_OUTPUT_DIR, exist_ok=True)
        with open(SAHI_RESULTS_CSV + '.tmp', 'w', newline='', encoding='utf-8') as f:
            writer = csv.DictWriter(f, fieldnames=list(result_rows[0].keys()))
            writer.writeheader()
            writer.writerows(result_rows)
        os.replace(SAHI_RESULTS_CSV + '.tmp', SAHI_RESULTS_CSV)
        with open(SAHI_VIDEO_CSV + '.tmp', 'w', newline='', encoding='utf-8') as f:
            writer = csv.DictWriter(f, fieldnames=list(video_rows[0].keys()))
            writer.writeheader()
            writer.writerows(video_rows)
        os.replace(SAHI_VIDEO_CSV + '.tmp', SAHI_VIDEO_CSV)
        atomic_write_json(SAHI_PREDICTIONS_FILE, hybrid_result['predictions'])
        summary = {
            'experiment': SAHI_EXPERIMENT_NAME,
            'status': 'experiment_only_not_auto_promoted',
            'purpose': (
                'Preserve every legacy full-frame candidate and add native-scale tiled '
                'candidates to test small-object recall without replacing the detector output.'
            ),
            'overfitting_control': (
                'One global tile geometry is used for all videos. GT is evaluated only '
                'after the immutable union; no object-specific rule or score calibration is used.'
            ),
            'fixed_matching': {
                'weights': default_weights,
                'matching_threshold': MATCHING_THRESHOLD,
            },
            'fixed_temporal': active_temporal_config,
            'tile_config': {
                'tile_size': SAHI_TILE_SIZE,
                'tile_imgsz': SAHI_TILE_IMGSZ,
                'requested_overlap_ratio': SAHI_OVERLAP_RATIO,
                'ownership': 'overlap midpoint by candidate center',
                'legacy_candidates_preserved': True,
                'tile_vs_legacy_suppression': False,
            },
            'production_control': {
                'mean_st_iou': default_result['mean_st_iou'],
                'detected_frames': default_result['detected_frames'],
                'per_video': default_result['per_video'],
            },
            'diagnostics': diagnostics,
            'results': result_rows,
            'per_video': video_rows,
            'hybrid_result': {
                'mean_st_iou': hybrid_result['mean_st_iou'],
                'detected_frames': hybrid_result['detected_frames'],
                'per_video': hybrid_result['per_video'],
            },
            'outputs': {
                'tile_cache': SAHI_TILE_CACHE_FILE,
                'variant_results': SAHI_RESULTS_CSV,
                'per_video_results': SAHI_VIDEO_CSV,
                'hybrid_predictions': SAHI_PREDICTIONS_FILE,
            },
        }
        atomic_write_json(SAHI_SUMMARY_FILE, summary)
        print('\nHYBRID FULL-FRAME + SAHI TILES V1')
        for row in result_rows:
            print(
                f"  {row['variant']:12s}: ST-IoU={row['mean_st_iou']:.6f} | "
                f"R@0.5={row['oracle_recall_iou_05']:.4f} | "
                f"sub16={row['sub16_oracle_recall_iou_05']:.4f} | "
                f"candidates={row['total_candidate_count']}"
            )
        promotion = diagnostics['promotion']
        runtime = diagnostics['tile_runtime']
        print(
            f"  Oracle monotonic: {diagnostics['oracle_monotonic_per_gt_frame']} | "
            f"improved GT frames={diagnostics['improved_oracle_gt_frames']}"
        )
        print(
            f"  Promotion passed: {promotion['passed']} | "
            f"ST-IoU delta={promotion['mean_st_iou_delta']:+.6f} | "
            f"sub16 delta={promotion['sub16_recall_iou_05_delta']:+.6f} | "
            f"improved identities={promotion['improved_identity_count']}/3 | "
            f"worst identity delta={promotion['worst_identity_delta']:+.6f}"
        )
        print(
            f"  Tile FPS={runtime.get('tile_branch_fps')} | "
            f"estimated sequential hybrid FPS={runtime.get('estimated_sequential_hybrid_fps')} | "
            f"estimated T4x2 parallel FPS={runtime.get('estimated_parallel_two_gpu_fps')}"
        )
        print(f'Hybrid SAHI summary: {SAHI_SUMMARY_FILE}')
        print(f'Variant results: {SAHI_RESULTS_CSV}')
        print(f'Per-video results: {SAHI_VIDEO_CSV}')
        return

    if RUN_RESOLUTION_AB:
        result_rows, video_rows, diagnostics, variants = run_resolution_candidate_ab(
            cache, signature, gt_map, default_result, active_temporal_config
        )
        os.makedirs(RESOLUTION_OUTPUT_DIR, exist_ok=True)
        with open(RESOLUTION_RESULTS_CSV + '.tmp', 'w', newline='', encoding='utf-8') as f:
            writer = csv.DictWriter(f, fieldnames=list(result_rows[0].keys()))
            writer.writeheader()
            writer.writerows(result_rows)
        os.replace(RESOLUTION_RESULTS_CSV + '.tmp', RESOLUTION_RESULTS_CSV)
        with open(RESOLUTION_VIDEO_CSV + '.tmp', 'w', newline='', encoding='utf-8') as f:
            writer = csv.DictWriter(f, fieldnames=list(video_rows[0].keys()))
            writer.writeheader()
            writer.writerows(video_rows)
        os.replace(RESOLUTION_VIDEO_CSV + '.tmp', RESOLUTION_VIDEO_CSV)
        prediction_files = {}
        for variant_name in ('explicit_640', 'explicit_960'):
            prediction_path = os.path.join(
                RESOLUTION_OUTPUT_DIR, f'predictions_{variant_name}.json'
            )
            atomic_write_json(prediction_path, variants[variant_name]['result']['predictions'])
            prediction_files[variant_name] = prediction_path
        summary = {
            'experiment': RESOLUTION_EXPERIMENT_NAME,
            'status': 'experiment_only_not_auto_promoted',
            'purpose': (
                'Compare legacy implicit resolution with explicit full-frame 640 and 960 '
                'candidate generation while freezing matching and temporal behavior.'
            ),
            'overfitting_control': (
                'One global imgsz is applied to all videos. Explicit 640 must reproduce '
                'legacy before 960 is eligible; no per-object rule or calibration is used.'
            ),
            'fixed_matching': {
                'weights': default_weights,
                'matching_threshold': MATCHING_THRESHOLD,
            },
            'fixed_temporal': active_temporal_config,
            'search_space': {
                'legacy_implicit': True,
                'explicit_imgsz': RESOLUTION_VARIANTS,
                'confidence_threshold': CONFIDENCE_THRESHOLD,
                'augment_tta': USE_TTA,
            },
            'production_control': {
                'mean_st_iou': default_result['mean_st_iou'],
                'detected_frames': default_result['detected_frames'],
                'per_video': default_result['per_video'],
            },
            'diagnostics': diagnostics,
            'results': result_rows,
            'outputs': {
                'variant_results': RESOLUTION_RESULTS_CSV,
                'per_video_results': RESOLUTION_VIDEO_CSV,
                'predictions': prediction_files,
            },
        }
        atomic_write_json(RESOLUTION_SUMMARY_FILE, summary)
        print('\nEXPLICIT RESOLUTION CANDIDATE A/B V1')
        for row in result_rows:
            print(
                f"  {row['variant']:14s}: ST-IoU={row['mean_st_iou']:.6f} | "
                f"R@0.5={row['oracle_recall_iou_05']:.4f} | "
                f"sub16={row['sub16_oracle_recall_iou_05']:.4f} | "
                f"FPS={row['aggregate_fps']}"
            )
        control = diagnostics['control']
        promotion = diagnostics['promotion']
        print(
            f"  Explicit-640 control passed: {control['passed']} | "
            f"mean abs delta={control['mean_st_iou_abs_delta']:.8f}"
        )
        print(
            f"  Explicit-960 promotion passed: {promotion['passed']} | "
            f"ST-IoU delta={promotion['mean_st_iou_delta']:+.6f} | "
            f"sub16 recall delta={promotion['sub16_recall_iou_05_delta']:+.6f} | "
            f"improved identities={promotion['improved_identity_count']}/3 | "
            f"worst identity delta={promotion['worst_identity_delta']:+.6f}"
        )
        print(f'Resolution A/B summary: {RESOLUTION_SUMMARY_FILE}')
        print(f'Variant results: {RESOLUTION_RESULTS_CSV}')
        print(f'Per-video results: {RESOLUTION_VIDEO_CSV}')
        return

    if RUN_DETECTOR_PROFILE:
        frame_rows, video_rows, diagnostics = run_detector_profile(
            cache, gt_map, default_result
        )
        os.makedirs(DETECTOR_PROFILE_OUTPUT_DIR, exist_ok=True)
        with open(DETECTOR_PROFILE_FRAME_CSV + '.tmp', 'w', newline='', encoding='utf-8') as f:
            writer = csv.DictWriter(f, fieldnames=list(frame_rows[0].keys()))
            writer.writeheader()
            writer.writerows(frame_rows)
        os.replace(DETECTOR_PROFILE_FRAME_CSV + '.tmp', DETECTOR_PROFILE_FRAME_CSV)
        with open(DETECTOR_PROFILE_VIDEO_CSV + '.tmp', 'w', newline='', encoding='utf-8') as f:
            writer = csv.DictWriter(f, fieldnames=list(video_rows[0].keys()))
            writer.writeheader()
            writer.writerows(video_rows)
        os.replace(DETECTOR_PROFILE_VIDEO_CSV + '.tmp', DETECTOR_PROFILE_VIDEO_CSV)
        summary = {
            'experiment': DETECTOR_PROFILE_EXPERIMENT_NAME,
            'status': 'diagnostic_only_no_inference_behavior_change',
            'purpose': (
                'Profile detector architecture, target scale, and cached candidate recall '
                'before testing explicit imgsz, SAHI, or a P2/P3 detector.'
            ),
            'overfitting_control': (
                'No per-object setting is selected. GT is read only after production '
                'predictions are frozen; profiling never changes candidates or output.'
            ),
            'cache_signature': signature,
            'size_definition': {
                'primary_measure': (
                    'minimum GT bbox side projected with isotropic letterbox scale to '
                    f'{DETECTOR_PROFILE_REFERENCE_IMGSZ}'
                ),
                'bins': DETECTOR_PROFILE_SIZE_BINS,
                'iou_thresholds': DETECTOR_PROFILE_IOU_THRESHOLDS,
            },
            'production_control': {
                'mean_st_iou': default_result['mean_st_iou'],
                'detected_frames': default_result['detected_frames'],
                'per_video': default_result['per_video'],
            },
            'diagnostics': diagnostics,
            'per_video': video_rows,
            'outputs': {
                'gt_frame_profile': DETECTOR_PROFILE_FRAME_CSV,
                'video_profile': DETECTOR_PROFILE_VIDEO_CSV,
            },
        }
        atomic_write_json(DETECTOR_PROFILE_SUMMARY_FILE, summary)
        print('\nDETECTOR PROFILE V1')
        architecture = diagnostics['detector_architecture']
        print(f"  Detection strides: {architecture['detection_strides']}")
        print(
            f"  P2<=4: {architecture['has_p2_stride_4_or_finer']} | "
            f"P3<=8: {architecture['has_p3_stride_8_or_finer']}"
        )
        for size_name, report in diagnostics['by_size_bin'].items():
            print(
                f"  {size_name:20s}: GT={report['gt_frames']} | "
                f"R@0.5={report['oracle_recall_iou_05']:.4f} | "
                f"no-candidate={report['no_candidate_frames']}"
            )
        print(f'Detector profile summary: {DETECTOR_PROFILE_SUMMARY_FILE}')
        print(f'Per-video profile: {DETECTOR_PROFILE_VIDEO_CSV}')
        print(f'Per-frame profile: {DETECTOR_PROFILE_FRAME_CSV}')
        return

    if RUN_MARGIN_DIAGNOSTICS:
        frame_rows, video_rows, diagnostics = run_margin_diagnostics(
            cache, gt_map, default_weights, MATCHING_THRESHOLD, default_result
        )
        os.makedirs(MARGIN_OUTPUT_DIR, exist_ok=True)
        with open(MARGIN_FRAME_CSV + '.tmp', 'w', newline='', encoding='utf-8') as f:
            writer = csv.DictWriter(f, fieldnames=list(frame_rows[0].keys()))
            writer.writeheader()
            writer.writerows(frame_rows)
        os.replace(MARGIN_FRAME_CSV + '.tmp', MARGIN_FRAME_CSV)
        with open(MARGIN_VIDEO_CSV + '.tmp', 'w', newline='', encoding='utf-8') as f:
            writer = csv.DictWriter(f, fieldnames=list(video_rows[0].keys()))
            writer.writeheader()
            writer.writerows(video_rows)
        os.replace(MARGIN_VIDEO_CSV + '.tmp', MARGIN_VIDEO_CSV)
        summary = {
            'experiment': MARGIN_EXPERIMENT_NAME,
            'status': 'diagnostic_only_no_inference_behavior_change',
            'purpose': (
                'Measure whether top1-top2 margin and YOLO/Siamese/Color components '
                'separate correct weak candidates from background across held-out identities.'
            ),
            'overfitting_control': (
                'GT labels are used after production predictions are frozen. No feature '
                'threshold is applied to inference in this run.'
            ),
            'cache_signature': signature,
            'fixed_matching': {
                'weights': default_weights, 'matching_threshold': MATCHING_THRESHOLD,
            },
            'fixed_temporal': PRODUCTION_TEMPORAL_CONFIG,
            'features': MARGIN_FEATURES,
            'weak_score_threshold': MARGIN_WEAK_SCORE_THRESHOLD,
            'production_control': {
                'mean_st_iou': default_result['mean_st_iou'],
                'detected_frames': default_result['detected_frames'],
                'per_video': default_result['per_video'],
            },
            'diagnostics': diagnostics,
            'per_video': video_rows,
            'outputs': {
                'frame_features': MARGIN_FRAME_CSV,
                'video_feature_metrics': MARGIN_VIDEO_CSV,
            },
        }
        atomic_write_json(MARGIN_SUMMARY_FILE, summary)
        print('\nCANDIDATE MARGIN & COMPONENT DIAGNOSTICS')
        for feature in MARGIN_FEATURES:
            report = diagnostics['feature_reports'][feature]['weak_region']
            print(
                f"  {feature:28s}: AUC={report['roc_auc']} | "
                f"AP={report['average_precision']} | rows={report['rows']}"
            )
        print(f'Margin summary: {MARGIN_SUMMARY_FILE}')
        print(f'Per-video metrics: {MARGIN_VIDEO_CSV}')
        print(f'Per-frame features: {MARGIN_FRAME_CSV}')
        return

    if RUN_HYSTERESIS_EXPERIMENT:
        rows, fold_rows, diagnostics, recommended_result = run_hysteresis_identity_cv(
            cache, gt_map, default_weights, default_result
        )
        os.makedirs(HYSTERESIS_OUTPUT_DIR, exist_ok=True)
        with open(HYSTERESIS_RESULTS_CSV + '.tmp', 'w', newline='', encoding='utf-8') as f:
            writer = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
            writer.writeheader()
            writer.writerows(rows)
        os.replace(HYSTERESIS_RESULTS_CSV + '.tmp', HYSTERESIS_RESULTS_CSV)
        with open(HYSTERESIS_FOLDS_CSV + '.tmp', 'w', newline='', encoding='utf-8') as f:
            writer = csv.DictWriter(f, fieldnames=list(fold_rows[0].keys()))
            writer.writeheader()
            writer.writerows(fold_rows)
        os.replace(HYSTERESIS_FOLDS_CSV + '.tmp', HYSTERESIS_FOLDS_CSV)
        atomic_write_json(HYSTERESIS_PREDICTIONS_FILE, recommended_result['predictions'])
        summary = {
            'experiment': HYSTERESIS_EXPERIMENT_NAME,
            'status': 'experiment_only_not_auto_promoted',
            'selection_protocol': (
                'Leave one identity out; both _0/_1 videos are held together. '
                'Each fold selects only on the other identities.'
            ),
            'inference_rule': (
                'A weak top-1 candidate is accepted only when bracketed by strong anchors '
                'within max_anchor_span and all past-weak, weak-future, and anchor-anchor '
                'center-speed checks pass.'
            ),
            'overfitting_warning': (
                'Only three identities are available. Held-out identity CV reduces direct '
                'video leakage but remains a small public-GT selection benchmark.'
            ),
            'cache_signature': signature,
            'fixed_matching': {
                'weights': default_weights, 'strong_threshold': HYSTERESIS_STRONG_THRESHOLD,
            },
            'fixed_temporal': PRODUCTION_TEMPORAL_CONFIG,
            'search_space': {
                'weak_threshold': HYSTERESIS_WEAK_THRESHOLD_GRID,
                'max_anchor_span': HYSTERESIS_ANCHOR_SPAN_GRID,
                'max_center_speed': HYSTERESIS_MAX_CENTER_SPEED,
                'config_count': len(rows),
            },
            'promotion_criteria': {
                'minimum_mean_cv_delta': HYSTERESIS_MIN_MEAN_CV_DELTA,
                'minimum_improved_identities': HYSTERESIS_MIN_IMPROVED_IDENTITIES,
                'maximum_worst_identity_drop': HYSTERESIS_MAX_WORST_IDENTITY_DROP,
            },
            'production_control': {
                'mean_st_iou': default_result['mean_st_iou'],
                'detected_frames': default_result['detected_frames'],
                'per_video': default_result['per_video'],
            },
            'cross_validation': diagnostics,
            'fold_results': fold_rows,
            'recommended_full_public_result_descriptive_only': {
                'mean_st_iou': recommended_result['mean_st_iou'],
                'detected_frames': recommended_result['detected_frames'],
                'rescued_frames': recommended_result['rescued_frames'],
                'rescued_per_video': recommended_result['rescued_per_video'],
                'per_video': recommended_result['per_video'],
            },
            'top_full_public_results_descriptive_only': rows[:10],
            'outputs': {
                'all_results': HYSTERESIS_RESULTS_CSV,
                'fold_results': HYSTERESIS_FOLDS_CSV,
                'recommended_predictions': HYSTERESIS_PREDICTIONS_FILE,
            },
        }
        atomic_write_json(HYSTERESIS_SUMMARY_FILE, summary)
        print('\nIDENTITY-HELD-OUT THRESHOLD HYSTERESIS RESULT')
        print(f"  recommended={diagnostics['recommended_config']}")
        print(
            f"  CV delta={diagnostics['mean_cv_delta']:+.6f} | "
            f"improved identities={diagnostics['improved_identity_count']}/"
            f"{len(fold_rows)} | worst delta={diagnostics['worst_identity_delta']:+.6f}"
        )
        print(f"  Promotion criteria passed: {diagnostics['promotion_passed']}")
        print(f"  Weak frames rescued before temporal: {recommended_result['rescued_frames']}")
        print_evaluation('RECOMMENDED HYSTERESIS - DESCRIPTIVE FULL PUBLIC', recommended_result)
        print(f'Hysteresis summary: {HYSTERESIS_SUMMARY_FILE}')
        print(f'All hysteresis rows: {HYSTERESIS_RESULTS_CSV}')
        print(f'Identity folds: {HYSTERESIS_FOLDS_CSV}')
        return

    if RUN_ORACLE_DIAGNOSTICS:
        frame_rows, video_rows, diagnostics = run_oracle_diagnostics(
            cache, gt_map, default_weights, MATCHING_THRESHOLD, default_result
        )
        os.makedirs(ORACLE_OUTPUT_DIR, exist_ok=True)
        with open(ORACLE_FRAME_CSV + '.tmp', 'w', newline='', encoding='utf-8') as f:
            writer = csv.DictWriter(f, fieldnames=list(frame_rows[0].keys()))
            writer.writeheader()
            writer.writerows(frame_rows)
        os.replace(ORACLE_FRAME_CSV + '.tmp', ORACLE_FRAME_CSV)
        with open(ORACLE_VIDEO_CSV + '.tmp', 'w', newline='', encoding='utf-8') as f:
            writer = csv.DictWriter(f, fieldnames=list(video_rows[0].keys()))
            writer.writeheader()
            writer.writerows(video_rows)
        os.replace(ORACLE_VIDEO_CSV + '.tmp', ORACLE_VIDEO_CSV)
        summary = {
            'experiment': ORACLE_EXPERIMENT_NAME,
            'status': 'diagnostic_only_no_inference_behavior_change',
            'purpose': (
                'Decompose candidate recall, matching, threshold, and temporal errors '
                'before selecting the next algorithmic intervention.'
            ),
            'overfitting_control': (
                'GT is used only after production predictions are frozen. Diagnostics do '
                'not rerank, threshold, interpolate, or modify prediction JSON.'
            ),
            'cache_signature': signature,
            'fixed_matching': {
                'weights': default_weights, 'matching_threshold': MATCHING_THRESHOLD,
            },
            'fixed_temporal': PRODUCTION_TEMPORAL_CONFIG,
            'iou_thresholds': ORACLE_IOU_THRESHOLDS,
            'primary_failure_iou': ORACLE_GOOD_IOU,
            'first_failure_hierarchy': ORACLE_GT_FAILURE_CATEGORIES,
            'production_control': {
                'mean_st_iou': default_result['mean_st_iou'],
                'detected_frames': default_result['detected_frames'],
                'per_video': default_result['per_video'],
            },
            'diagnostics': diagnostics,
            'per_video': video_rows,
            'outputs': {
                'frame_diagnostics': ORACLE_FRAME_CSV,
                'video_diagnostics': ORACLE_VIDEO_CSV,
            },
        }
        atomic_write_json(ORACLE_SUMMARY_FILE, summary)
        print('\nCANDIDATE-CACHE ORACLE DIAGNOSTICS')
        for row in video_rows:
            print(
                f"  {row['video_id']:30s}: oracle={row['oracle_st_iou_upper_bound']:.4f} | "
                f"selected={row['selected_prethreshold_st_iou']:.4f} | "
                f"accepted={row['accepted_pretemporal_st_iou']:.4f} | "
                f"final={row['final_production_st_iou']:.4f}"
            )
        global_diag = diagnostics['global']
        print(f"  Dominant non-success failure: {global_diag['dominant_non_success_failure']}")
        print(f"  Failure counts: {global_diag['failure_counts']}")
        print(f'Oracle summary: {ORACLE_SUMMARY_FILE}')
        print(f'Per-video diagnostics: {ORACLE_VIDEO_CSV}')
        print(f'Per-frame diagnostics: {ORACLE_FRAME_CSV}')
        return

    if RUN_RERANKING_EXPERIMENT:
        rows, fold_rows, diagnostics, recommended_result = run_reranking_identity_cv(
            cache, gt_map, default_weights, MATCHING_THRESHOLD, default_result
        )
        os.makedirs(RERANK_OUTPUT_DIR, exist_ok=True)
        with open(RERANK_RESULTS_CSV + '.tmp', 'w', newline='', encoding='utf-8') as f:
            writer = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
            writer.writeheader()
            writer.writerows(rows)
        os.replace(RERANK_RESULTS_CSV + '.tmp', RERANK_RESULTS_CSV)
        with open(RERANK_FOLDS_CSV + '.tmp', 'w', newline='', encoding='utf-8') as f:
            writer = csv.DictWriter(f, fieldnames=list(fold_rows[0].keys()))
            writer.writeheader()
            writer.writerows(fold_rows)
        os.replace(RERANK_FOLDS_CSV + '.tmp', RERANK_FOLDS_CSV)
        atomic_write_json(RERANK_PREDICTIONS_FILE, recommended_result['predictions'])
        summary = {
            'experiment': RERANK_EXPERIMENT_NAME,
            'status': 'experiment_only_not_auto_promoted',
            'selection_protocol': (
                'Leave one identity out; both _0/_1 videos of an identity are held together. '
                'Each fold selects only on the other identities.'
            ),
            'overfitting_warning': (
                'Only three identities are available. Identity-held-out evaluation reduces '
                'video leakage but is not an independent test-set estimate.'
            ),
            'cache_signature': signature,
            'fixed_matching': {
                'weights': default_weights, 'matching_threshold': MATCHING_THRESHOLD,
            },
            'fixed_temporal': PRODUCTION_TEMPORAL_CONFIG,
            'search_space': {
                'top_k': RERANK_TOP_K_GRID,
                'neighbor_radius': RERANK_NEIGHBOR_RADIUS_GRID,
                'rerank_weight': RERANK_WEIGHT_GRID,
                'motion_scale': RERANK_MOTION_SCALE,
                'config_count': len(rows),
            },
            'promotion_criteria': {
                'minimum_mean_cv_delta': RERANK_MIN_MEAN_CV_DELTA,
                'minimum_improved_identities': RERANK_MIN_IMPROVED_IDENTITIES,
                'maximum_worst_identity_drop': RERANK_MAX_WORST_IDENTITY_DROP,
            },
            'production_control': {
                'mean_st_iou': default_result['mean_st_iou'],
                'detected_frames': default_result['detected_frames'],
                'per_video': default_result['per_video'],
            },
            'cross_validation': diagnostics,
            'fold_results': fold_rows,
            'recommended_full_public_result_descriptive_only': {
                'mean_st_iou': recommended_result['mean_st_iou'],
                'detected_frames': recommended_result['detected_frames'],
                'per_video': recommended_result['per_video'],
            },
            'top_full_public_results_descriptive_only': rows[:10],
            'outputs': {
                'all_results': RERANK_RESULTS_CSV,
                'fold_results': RERANK_FOLDS_CSV,
                'recommended_predictions': RERANK_PREDICTIONS_FILE,
            },
        }
        atomic_write_json(RERANK_SUMMARY_FILE, summary)
        print('\nIDENTITY-HELD-OUT RERANKING RESULT')
        print(f"  recommended={diagnostics['recommended_config']}")
        print(
            f"  CV delta={diagnostics['mean_cv_delta']:+.6f} | "
            f"improved identities={diagnostics['improved_identity_count']}/"
            f"{len(fold_rows)} | worst delta={diagnostics['worst_identity_delta']:+.6f}"
        )
        print(f"  Promotion criteria passed: {diagnostics['promotion_passed']}")
        print_evaluation('RECOMMENDED CONFIG - DESCRIPTIVE FULL PUBLIC', recommended_result)
        print(f'Reranking summary: {RERANK_SUMMARY_FILE}')
        print(f'All reranking rows: {RERANK_RESULTS_CSV}')
        print(f'Identity folds: {RERANK_FOLDS_CSV}')
        return

    if RUN_TEMPORAL_CALIBRATION:
        rows, best_config, best_result = run_temporal_calibration(
            cache, gt_map, default_weights, MATCHING_THRESHOLD
        )
        print('\nBEST TEMPORAL CONFIG')
        print(f'  {best_config}')
        print_evaluation('TEMPORAL CALIBRATED', best_result)
        summary = {
            'experiment': TEMPORAL_EXPERIMENT_NAME,
            'selection_objective': 'maximum mean ST-IoU on all supplied manual GT videos',
            'overfitting_warning': (
                'The same six manually annotated videos select and report this configuration; '
                'this is an offline optimum, not an unbiased generalization estimate.'
            ),
            'cache_signature': signature,
            'candidate_cache_file': CANDIDATE_CACHE_FILE,
            'fixed_matching': {
                'weights': default_weights,
                'matching_threshold': MATCHING_THRESHOLD,
            },
            'motion_definition': {
                'center_speed': 'center displacement / (frame steps * mean sqrt bbox area)',
                'log_scale_speed': 'abs(log(scale2 / scale1)) / frame steps',
                'scale': 'sqrt(bbox width * bbox height)',
            },
            'search_space': {
                'max_gap': TEMPORAL_MAX_GAP_GRID,
                'min_seg_len': TEMPORAL_MIN_SEG_GRID,
                'max_center_speed': TEMPORAL_CENTER_SPEED_GRID,
                'max_log_scale_speed': TEMPORAL_LOG_SCALE_SPEED_GRID,
                'config_count': len(rows),
            },
            'legacy': {
                'config': {
                    'max_gap': TEMPORAL_MAX_GAP,
                    'min_seg_len': TEMPORAL_MIN_SEG,
                    'max_center_speed': None,
                    'max_log_scale_speed': None,
                },
                'mean_st_iou': default_result['mean_st_iou'],
                'detected_frames': default_result['detected_frames'],
                'per_video': default_result['per_video'],
            },
            'best': {
                'config': best_config,
                'mean_st_iou': best_result['mean_st_iou'],
                'detected_frames': best_result['detected_frames'],
                'per_video': best_result['per_video'],
                'delta_vs_legacy': best_result['mean_st_iou'] - default_result['mean_st_iou'],
            },
            'top_results': rows[:TEMPORAL_TOP_K],
            'outputs': {
                'legacy_predictions': OUTPUT_FILE,
                'best_predictions': TEMPORAL_PREDICTIONS_FILE,
                'all_grid_results': TEMPORAL_RESULTS_CSV,
            },
        }
        atomic_write_json(TEMPORAL_SUMMARY_FILE, summary)
        print(f'\nTemporal summary: {TEMPORAL_SUMMARY_FILE}')
        print(f'All temporal rows: {TEMPORAL_RESULTS_CSV}')
        print(f'Best temporal predictions: {TEMPORAL_PREDICTIONS_FILE}')
        return

    if not RUN_CALIBRATION:
        visualize_sample_results(default_result['predictions'], default_result['debug_info'], num_videos=3)
        return

    rows, best, best_result = run_calibration(cache, gt_map)
    best_weights = {
        'yolo': best['weight_yolo'],
        'siamese': best['weight_siamese'],
        'color': best['weight_color'],
    }
    print('\nBEST CALIBRATION CONFIG')
    print(f"  weights={best_weights} | threshold={best['matching_threshold']:.2f}")
    print_evaluation('CALIBRATED', best_result)

    top_rows = rows[:CALIBRATION_TOP_K]
    summary = {
        'experiment': CALIBRATION_EXPERIMENT_NAME,
        'calibration_profile': CALIBRATION_PROFILE,
        'selection_objective': 'maximum mean ST-IoU on all supplied manual GT videos',
        'overfitting_warning': (
            'The same six manually annotated videos select and report this configuration; '
            'this is an offline optimum, not an unbiased generalization estimate.'
        ),
        'cache_signature': signature,
        'candidate_cache_file': CANDIDATE_CACHE_FILE,
        'search_space': {
            'thresholds': CALIBRATION_THRESHOLDS,
            'siamese_weights': CALIBRATION_SIAMESE_WEIGHTS,
            'color_weights': CALIBRATION_COLOR_WEIGHTS,
            'minimum_yolo_weight': CALIBRATION_MIN_YOLO_WEIGHT,
            'config_count': len(rows),
        },
        'default': {
            'weights': default_weights,
            'matching_threshold': MATCHING_THRESHOLD,
            'mean_st_iou': default_result['mean_st_iou'],
            'detected_frames': default_result['detected_frames'],
            'per_video': default_result['per_video'],
        },
        'best': {
            'weights': best_weights,
            'matching_threshold': best['matching_threshold'],
            'mean_st_iou': best_result['mean_st_iou'],
            'detected_frames': best_result['detected_frames'],
            'per_video': best_result['per_video'],
            'delta_vs_default': best_result['mean_st_iou'] - default_result['mean_st_iou'],
        },
        'top_results': top_rows,
        'outputs': {
            'default_predictions': OUTPUT_FILE,
            'calibrated_predictions': CALIBRATED_OUTPUT_FILE,
            'all_grid_results': CALIBRATION_CSV,
        },
    }
    atomic_write_json(CALIBRATION_SUMMARY, summary)
    print(f'\nCalibration summary: {CALIBRATION_SUMMARY}')
    print(f'All grid rows: {CALIBRATION_CSV}')
    print(f'Calibrated predictions: {CALIBRATED_OUTPUT_FILE}')
    visualize_sample_results(best_result['predictions'], best_result['debug_info'], num_videos=3)


if __name__ == "__main__":
    main()
